# Malicious-insider threat detection on CERT r4.2: sampler selection, HGB detector, temporal analysis

Single self-contained Kaggle notebook. **Nothing to upload except the data:**

1. Right side → **Add Input** → attach the CERT insider-threat dataset (it appears under `/kaggle/input/...`).
2. (Optional) Settings → **Accelerator → GPU**. Only the 7d ablation, the DeepSMOTE candidate and the optional LSTM Layer-1 evaluator use PyTorch; everything else runs on CPU.
3. **Run All.** The collapsed cells in section 1 write the code files; the rest is the analysis. Edit the **Config** cell to taste.

**Pipeline:** stream the raw CSVs into behavioural features per user per 2-day window → T×F sequences → user-level train/val/test split →
**Layer 1** picks a sampler (scored with HGB, the same model as Layer 2; `LAYER1_EVALUATOR` in Config switches back to the LSTM proxy) (NM-2, RUS, DeepSMOTE, class-weighted loss, none) → **Layer 2** is one gradient-boosting model (HGB) on
last / max / mean summaries of the 7-window history → thresholds tuned on validation (alert budget) → out-of-fold scores for every user.
**CERT r4.2 only.** No database is used.

| Section | What it answers |
|---|---|
| 5 | Which sampler works best? Does the paper's NM-2 hold up? |
| 6, 7 | One held-out split: window- and user-level results, who gets flagged |
| 7b | **Main evidence:** 7-fold out-of-fold scores for all 1000 users, alert-budget metrics, paper comparison |
| 7c | Which features drive the score? |
| 7d | Do LSTM, BiLSTM-attention or a blend beat HGB? (paired bootstraps) |
| 7e | Does a temporal component (trend features, CUSUM) beat plain HGB? |
| 8 | Summary of results |

Approximate run times on the full data: features 18 min (cached after the first run), Layer 1 (LSTM evaluator, old) 35 min; with the HGB evaluator it is much shorter, re-time it after the first run, 7b 2 min, 7c 5 min, 7d 89 min, 7e 6 min.


## 1. Setup — writes the code files (collapsed cells below), then imports them

In [1]:
import os
CODE_DIR = "/kaggle/working/insider_code"
for d in ("app/ml/sampling",):
    os.makedirs(f"{CODE_DIR}/{d}", exist_ok=True)

In [2]:
%%writefile /kaggle/working/insider_code/app/__init__.py
# (package marker)

Writing /kaggle/working/insider_code/app/__init__.py


In [3]:
%%writefile /kaggle/working/insider_code/app/ml/__init__.py
# (package marker)

Writing /kaggle/working/insider_code/app/ml/__init__.py


In [4]:
%%writefile /kaggle/working/insider_code/app/ml/sampling/__init__.py
"""
Layer-1 sampling stage (architecture.md section 3.5).

A candidate is a *spec string*:

    base[@ratio][+cw][^power]

    base   none | class_weight | NM-2 | RUS | DeepSMOTE
    @ratio target minority:majority ratio AFTER resampling (0.05 = 1 insider
           window per 20 normal ones). Omitted = 1.0 (fully balanced, the
           original behaviour). Ignored by none / class_weight.
    +cw    hybrid: after resampling, ALSO weight the positive class in the loss
           to cover the residual imbalance.
    ^power softens a class weight: pos_weight = (n_neg / n_pos) ** power.
           Default 1.0 for the class_weight base, 0.5 for +cw.

    "NM-2"               full 1:1 balance (old behaviour)
    "NM-2@0.05"          keep majority windows until minority:majority = 1:20
    "DeepSMOTE@0.05+cw"  oversample to 1:20, then half-strength class weight
    "class_weight^0.5"   no resampling, square-root-softened loss weight

Candidates
    NM-2          undersampler (paper's best; baseline to beat)
    RUS           random undersampler (diagnostic: is NM-2's *selection* the
                  problem, or undersampling itself?)
    DeepSMOTE     encoder/decoder latent-space oversampler
    none          control -- no imbalance handling
    class_weight  control -- imbalance handled in the loss, data untouched

All samplers take FLATTENED (N, T*F) training vectors; callers reshape back
to (N, T, F). Only ever call these on the TRAIN split.
"""
import re
from dataclasses import dataclass, field
from typing import Optional

import numpy as np

SAMPLER_CANDIDATES = ("NM-2", "DeepSMOTE", "none", "class_weight")     # original four (full balance)

# Default sweep used by the notebook: partial ratios + hybrids.
SWEEP_CANDIDATES = (
    "none", "class_weight", "class_weight^0.5",
    "NM-2@0.02", "NM-2@0.05", "NM-2@0.1", "NM-2@0.05+cw",
    "RUS@0.05",
    "DeepSMOTE@0.02", "DeepSMOTE@0.05", "DeepSMOTE@0.05+cw",
)

_BASES = ("none", "class_weight", "NM-2", "RUS", "DeepSMOTE")
_SPEC = re.compile(r"^(?P<base>NM-2|DeepSMOTE|RUS|none|class_weight)"
                   r"(?:@(?P<ratio>[0-9]*\.?[0-9]+))?(?P<cw>\+cw)?(?:\^(?P<pow>[0-9]*\.?[0-9]+))?$")


@dataclass
class ResampleResult:
    X: np.ndarray
    y: np.ndarray
    pos_weight: Optional[float] = None     # loss weight for the positive class (class_weight / +cw)
    info: dict = field(default_factory=dict)


def parse_spec(spec: str) -> dict:
    m = _SPEC.match(spec.strip())
    if not m:
        raise ValueError(f"Unknown sampler spec '{spec}'. Format: base[@ratio][+cw][^power] with base in {list(_BASES)}")
    ratio = float(m["ratio"]) if m["ratio"] else None
    if ratio is not None and not (0 < ratio <= 1):
        raise ValueError(f"ratio in '{spec}' must be in (0, 1]")
    return {"base": m["base"], "ratio": ratio, "cw": bool(m["cw"]), "power": float(m["pow"]) if m["pow"] else None}


def base_of(spec: str) -> str:
    return parse_spec(spec)["base"]


def kwargs_for(spec: str, sampler_kwargs: Optional[dict]) -> dict:
    """Per-candidate overrides: looked up by the full spec first, then by its base name."""
    sampler_kwargs = sampler_kwargs or {}
    return dict(sampler_kwargs.get(spec) or sampler_kwargs.get(base_of(spec)) or {})


def _pos_weight(y: np.ndarray, power: float) -> float:
    n_pos = max(int((y == 1).sum()), 1)
    return float((int((y == 0).sum()) / n_pos) ** power)


def resample(name: str, X: np.ndarray, y: np.ndarray, seed: int = 0, **kwargs) -> ResampleResult:
    p = parse_spec(name)
    base, ratio = p["base"], p["ratio"]
    cache = kwargs.pop("cache", None)                      # DeepSMOTE only: reuse the trained autoencoder across ratios
    y = np.asarray(y).astype(int)

    if base == "none":
        return ResampleResult(X, y, None, {"kept": int(len(y)), "spec": name})
    if base == "class_weight":
        pw = _pos_weight(y, p["power"] if p["power"] is not None else 1.0)
        return ResampleResult(X, y, pw, {"kept": int(len(y)), "pos_weight": round(pw, 3), "spec": name})

    if base == "NM-2":
        from app.ml.sampling.nm2 import resample_nm2
        Xr, yr, info = resample_nm2(X, y, ratio=ratio or 1.0, **kwargs)
    elif base == "RUS":
        from app.ml.sampling.rus import resample_rus
        Xr, yr, info = resample_rus(X, y, seed=seed, ratio=ratio or 1.0, **kwargs)
    else:  # DeepSMOTE
        from app.ml.sampling.deepsmote import resample_deepsmote   # imports torch lazily
        Xr, yr, info = resample_deepsmote(X, y, seed=seed, target_ratio=ratio or 1.0, cache=cache, **kwargs)

    pw = None
    if p["cw"]:
        pw = _pos_weight(yr, p["power"] if p["power"] is not None else 0.5)
        info = {**info, "pos_weight": round(pw, 3)}
    return ResampleResult(Xr, yr, pw, {**info, "spec": name, "ratio_after": round(float((yr == 1).sum() / max((yr == 0).sum(), 1)), 4)})


Writing /kaggle/working/insider_code/app/ml/sampling/__init__.py


In [5]:
%%writefile /kaggle/working/insider_code/app/ml/feature_core.py
"""
DB-free feature engineering core.

Everything that turns raw events into one feature vector per (user, window)
lives here with NO database / SQLAlchemy / FastAPI imports, so the same code
serves both the Postgres-backed API (app/ml/features.py) and the streaming
Kaggle/offline path (app/ml/offline_features.py). Sharing one implementation
is what guarantees the two paths produce identical features.

Two stages, so aggregation can be done chunk-by-chunk on data too big for RAM:
  event_indicators()   per-event 0/1 indicator columns (pure row-wise)
  finalize_features()  sums + distinct counts -> the 12 feature columns
Additive columns (ADDITIVE) can be summed across chunks; the two DISTINCT
features (distinct PCs, distinct URLs) cannot, so chunked callers collect the
distinct (user, window, value) triples and count them at the end.
"""
from datetime import timedelta
from typing import Optional

import numpy as np
import pandas as pd

AFTER_HOURS_START = 19   # 7pm
AFTER_HOURS_END = 7      # 7am
DEFAULT_INTERNAL_DOMAIN = "@company.com"

FEATURE_COLUMNS = [
    "logon_count", "after_hours_logon_count", "weekend_logon_count",
    "distinct_pc_count", "device_connect_count",
    "email_count", "email_external_ratio", "email_attachment_count",
    "file_count", "file_usb_count",
    "http_count", "http_distinct_domains",
    # --- added ---
    "after_hours_device_count", "after_hours_file_count", "after_hours_http_count", "after_hours_email_count",
    "http_leak_count", "http_job_count", "http_cloud_count",
]

ADDITIVE = [
    "logon_count", "after_hours_logon_count", "weekend_logon_count", "device_connect_count",
    "email_count", "_ext", "email_attachment_count", "file_count", "file_usb_count", "http_count",
    "after_hours_device_count", "after_hours_file_count", "after_hours_http_count", "after_hours_email_count",
    "http_leak_count", "http_job_count", "http_cloud_count",
]

# starting lists -- tune to the domains that actually occur in your http.csv
LEAK_RE = r"wikileaks|pastebin|anonfiles|keylogger"
JOB_RE = r"monster\.com|careerbuilder|indeed\.com|linkedin\.com|simplyhired|jobhuntersbible|hotjobs|job-?hunt|glassdoor"
CLOUD_RE = r"dropbox|mega\.nz|wetransfer|drive\.google|onedrive|box\.com|mediafire|4shared|sendspace"


def _has_external(cell, dom: str) -> bool:
    """True if ANY recipient in a ';'-separated list is outside the internal domain."""
    if not isinstance(cell, str):
        return False
    return any(r.strip() and not r.strip().lower().endswith(dom) for r in cell.split(";"))

EVENT_COLUMNS = ["user_id", "source", "pc", "activity", "detail", "timestamp"]


def event_indicators(events: pd.DataFrame, window_days: int, origin: pd.Timestamp,
                     internal_domain: str = DEFAULT_INTERNAL_DOMAIN) -> pd.DataFrame:
    """Per-event indicator columns + window index (widx = whole windows since `origin`)."""
    ts = events["timestamp"]
    window = timedelta(days=window_days)
    widx = ((ts - origin) // window).astype("int64")

    src = events["source"].astype("category")
    src = src.cat.rename_categories([getattr(c, "value", c) for c in src.cat.categories])
    is_logon, is_dev = src == "logon", src == "device"
    is_email, is_file, is_http = src == "email", src == "file", src == "http"

    detail = events["detail"]
    ext = pd.Series(False, index=events.index)
    if is_email.any():
        dom = internal_domain.lower()
        ext.loc[is_email] = detail[is_email].map(lambda x: _has_external(x, dom)).to_numpy(dtype=bool)

    # real attachment flag (offline_features passes the e-mail `attachments` column as `n_att`)
    if "n_att" in events.columns:
        a = events["n_att"].astype("string").str.strip()
        num = pd.to_numeric(a, errors="coerce")
        has_att = np.asarray(((num > 0) | (num.isna() & a.notna() & (a.str.len() > 0))).fillna(False), dtype=bool)
    else:
        has_att = detail.notna().to_numpy()          # legacy fallback (NOT a real attachment signal)

    hour = ts.dt.hour
    ah = (hour >= AFTER_HOURS_START) | (hour < AFTER_HOURS_END)
    url = detail.where(is_http)
    return pd.DataFrame({
        "widx": widx.to_numpy(),
        "user_id": events["user_id"].to_numpy(),
        "logon_count": is_logon.to_numpy().astype(int),
        "after_hours_logon_count": (is_logon & ((hour >= AFTER_HOURS_START) | (hour < AFTER_HOURS_END))).to_numpy().astype(int),
        "weekend_logon_count": (is_logon & (ts.dt.weekday >= 5)).to_numpy().astype(int),
        "pc": events["pc"].to_numpy(),
        "device_connect_count": (is_dev & (events["activity"] == "Connect")).to_numpy().astype(int),
        "email_count": is_email.to_numpy().astype(int),
        "_ext": ext.to_numpy().astype(int),
        "email_attachment_count": (is_email.to_numpy() & has_att).astype(int),
        "file_count": is_file.to_numpy().astype(int),
        "file_usb_count": (is_file & events["pc"].astype(str).str.contains("usb", case=False, na=False)).to_numpy().astype(int),
        "http_count": is_http.to_numpy().astype(int),
        "after_hours_device_count": (is_dev & (events["activity"] == "Connect") & ah).to_numpy().astype(int),
        "after_hours_file_count": (is_file & ah).to_numpy().astype(int),
        "after_hours_http_count": (is_http & ah).to_numpy().astype(int),
        "after_hours_email_count": (is_email & ah).to_numpy().astype(int),
        "http_leak_count": url.str.contains(LEAK_RE, case=False, regex=True, na=False).to_numpy().astype(int),
        "http_job_count": url.str.contains(JOB_RE, case=False, regex=True, na=False).to_numpy().astype(int),
        "http_cloud_count": url.str.contains(CLOUD_RE, case=False, regex=True, na=False).to_numpy().astype(int),
        "_http_detail": url.to_numpy(),
    })


def finalize_features(sums: pd.DataFrame, distinct_pc: pd.Series, distinct_http: pd.Series,
                      origin: pd.Timestamp, window_days: int) -> pd.DataFrame:
    """
    sums: index (widx, user_id), columns ADDITIVE (already summed over all events).
    distinct_pc / distinct_http: Series on the same (widx, user_id) key -> distinct counts.
    """
    out = sums.copy()
    out["distinct_pc_count"] = distinct_pc.reindex(out.index).fillna(0).astype(int)
    out["http_distinct_domains"] = distinct_http.reindex(out.index).fillna(0).astype(int)
    out["email_external_ratio"] = np.where(out["email_count"] > 0, out["_ext"] / out["email_count"].clip(lower=1), 0.0)
    out = out.reset_index()
    window = timedelta(days=window_days)
    out["window_start"] = origin + out["widx"] * window
    out["window_end"] = out["window_start"] + window
    return out[["user_id", "window_start", "window_end"] + FEATURE_COLUMNS]


def compute_feature_windows(events: pd.DataFrame, window_days: int = 7,
                            internal_domain: str = DEFAULT_INTERNAL_DOMAIN,
                            origin: Optional[pd.Timestamp] = None) -> pd.DataFrame:
    """
    Vectorized feature engineering: assigns every event to its window with
    integer division and aggregates with one groupby (no per-window rescan).
    Output is identical to the original per-window implementation (verified in
    tests/test_features.py). `origin` defaults to the earliest event.
    `internal_domain`: recipients ending with it count as internal e-mail
    (default keeps the original behavior; real CERT uses "@dtaa.com").
    """
    cols = ["user_id", "window_start", "window_end"] + FEATURE_COLUMNS
    if events.empty:
        return pd.DataFrame(columns=cols)
    origin = events["timestamp"].min() if origin is None else origin
    work = event_indicators(events, window_days, origin, internal_domain)
    g = work.groupby(["widx", "user_id"], sort=True)
    return finalize_features(g[ADDITIVE].sum(), g["pc"].nunique(), g["_http_detail"].nunique(), origin, window_days)

Writing /kaggle/working/insider_code/app/ml/feature_core.py


In [6]:
%%writefile /kaggle/working/insider_code/app/ml/offline_features.py
"""
Streaming CSV -> per-window features, with NO database (for Kaggle / big data).

The CERT release is many GB across logon/device/email/file/http CSVs. This
module reads each CSV in chunks (only the columns it needs -- e.g. it never
loads http.csv's huge `content` column), computes the same 12 features as the
DB path (it calls the same app/ml/feature_core.py), and keeps only small
aggregates in memory:

  * additive counts  -> summed per (window, user) chunk by chunk
  * distinct PCs / distinct URLs -> the set of distinct (window, user, value)
    triples (URLs are hashed to 64-bit ints), counted at the end

Peak memory is therefore ~ one chunk + the aggregates, not the dataset.
The result is a table with one row per (user, window) -- typically a few
hundred MB at most -- which is cached to disk so the slow pass runs once.

Optional user subsampling (`keep_users`) drops rows for other users while
streaming, cutting both time and memory; see select_users().
"""
import json
import pickle
import time
from pathlib import Path
from typing import Callable, Dict, Iterable, List, Optional, Set

import numpy as np
import pandas as pd

from app.ml.feature_core import (
    ADDITIVE, DEFAULT_INTERNAL_DOMAIN, EVENT_COLUMNS, event_indicators, finalize_features,
)

# file -> (source, detail column). Same mapping as app/ml/ingestion.py.
SOURCE_FILES = {
    "logon.csv": ("logon", None),
    "device.csv": ("device", None),
    "email.csv": ("email", "to"),
    "file.csv": ("file", "filename"),
    "http.csv": ("http", "url"),
}
CERT_DATE_FORMAT = "%m/%d/%Y %H:%M:%S"


def find_dataset_dir(root: str) -> Path:
    """Finds the directory holding the CERT CSVs (Kaggle nests them under /kaggle/input/<name>/...)."""
    root_p = Path(root)
    for p in [root_p, *sorted(root_p.rglob("*"))]:
        if p.is_dir() and any((p / f).exists() for f in SOURCE_FILES):
            return p
    raise FileNotFoundError(f"No CERT CSVs ({', '.join(SOURCE_FILES)}) found under {root}")


def _parse_dates(s: pd.Series, date_format: Optional[str]) -> pd.Series:
    if date_format:
        out = pd.to_datetime(s, format=date_format, errors="coerce")
        if out.isna().mean() <= 0.5:
            return out
    return pd.to_datetime(s, errors="coerce")      # slower fallback: let pandas infer


def _iter_files(directory: Path):
    for fname, (source, detail_col) in SOURCE_FILES.items():
        path = directory / fname
        if path.exists():
            yield path, source, detail_col


def scan_min_timestamp(directory, chunksize: int = 2_000_000, date_format: Optional[str] = CERT_DATE_FORMAT,
                       progress: Callable[[str], None] = print) -> pd.Timestamp:
    """Cheap pass over just the `date` column to anchor the window grid (same origin the DB path uses)."""
    best = None
    for path, _, _ in _iter_files(Path(directory)):
        for chunk in pd.read_csv(path, usecols=["date"], chunksize=chunksize, dtype=str):
            m = _parse_dates(chunk["date"], date_format).min()
            if pd.notna(m) and (best is None or m < best):
                best = m
        progress(f"[origin] {path.name}: earliest so far {best}")
    if best is None:
        raise ValueError("no parseable timestamps found")
    return best


def list_users(directory, chunksize: int = 2_000_000) -> List[str]:
    """All user codes (from the first available CSV -- logon.csv in CERT)."""
    users: Set[str] = set()
    for path, _, _ in _iter_files(Path(directory)):
        for chunk in pd.read_csv(path, usecols=["user"], chunksize=chunksize, dtype=str):
            users.update(chunk["user"].dropna().unique())
        break
    return sorted(users)


def select_users(all_users: Iterable[str], insider_users: Iterable[str],
                 max_benign: Optional[int] = None, seed: int = 0) -> Set[str]:
    """
    Keep every insider + a random sample of `max_benign` non-insiders (all if None).
    CAUTION: dropping benign users raises the insider prevalence in val/test, so
    precision / PR-AUC become optimistic relative to the real population. The
    training job records the sampling rate so this can be reported.
    """
    all_users, insiders = set(all_users), set(insider_users)
    benign = sorted(all_users - insiders)
    if max_benign is not None and max_benign < len(benign):
        rng = np.random.RandomState(seed)
        benign = [benign[i] for i in rng.choice(len(benign), size=max_benign, replace=False)]
    return (insiders & all_users) | set(benign)


class _Aggregator:
    """Accumulates per-chunk partial aggregates and compacts them periodically."""

    def __init__(self, compact_every: int = 8):
        self.sums: List[pd.DataFrame] = []
        self.pcs: List[pd.DataFrame] = []
        self.urls: List[pd.DataFrame] = []
        self.compact_every, self._n = compact_every, 0

    def add(self, work: pd.DataFrame):
        self.sums.append(work.groupby(["widx", "user_id"])[ADDITIVE].sum())
        pcs = work.loc[work["pc"].notna(), ["widx", "user_id", "pc"]].drop_duplicates()
        self.pcs.append(pcs)
        http = work.loc[work["_http_detail"].notna(), ["widx", "user_id", "_http_detail"]]
        if len(http):
            h = pd.util.hash_pandas_object(http["_http_detail"], index=False).to_numpy()
            self.urls.append(pd.DataFrame({"widx": http["widx"].to_numpy(), "user_id": http["user_id"].to_numpy(),
                                           "h": h}).drop_duplicates())
        self._n += 1
        if self._n % self.compact_every == 0:
            self.compact()

    def compact(self):
        if len(self.sums) > 1:
            self.sums = [pd.concat(self.sums).groupby(level=[0, 1]).sum()]
        if len(self.pcs) > 1:
            self.pcs = [pd.concat(self.pcs).drop_duplicates()]
        if len(self.urls) > 1:
            self.urls = [pd.concat(self.urls).drop_duplicates()]

    def result(self, origin: pd.Timestamp, window_days: int) -> pd.DataFrame:
        self.compact()
        if not self.sums:
            return pd.DataFrame()
        sums = self.sums[0].sort_index()
        dpc = self.pcs[0].groupby(["widx", "user_id"]).size() if self.pcs else pd.Series(dtype=int)
        dht = self.urls[0].groupby(["widx", "user_id"]).size() if self.urls else pd.Series(dtype=int)
        return finalize_features(sums, dpc, dht, origin, window_days)


def build_features_from_csvs(
    directory, window_days: int = 2, internal_domain: str = DEFAULT_INTERNAL_DOMAIN,
    keep_users: Optional[Set[str]] = None, chunksize: int = 2_000_000,
    origin: Optional[pd.Timestamp] = None, date_format: Optional[str] = CERT_DATE_FORMAT,
    progress: Callable[[str], None] = print,
):
    """
    Streams the CERT CSVs in `directory` and returns (features_df, user_codes)
    where features_df has the same columns/semantics as build_feature_windows()
    and user_codes maps the integer user_id -> user code (e.g. "AAM0658").
    """
    directory = Path(directory)
    files = list(_iter_files(directory))
    if not files:
        raise FileNotFoundError(f"No CERT CSVs found in {directory}")
    if origin is None:
        origin = scan_min_timestamp(directory, chunksize, date_format, progress)
    progress(f"[features] window grid origin = {origin}, window_days = {window_days}")

    agg = _Aggregator()
    user_ids: Dict[str, int] = {}
    t0 = time.time()
    for path, source, detail_col in files:
        usecols = ["date", "user", "pc", "activity"] + ([detail_col] if detail_col else []) + (["attachments"] if source == "email" else [])
        rows_seen = rows_kept = 0
        for chunk in pd.read_csv(path, usecols=lambda c: c in usecols, chunksize=chunksize, dtype=str):
            rows_seen += len(chunk)
            if keep_users is not None:
                chunk = chunk[chunk["user"].isin(keep_users)]
            ts = _parse_dates(chunk["date"], date_format)
            ok = ts.notna() & chunk["user"].notna()
            chunk, ts = chunk[ok], ts[ok]
            if chunk.empty:
                continue
            for u in chunk["user"].unique():
                if u not in user_ids:
                    user_ids[u] = len(user_ids) + 1
            events = pd.DataFrame({
                "user_id": chunk["user"].map(user_ids).to_numpy(),
                "source": source,
                "pc": chunk["pc"].to_numpy() if "pc" in chunk else None,
                "activity": chunk["activity"].to_numpy() if "activity" in chunk else None,
                "detail": chunk[detail_col].to_numpy() if detail_col and detail_col in chunk else None,
                "timestamp": ts.to_numpy(),
            })[EVENT_COLUMNS]
            if "attachments" in chunk:
                events["n_att"] = chunk["attachments"].to_numpy()
            events = events[events["timestamp"] >= origin]
            rows_kept += len(events)
            agg.add(event_indicators(events, window_days, origin, internal_domain))
            progress(f"[features] {path.name}: {rows_seen:,} rows read, {rows_kept:,} kept ({time.time() - t0:.0f}s)")
        progress(f"[features] finished {path.name}")

    features = agg.result(origin, window_days)
    if features.empty:
        raise ValueError("No events survived filtering (check keep_users / date format).")
    user_codes = {i: u for u, i in user_ids.items()}
    progress(f"[features] done: {len(features):,} (user, window) rows for {len(user_codes):,} users in {time.time() - t0:.0f}s")
    return features, user_codes


# ------------------------------------------------------------- caching

def save_features(path, features_df: pd.DataFrame, user_codes: Dict[int, str], meta: Optional[dict] = None) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump({"features": features_df, "user_codes": user_codes, "meta": meta or {}}, f, protocol=4)


def load_features(path):
    with open(path, "rb") as f:
        d = pickle.load(f)
    return d["features"], {int(k): v for k, v in d["user_codes"].items()}, d.get("meta", {})

Writing /kaggle/working/insider_code/app/ml/offline_features.py


In [7]:
%%writefile /kaggle/working/insider_code/app/ml/metrics.py
"""
Shared evaluation metrics + threshold tuning.

Every experiment in the evaluation plan (architecture.md section 5) reports
the same four selection metrics -- recall, precision, F1, PR-AUC -- plus
accuracy for reference only. Accuracy is *not* used for any selection
decision: under heavy class imbalance a model that flags nobody scores
>95% accuracy.
"""
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional

import numpy as np
from sklearn.metrics import (
    accuracy_score, average_precision_score, f1_score,
    precision_recall_curve, precision_score, recall_score,
)

METRIC_NAMES = ["recall", "precision", "f1", "pr_auc", "accuracy"]


def pr_auc(y_true, score) -> float:
    """Average precision (area under the precision-recall curve). NaN if no positives."""
    y = np.asarray(y_true).astype(int)
    if y.sum() == 0:
        return float("nan")
    return float(average_precision_score(y, np.asarray(score, dtype=float)))


def tune_threshold(y_true, score, min_recall: float = 0.9) -> float:
    """
    Picks a decision threshold on a *validation* set, prioritizing recall
    (a missed insider costs more than a false alert): among all thresholds
    that reach at least `min_recall`, take the one with the highest
    precision (ties -> the higher threshold, i.e. fewer alerts).

    Works on any real-valued score (LSTM probability, SVM margin, anomaly
    score). Prediction rule is `score >= threshold`. The lowest threshold
    always reaches recall 1.0, so a valid choice always exists.
    """
    y = np.asarray(y_true).astype(int)
    s = np.asarray(score, dtype=float)
    if y.sum() == 0:
        raise ValueError("Cannot tune a threshold: the validation set has no positive samples.")
    precision, recall, thresholds = precision_recall_curve(y, s)
    precision, recall = precision[:-1], recall[:-1]   # align with `thresholds`
    ok = np.flatnonzero(recall >= min_recall)
    best = ok[np.lexsort((thresholds[ok], precision[ok]))[-1]]
    return float(thresholds[best])


@dataclass
class ThresholdPolicy:
    """
    How the alert threshold is chosen on the VALIDATION set.

      budget     flag the top `budget` fraction of validation items (windows or
                 users). Depends on the score distribution of the many normal
                 items, not on the handful of positives, so it transfers to new
                 data far better than a recall target. Think "analyst capacity".
      precision  highest-recall threshold whose validation precision >= min_precision
                 (falls back to the best-F1 threshold if none reaches it)
      fbeta      threshold maximizing F-beta (beta < 1 favours precision)
      recall     legacy rule: best precision subject to recall >= min_recall
    """
    mode: str = "budget"
    budget: float = 0.01
    min_precision: float = 0.2
    min_recall: float = 0.9
    beta: float = 1.0

    @classmethod
    def coerce(cls, p) -> "ThresholdPolicy":
        if p is None:
            return cls()
        return p if isinstance(p, cls) else cls(**p)


def choose_threshold(y_true, score, policy: Optional[ThresholdPolicy] = None) -> float:
    """Pick a decision threshold on validation data according to `policy`. Prediction rule: score >= threshold."""
    pol = ThresholdPolicy.coerce(policy)
    s = np.asarray(score, dtype=float)
    y = None if y_true is None else np.asarray(y_true).astype(int)
    if pol.mode == "recall":
        return tune_threshold(y, s, pol.min_recall)
    if pol.mode == "budget":
        if not 0 < pol.budget < 1:
            raise ValueError("budget must be in (0, 1)")
        n_flag = max(1, int(round(pol.budget * len(s))))
        return float(np.sort(s)[::-1][n_flag - 1])               # the n_flag-th highest score is the cut-off
    if y is None or y.sum() == 0:
        raise ValueError("Cannot tune a threshold: the validation set has no positive samples.")
    precision, recall, thresholds = precision_recall_curve(y, s)
    precision, recall = precision[:-1], recall[:-1]
    beta2 = pol.beta ** 2
    f = (1 + beta2) * precision * recall / np.maximum(beta2 * precision + recall, 1e-12)
    if pol.mode == "precision":
        ok = np.flatnonzero(precision >= pol.min_precision)
        if len(ok):
            return float(thresholds[ok[np.argmax(recall[ok])]])
        return float(thresholds[int(np.argmax(f))])
    if pol.mode == "fbeta":
        return float(thresholds[int(np.argmax(f))])
    raise ValueError(f"unknown threshold mode '{pol.mode}'")


def precision_recall_at_k(y_true, score, ks: Iterable[int]) -> Dict[str, float]:
    """Threshold-free view: if an analyst reviews the K highest-scored items, how many are real?"""
    y = np.asarray(y_true).astype(int)
    order = np.argsort(-np.asarray(score, dtype=float), kind="stable")
    out = {}
    for k in ks:
        k = int(min(k, len(y)))
        hit = int(y[order[:k]].sum())
        out[f"precision@{k}"] = round(hit / max(k, 1), 4)
        out[f"recall@{k}"] = round(hit / max(int(y.sum()), 1), 4)
    return out


def metrics_at_budget(y_true, score, frac: float) -> Dict[str, float]:
    """Flag the top `frac` of THIS set by score -- no threshold carried over from validation, so no drift."""
    y, sc = np.asarray(y_true).astype(int), np.asarray(score, dtype=float)
    k = max(1, int(round(frac * len(y))))
    tp = int(y[np.argsort(-sc, kind="stable")[:k]].sum())
    return {"recall": round(tp / max(int(y.sum()), 1), 4), "precision": round(tp / k, 4), "n_flagged": k, "true_flagged": tp}


def compute_metrics(y_true, score, threshold: float) -> Dict[str, float]:
    y = np.asarray(y_true).astype(int)
    s = np.asarray(score, dtype=float)
    pred = (s >= threshold).astype(int)
    return {
        "recall": round(float(recall_score(y, pred, zero_division=0)), 4),
        "precision": round(float(precision_score(y, pred, zero_division=0)), 4),
        "f1": round(float(f1_score(y, pred, zero_division=0)), 4),
        "pr_auc": round(pr_auc(y, s), 4),
        "accuracy": round(float(accuracy_score(y, pred)), 4),
        "n": int(len(y)),
        "n_positive": int(y.sum()),
        "n_flagged": int(pred.sum()),
    }


def aggregate(runs: Iterable[Dict[str, float]], keys: List[str] = None) -> Dict[str, Dict[str, float]]:
    """mean +/- std across seeds (NaN-safe). Std is the sample std when n > 1."""
    runs = list(runs)
    keys = keys or METRIC_NAMES
    out = {}
    for k in keys:
        vals = np.array([r[k] for r in runs if r.get(k) is not None], dtype=float)
        vals = vals[~np.isnan(vals)]
        if len(vals) == 0:
            out[k] = {"mean": None, "std": None, "n": 0}
        else:
            out[k] = {
                "mean": round(float(vals.mean()), 4),
                "std": round(float(vals.std(ddof=1)) if len(vals) > 1 else 0.0, 4),
                "n": int(len(vals)),
            }
    return out


def to_jsonable(obj):
    """Recursively converts numpy scalars/arrays and NaN/inf to JSON-safe values
    (Postgres JSON columns reject NaN). Keys starting with '_' are dropped --
    they carry in-memory-only arrays."""
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items() if not str(k).startswith("_")}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return to_jsonable(obj.tolist())
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (float, np.floating)):
        f = float(obj)
        return None if (np.isnan(f) or np.isinf(f)) else f
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj

Writing /kaggle/working/insider_code/app/ml/metrics.py


In [8]:
%%writefile /kaggle/working/insider_code/app/ml/user_level.py
"""
User-level scoring.

The LSTM scores every (user, window). Analysts act on USERS, and taking the max
window probability per user is the noisiest possible aggregate: a user with 250
windows gets 250 chances for one random spike. Three aggregators:

  max          legacy: highest window probability
  topk         mean of the user's k highest window probabilities
  consecutive  highest value v such that the user has n back-to-back windows
               ALL with probability >= v. A user is flagged at threshold t iff
               n consecutive windows are >= t (a single spike never flags).
               Windows with no activity (gaps in the grid) break a run.

All three are continuous scores, so the same threshold policies and PR-AUC
apply to them.
"""
from typing import Dict, Optional, Sequence

import numpy as np
import pandas as pd

from app.ml.metrics import ThresholdPolicy, choose_threshold, pr_auc, precision_recall_at_k

AGG_METHODS = ("max", "topk", "consecutive")
DEFAULT_AGG = {"method": "topk", "k": 3, "n": 3}


def _consecutive_min(user: np.ndarray, widx: np.ndarray, p: np.ndarray, n: int) -> np.ndarray:
    """Arrays sorted by (user, widx). out[t] = min(p[t..t+n-1]) if those windows are contiguous, else -inf."""
    N = len(p)
    out = np.full(N, -np.inf)
    if n <= 1:
        return p.astype(float).copy()
    if N < n:
        return out
    m = N - n + 1
    ok = np.ones(m, dtype=bool)
    val = p[:m].astype(float).copy()
    for i in range(1, n):
        ok &= (user[i:m + i] == user[:m]) & (widx[i:m + i] - widx[:m] == i)
        val = np.minimum(val, p[i:m + i])
    out[:m][ok] = val[ok]
    return out


def aggregate_user_scores(user_ids, window_starts, probs, method: str = "topk", k: int = 3, n: int = 3,
                          window_days: Optional[int] = None) -> pd.Series:
    """Returns a Series indexed by user_id: one score per user."""
    df = pd.DataFrame({"user": np.asarray(user_ids), "ws": pd.to_datetime(np.asarray(window_starts)),
                       "p": np.asarray(probs, dtype=float)})
    if df.empty:
        return pd.Series(dtype=float)
    if method == "max":
        return df.groupby("user")["p"].max()
    if method == "topk":
        return df.groupby("user")["p"].apply(lambda s: float(np.sort(s.to_numpy())[::-1][:max(1, k)].mean()))
    if method == "consecutive":
        df = df.sort_values(["user", "ws"]).reset_index(drop=True)
        step = pd.Timedelta(days=window_days) if window_days else (df["ws"].diff()[df["ws"].diff() > pd.Timedelta(0)].min())
        widx = ((df["ws"] - df["ws"].min()) / step).round().astype("int64").to_numpy()
        run = _consecutive_min(df["user"].to_numpy(), widx, df["p"].to_numpy(), n)
        df["run"] = run
        score = df.groupby("user")["run"].max()
        return score.where(np.isfinite(score), 0.0)         # no n contiguous windows -> cannot show a sustained pattern
    raise ValueError(f"unknown aggregation '{method}'; choose from {AGG_METHODS}")


def user_truth(users: Sequence, unique_users: np.ndarray, user_is_insider: np.ndarray) -> np.ndarray:
    pos = np.searchsorted(unique_users, np.asarray(users))
    return user_is_insider[pos].astype(int)


def user_metrics(truth, score, threshold: float, ks=(5, 10, 15, 20, 30)) -> Dict[str, float]:
    t, s = np.asarray(truth).astype(int), np.asarray(score, dtype=float)
    pred = (s >= threshold).astype(int)
    tp = int((pred & t).sum())
    n_flag, n_pos = int(pred.sum()), int(t.sum())
    prec = tp / n_flag if n_flag else 0.0
    rec = tp / n_pos if n_pos else float("nan")
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return {"recall": round(rec, 4), "precision": round(prec, 4), "f1": round(f1, 4), "pr_auc": round(pr_auc(t, s), 4),
            "n_users": int(len(t)), "n_insiders": n_pos, "n_flagged": n_flag, "true_flagged": tp,
            **precision_recall_at_k(t, s, [k for k in ks if k <= len(t)])}


def evaluate_user_level(prep, idx_val: np.ndarray, idx_test: np.ndarray, p_val: np.ndarray, p_test: np.ndarray,
                        policy=None, methods: Optional[Dict[str, dict]] = None) -> Dict[str, dict]:
    """
    For each aggregation: score val/test users, tune the user threshold on VALIDATION users only,
    report val/test user-level metrics. Returns {method_name: {"threshold", "val", "test", "agg"}}.
    """
    pol = ThresholdPolicy.coerce(policy) if policy is not None else ThresholdPolicy(mode="budget", budget=0.10)
    uid = prep.features_df["user_id"].to_numpy()
    ws = prep.seq.window_starts
    methods = methods or {"max": {"method": "max"}, "topk": {"method": "topk", "k": 3},
                          "consecutive": {"method": "consecutive", "n": 3}}
    out = {}
    for name, cfg in methods.items():
        kw = {k: v for k, v in cfg.items() if k in ("method", "k", "n")}
        sv = aggregate_user_scores(uid[idx_val], ws[idx_val], p_val, window_days=prep.window_days, **kw)
        st = aggregate_user_scores(uid[idx_test], ws[idx_test], p_test, window_days=prep.window_days, **kw)
        tv = user_truth(sv.index.to_numpy(), prep.unique_users, prep.user_is_insider)
        tt = user_truth(st.index.to_numpy(), prep.unique_users, prep.user_is_insider)
        thr = choose_threshold(tv, sv.to_numpy(), pol)
        out[name] = {"threshold": round(float(thr), 6), "agg": kw,
                     "val": user_metrics(tv, sv.to_numpy(), thr), "test": user_metrics(tt, st.to_numpy(), thr)}
    return out


def flag_users(scored: pd.DataFrame, agg: dict, threshold: float, window_days: Optional[int] = None,
               user_col: str = "user", score_col: str = "score") -> pd.DataFrame:
    """Apply a stored aggregation + user threshold to scored windows (output of LSTMDetector.score)."""
    kw = {k: v for k, v in agg.items() if k in ("method", "k", "n")}
    s = aggregate_user_scores(scored[user_col].to_numpy(), scored["window_start"].to_numpy(),
                              scored[score_col].to_numpy(), window_days=window_days, **kw)
    out = s.sort_values(ascending=False).to_frame("user_score")
    out["flagged"] = out["user_score"] >= threshold
    return out


Writing /kaggle/working/insider_code/app/ml/user_level.py


In [9]:
%%writefile /kaggle/working/insider_code/app/ml/sequences.py
"""
Sequence builder, labels, user-level splits and the train-only scaler
(architecture.md sections 3.3 and 3.4).

Sample i <-> row i of the feature-window DataFrame: sample i is the T x F
matrix of the user's last T *time windows* ending at row i's window. Windows
in which the user had no events are real zero-activity windows (filled with
0), not skipped -- so the sequence is time-aligned. Windows before the
user's first observed window are unknown history: NaN here, 0 after scaling
(the population-mean equivalent).

Leakage rules enforced by this module:
  * splits are by USER (overlapping windows of one user never straddle splits)
  * the scaler is fit on train-split users' rows only
"""
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

from app.ml.feature_core import FEATURE_COLUMNS


@dataclass
class SequenceData:
    X: np.ndarray            # (N, T, F) raw features, NaN = no history yet
    lengths: np.ndarray      # (N,) number of real windows in the sequence (<= T)
    user_ids: np.ndarray     # (N,)
    window_starts: np.ndarray
    window_ends: np.ndarray
    seq_len: int
    feature_columns: List[str]


def _window_grid(features_df: pd.DataFrame):
    starts = pd.to_datetime(features_df["window_start"])
    ends = pd.to_datetime(features_df["window_end"])
    step = (ends - starts).iloc[0]
    origin = starts.min()
    idx = ((starts - origin) / step).round().astype("int64").to_numpy()
    return idx, step


def build_sequences(features_df: pd.DataFrame, seq_len: int = 7,
                    feature_columns: Optional[List[str]] = None) -> SequenceData:
    cols = list(feature_columns or FEATURE_COLUMNS)
    n, F = len(features_df), len(cols)
    X = np.full((n, seq_len, F), np.nan, dtype=np.float32)
    lengths = np.zeros(n, dtype=np.int32)
    if n == 0:
        return SequenceData(X, lengths, np.array([]), np.array([]), np.array([]), seq_len, cols)

    win_idx, _ = _window_grid(features_df)
    feats = features_df[cols].fillna(0).to_numpy(dtype=np.float32)
    users = features_df["user_id"].to_numpy()

    order = np.lexsort((win_idx, users))
    su = users[order]
    bounds = np.flatnonzero(np.r_[True, su[1:] != su[:-1], True])
    for a, b in zip(bounds[:-1], bounds[1:]):
        pos = order[a:b]
        w = win_idx[pos]
        first = w[0]
        dense = np.zeros((w[-1] - first + 1, F), dtype=np.float32)   # gaps = zero activity
        dense[w - first] = feats[pos]
        for p, wi in zip(pos, w):
            hi = wi - first + 1
            chunk = dense[max(0, hi - seq_len):hi]
            X[p, seq_len - len(chunk):] = chunk
            lengths[p] = len(chunk)

    return SequenceData(
        X=X, lengths=lengths, user_ids=users,
        window_starts=pd.to_datetime(features_df["window_start"]).to_numpy(),
        window_ends=pd.to_datetime(features_df["window_end"]).to_numpy(),
        seq_len=seq_len, feature_columns=cols,
    )


# ---------------------------------------------------------------- labels

def load_labels(path: str, dataset: Optional[str] = None) -> pd.DataFrame:
    """
    Loads a ground-truth file into columns: user, malicious (0/1), start, end.

    Accepts either
      * `user,malicious[,start,end]`  (this project's answers.csv), or
      * CERT's own insiders.csv (`dataset,scenario,details,user,start,end`) --
        every listed user is an insider; pass `dataset` (e.g. "r4.2") to keep
        only that release's rows.

    If start/end are present a sample is malicious only when its window
    overlaps the insider's malicious period ("labeled malicious if the user
    is a known insider in that period"); without them the whole user is
    labeled malicious.
    """
    df = pd.read_csv(path)
    if "user" not in df.columns:
        raise ValueError("labels file must have a 'user' column")
    if dataset and "dataset" in df.columns:
        df = df[df["dataset"].astype(str) == dataset]
    df = df.copy()
    df["user"] = df["user"].astype(str)
    if "malicious" in df.columns:
        df["malicious"] = df["malicious"].astype(str).str.lower().isin(["1", "true", "yes"]).astype(int)
    else:
        df["malicious"] = 1
    for c in ("start", "end"):
        df[c] = pd.to_datetime(df[c], errors="coerce") if c in df.columns else pd.NaT
    return df[["user", "malicious", "start", "end"]].reset_index(drop=True)


def label_samples(features_df: pd.DataFrame, labels: pd.DataFrame, user_code_by_id: Dict[int, str]) -> np.ndarray:
    codes = features_df["user_id"].map(user_code_by_id).to_numpy()
    ws = pd.to_datetime(features_df["window_start"]).to_numpy()
    we = pd.to_datetime(features_df["window_end"]).to_numpy()
    y = np.zeros(len(features_df), dtype=np.int64)
    for row in labels.itertuples(index=False):
        if not row.malicious:
            continue
        m = codes == row.user
        if pd.notna(row.start):
            m &= we > np.datetime64(row.start)
        if pd.notna(row.end):
            m &= ws <= np.datetime64(row.end)
        y[m] = 1
    return y


def insider_flags(unique_user_ids: np.ndarray, labels: pd.DataFrame, user_code_by_id: Dict[int, str]) -> np.ndarray:
    insiders = set(labels.loc[labels["malicious"] == 1, "user"])
    return np.array([user_code_by_id.get(int(u)) in insiders for u in unique_user_ids])


# ---------------------------------------------------------------- splits

@dataclass
class UserSplit:
    train: np.ndarray
    val: np.ndarray
    test: np.ndarray
    seed: int

    def masks(self, user_ids: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        return (np.isin(user_ids, self.train), np.isin(user_ids, self.val), np.isin(user_ids, self.test))


def _allocate(n: int, val_frac: float, test_frac: float, what: str) -> Tuple[int, int]:
    n_test = max(1, int(round(n * test_frac)))
    n_val = max(1, int(round(n * val_frac)))
    if n - n_val - n_test < 1:
        raise ValueError(
            f"Need at least 3 {what} users to give train/val/test each at least one "
            f"(got {n}). Add more users/insiders or use a larger dataset."
        )
    return n_val, n_test


def user_level_split(user_ids: np.ndarray, insider_mask: np.ndarray, val_frac: float = 0.15,
                     test_frac: float = 0.15, seed: int = 0) -> UserSplit:
    """
    Stratified split over *users*: insiders and non-insiders are each divided
    so every split contains at least one insider. No user appears in more than
    one split. Val/test keep the population's natural class ratio.
    """
    rng = np.random.RandomState(seed)
    parts = {"train": [], "val": [], "test": []}
    for flag, what in ((True, "insider"), (False, "non-insider")):
        ids = np.array(user_ids)[insider_mask == flag]
        ids = ids[rng.permutation(len(ids))]
        n_val, n_test = _allocate(len(ids), val_frac, test_frac, what)
        parts["val"].append(ids[:n_val])
        parts["test"].append(ids[n_val:n_val + n_test])
        parts["train"].append(ids[n_val + n_test:])
    return UserSplit(*(np.concatenate(parts[k]) for k in ("train", "val", "test")), seed=seed)


# ---------------------------------------------------------------- scaler

class FeatureScaler:
    """Per-feature standardization, fit on train-split rows only."""

    def __init__(self, mean: Optional[np.ndarray] = None, std: Optional[np.ndarray] = None):
        self.mean, self.std = mean, std

    def fit(self, features_df: pd.DataFrame, feature_columns: List[str]) -> "FeatureScaler":
        X = features_df[feature_columns].fillna(0).to_numpy(dtype=np.float64)
        self.mean = X.mean(axis=0)
        std = X.std(axis=0)
        self.std = np.where(std < 1e-8, 1.0, std)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        """Works on (N, F) or (N, T, F); NaN (no-history padding) becomes 0."""
        out = (X - self.mean) / self.std
        return np.nan_to_num(out, nan=0.0).astype(np.float32)

    def to_dict(self) -> dict:
        return {"mean": self.mean.tolist(), "std": self.std.tolist()}

    @classmethod
    def from_dict(cls, d: dict) -> "FeatureScaler":
        return cls(np.asarray(d["mean"], dtype=np.float64), np.asarray(d["std"], dtype=np.float64))


def flatten(X: np.ndarray) -> np.ndarray:
    return X.reshape(len(X), -1)


def unflatten(X: np.ndarray, seq_len: int, n_features: int) -> np.ndarray:
    return X.reshape(len(X), seq_len, n_features)

# ---------------------------------------------------------------- per-user baseline features (added)

COUNT_COLUMNS = [c for c in FEATURE_COLUMNS if c != "email_external_ratio"]


def engineer_features(features_df: pd.DataFrame, span: int = 15, min_periods: int = 5):
    """
    Adds one `<col>_z` column per count feature: how unusual this window is versus the SAME user's own
    previous `span` windows (log1p scale; strictly past data, shift(1) -> no leakage, independent of the
    train/val/test split). Then log1p-compresses the raw counts. Returns (df, feature_columns).
    Call it on the cached features_df BEFORE prepare_from_features, and again before scoring in the app.
    """
    df = features_df.copy()
    ws = pd.to_datetime(df["window_start"])
    step = (pd.to_datetime(df["window_end"]) - ws).iloc[0]
    df["_w"] = ((ws - ws.min()) / step).round().astype("int64")
    zcols = [f"{c}_z" for c in COUNT_COLUMNS]
    z_all = pd.DataFrame(0.0, index=df.index, columns=zcols)
    for _, g in df.groupby("user_id", sort=False):
        g = g.sort_values("_w")
        grid = range(int(g["_w"].iloc[0]), int(g["_w"].iloc[-1]) + 1)
        dense = np.log1p(g.set_index("_w")[COUNT_COLUMNS].reindex(grid, fill_value=0).astype(float))   # gaps = zero activity
        past = dense.shift(1)
        mu = past.rolling(span, min_periods=min_periods).mean()
        sd = past.rolling(span, min_periods=min_periods).std()
        z = ((dense - mu) / (sd.fillna(0) + 0.5)).clip(-5, 5).fillna(0.0)    # +0.5 keeps near-constant users from exploding
        z_all.loc[g.index, zcols] = z.loc[g["_w"].to_numpy()].to_numpy()
    df[zcols] = z_all
    df[COUNT_COLUMNS] = np.log1p(df[COUNT_COLUMNS].astype(float))
    return df.drop(columns="_w"), list(FEATURE_COLUMNS) + zcols


# ---------------------------------------------------------------- k-fold user splits (added)

def kfold_user_splits(user_ids: np.ndarray, insider_mask: np.ndarray, k: int = 7, seed: int = 0) -> List[UserSplit]:
    """
    Stratified k-fold over USERS. Fold i is the TEST set, fold i+1 is VALIDATION, the rest is TRAIN, so across the
    k splits every user (and every insider) is a test user exactly once. k=7 with 70 insiders -> 10 insiders/fold.
    """
    rng = np.random.RandomState(seed)
    folds = [[] for _ in range(k)]
    for flag in (True, False):
        ids = np.array(user_ids)[insider_mask == flag]
        ids = ids[rng.permutation(len(ids))]
        for i, chunk in enumerate(np.array_split(ids, k)):
            folds[i].append(chunk)
    folds = [np.concatenate(f) for f in folds]
    out = []
    for i in range(k):
        v = (i + 1) % k
        train = np.concatenate([folds[j] for j in range(k) if j not in (i, v)])
        out.append(UserSplit(train=train, val=folds[v], test=folds[i], seed=i))
    return out


Writing /kaggle/working/insider_code/app/ml/sequences.py


In [10]:
%%writefile /kaggle/working/insider_code/app/ml/torch_utils.py
"""Small torch helpers shared by the LSTM and DeepSMOTE modules."""
import random

import numpy as np


def require_torch():
    try:
        import torch  # noqa: F401
    except ImportError as e:   # pragma: no cover
        raise ImportError(
            "PyTorch is required for the LSTM / DeepSMOTE features. "
            "Install it with `pip install torch` (see README)."
        ) from e


def resolve_device(device=None):
    import torch
    if device is not None:
        return torch.device(device)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int) -> None:
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

Writing /kaggle/working/insider_code/app/ml/torch_utils.py


In [11]:
%%writefile /kaggle/working/insider_code/app/ml/sampling/nm2.py
"""
NM-2 (NearMiss version 2) -- the paper's best sampler and the baseline to beat.
Distance-based undersampler: keeps the majority samples whose summed distance
to their `n_neighbors` FARTHEST minority samples is smallest, until the
minority:majority ratio reaches `ratio` (1.0 = balanced, 0.05 = 1:20). Works directly on flattened T x F vectors.

This is a memory-bounded reimplementation of imbalanced-learn's
NearMiss(version=2). imblearn materializes the full (n_majority x n_minority)
distance matrix (twice: distances and indices); on a large dataset that is
tens of GB. Here majority rows are processed in chunks so peak memory is
bounded by `max_chunk_elements` regardless of dataset size. It selects the
same samples as imblearn (verified in tests/test_sampling.py).
"""
import numpy as np
from sklearn.metrics import pairwise_distances


def resample_nm2(X: np.ndarray, y: np.ndarray, n_neighbors: int = 3, max_chunk_elements: int = 20_000_000,
                 ratio: float = 1.0):
    y = np.asarray(y).astype(int)
    min_idx, maj_idx = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    n_min, n_maj = len(min_idx), len(maj_idx)
    if n_min == 0 or n_maj == 0:
        raise ValueError("NM-2 needs both classes present in the training split.")
    k = min(n_neighbors, n_min)
    X_min = np.asarray(X[min_idx], dtype=np.float64)

    chunk = max(1, max_chunk_elements // n_min)
    scores = np.empty(n_maj, dtype=np.float64)
    for i in range(0, n_maj, chunk):
        d = pairwise_distances(np.asarray(X[maj_idx[i:i + chunk]], dtype=np.float64), X_min)   # (chunk, n_min)
        scores[i:i + chunk] = np.partition(d, n_min - k, axis=1)[:, n_min - k:].sum(axis=1)   # sum of k farthest

    # keep the n_min/ratio majority samples nearest to the minority class (ratio=1.0 -> fully balanced)
    n_keep = min(n_maj, max(n_min, int(round(n_min / ratio))))
    keep_maj = maj_idx[np.argsort(scores, kind="stable")[:n_keep]]
    idx = np.concatenate([keep_maj, min_idx])
    info = {"kept": int(len(idx)), "minority": int(n_min), "majority": int(len(keep_maj)), "n_neighbors": int(k), "ratio": ratio}
    return X[idx], y[idx], info

Writing /kaggle/working/insider_code/app/ml/sampling/nm2.py


In [12]:
%%writefile /kaggle/working/insider_code/app/ml/sampling/rus.py
"""
Random undersampling (RUS). Diagnostic control for NM-2: same number of majority
windows kept, but chosen at random instead of "closest to the minority class".
If RUS@r beats NM-2@r, the problem is NM-2's biased selection; if both are
worse than `none`, discarding normal data is the problem.
"""
import numpy as np


def resample_rus(X: np.ndarray, y: np.ndarray, seed: int = 0, ratio: float = 1.0):
    y = np.asarray(y).astype(int)
    min_idx, maj_idx = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    n_min, n_maj = len(min_idx), len(maj_idx)
    if n_min == 0 or n_maj == 0:
        raise ValueError("RUS needs both classes present in the training split.")
    n_keep = min(n_maj, max(n_min, int(round(n_min / ratio))))
    keep = np.random.RandomState(seed).choice(maj_idx, size=n_keep, replace=False)
    idx = np.sort(np.concatenate([keep, min_idx]))
    return X[idx], y[idx], {"kept": int(len(idx)), "minority": int(n_min), "majority": int(n_keep)}


Writing /kaggle/working/insider_code/app/ml/sampling/rus.py


In [13]:
%%writefile /kaggle/working/insider_code/app/ml/sampling/deepsmote.py
"""
DeepSMOTE for flattened behavioral sequences (architecture.md section 3.5).

DeepSMOTE (Dablain, Krawczyk & Chawla, 2022) oversamples in a learned latent
space: train an encoder/decoder, run SMOTE-style interpolation between
minority latents, then decode the new latents back to input space. The
original targets images (convolutional AE); here the inputs are flattened
T x F vectors so the encoder/decoder are MLPs.

Training loss = reconstruction MSE
              + mix_weight * latent-mix consistency loss

The second term is this project's adaptation, NOT a verbatim copy of the
paper's permutation loss: for pairs of same-class samples we interpolate in
latent space, decode, re-encode, and require the re-encoded latent to land
where we interpolated. It plays the same role -- keeping in-class latent
interpolations decodable into plausible samples.

Batches are class-balanced (minority drawn as often as majority) because the
autoencoder would otherwise barely see the few insider samples.

Stability: DeepSMOTE can be unreliable on very small minority classes. After
fitting we compute `recon_ratio` (reconstruction MSE / MSE of predicting the
feature mean, overall and on the minority class). If it is non-finite or too
high, `info["converged"]` is False. Callers treat that as a *finding*
("did not train reliably") rather than forcing the sampler to win.
"""
from typing import Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from sklearn.neighbors import NearestNeighbors

from app.ml.torch_utils import resolve_device, set_seed


class _AutoEncoder(nn.Module):
    def __init__(self, d_in: int, hidden: Tuple[int, int], latent: int):
        super().__init__()
        h1, h2 = hidden
        self.enc = nn.Sequential(nn.Linear(d_in, h1), nn.ReLU(), nn.Linear(h1, h2), nn.ReLU(), nn.Linear(h2, latent))
        self.dec = nn.Sequential(nn.Linear(latent, h2), nn.ReLU(), nn.Linear(h2, h1), nn.ReLU(), nn.Linear(h1, d_in))


def _same_class_partner(y: torch.Tensor) -> torch.Tensor:
    """For each batch element, the index of a random same-class element (itself if alone)."""
    partner = torch.arange(len(y), device=y.device)
    for c in torch.unique(y):
        idx = torch.nonzero(y == c, as_tuple=True)[0]
        partner[idx] = idx[torch.randint(len(idx), (len(idx),), device=y.device)]
    return partner


def resample_deepsmote(
    X: np.ndarray, y: np.ndarray, seed: int = 0, latent_dim: int = 16,
    hidden: Tuple[int, int] = (128, 64), epochs: int = 60, batch_size: int = 64,
    lr: float = 1e-3, k_neighbors: int = 5, target_ratio: float = 1.0,
    mix_weight: float = 0.1, max_recon_ratio: float = 0.7,
    max_synthetic: Optional[int] = 200_000, max_steps_per_epoch: int = 200, device=None,
    cache: Optional[dict] = None,
):
    """
    Returns (X_resampled, y_resampled, info). `target_ratio` is the desired
    minority:majority ratio after oversampling (1.0 = fully balanced).

    `cache`: optional dict. The autoencoder depends only on (X, y, seed), not on
    `target_ratio`, so when several ratios are compared on the same split pass
    the same dict each time and the autoencoder is trained once.
    """
    y = np.asarray(y).astype(int)
    n_min, n_maj = int((y == 1).sum()), int((y == 0).sum())
    if n_min == 0 or n_maj == 0:
        raise ValueError("DeepSMOTE needs both classes present in the training split.")

    set_seed(seed)
    device = resolve_device(device)

    if cache is not None and "state" in cache:
        Z, health = cache["state"]["Z"], cache["state"]["health"]
        model = cache["state"]["model"]
        last_loss, recon_ratio, recon_ratio_min, converged = health
    else:
        Xt = torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32)).to(device)
        yt = torch.from_numpy(y).to(device)
        n, d = Xt.shape

        model = _AutoEncoder(d, hidden, latent_dim).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        mse = nn.MSELoss()

        # class-balanced batch sampling
        w = np.where(y == 1, 0.5 / n_min, 0.5 / n_maj)
        rng = np.random.RandomState(seed)
        steps = max(1, min(max_steps_per_epoch, int(np.ceil(n / batch_size))))   # cap keeps huge datasets tractable
        last_loss = float("nan")
        for _ in range(epochs):
            model.train()
            for _ in range(steps):
                idx = torch.from_numpy(rng.choice(n, size=min(batch_size, n), p=w)).to(device)
                xb, yb = Xt[idx], yt[idx]
                z = model.enc(xb)
                loss = mse(model.dec(z), xb)
                if mix_weight > 0 and len(idx) > 1:
                    partner = _same_class_partner(yb)
                    lam = torch.rand(len(idx), 1, device=device)
                    z_mix = z + lam * (z[partner] - z)
                    loss = loss + mix_weight * mse(model.enc(model.dec(z_mix)), z_mix.detach())
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                opt.step()
                last_loss = float(loss.detach().cpu())

        # ---- health check + latent codes
        model.eval()
        with torch.no_grad():
            Z = model.enc(Xt)
            recon = model.dec(Z)
            err = ((recon - Xt) ** 2).mean(dim=1).cpu().numpy()
        Xn = Xt.cpu().numpy()
        base = ((Xn - Xn.mean(axis=0)) ** 2).mean(axis=1)
        recon_ratio = float(err.mean() / max(base.mean(), 1e-12))
        recon_ratio_min = float(err[y == 1].mean() / max(base[y == 1].mean(), 1e-12)) if n_min else float("nan")
        converged = bool(np.isfinite(last_loss) and np.isfinite(recon_ratio) and np.isfinite(recon_ratio_min)
                         and recon_ratio <= max_recon_ratio and recon_ratio_min <= max_recon_ratio + 0.2)
        if cache is not None:
            cache["state"] = {"Z": Z, "model": model, "health": (last_loss, recon_ratio, recon_ratio_min, converged)}

    # ---- SMOTE in latent space over the minority class
    n_new = max(0, int(round(target_ratio * n_maj)) - n_min)
    if max_synthetic is not None:
        n_new = min(n_new, max_synthetic)
    info = {
        "converged": converged, "recon_ratio": round(recon_ratio, 4),
        "recon_ratio_minority": round(recon_ratio_min, 4), "final_loss": round(last_loss, 6),
        "n_synthetic": n_new, "epochs": epochs, "latent_dim": latent_dim,
    }
    if n_new == 0:
        return X, y, info

    Zm = Z[torch.from_numpy(y == 1).to(device)].cpu().numpy()
    gen = np.random.RandomState(seed + 1)
    if len(Zm) >= 2:
        k = min(k_neighbors, len(Zm) - 1)
        nn_idx = NearestNeighbors(n_neighbors=k + 1).fit(Zm).kneighbors(Zm, return_distance=False)[:, 1:]
        base_i = gen.randint(len(Zm), size=n_new)
        nb_i = nn_idx[base_i, gen.randint(k, size=n_new)]
        lam = gen.rand(n_new, 1)
        z_new = Zm[base_i] + lam * (Zm[nb_i] - Zm[base_i])
    else:                                    # a single insider sample: jitter it
        info["single_minority_fallback"] = True
        z_new = Zm[np.zeros(n_new, dtype=int)] + gen.normal(0, 0.05 * float(Z.std().cpu()), size=(n_new, Zm.shape[1]))

    with torch.no_grad():
        X_new = model.dec(torch.from_numpy(z_new.astype(np.float32)).to(device)).cpu().numpy()
    if not np.isfinite(X_new).all():
        info["converged"] = False
        X_new = np.nan_to_num(X_new, nan=0.0, posinf=0.0, neginf=0.0)

    X_res = np.concatenate([np.asarray(X, dtype=np.float32), X_new.astype(np.float32)])
    y_res = np.concatenate([y, np.ones(n_new, dtype=int)])
    return X_res, y_res, info

Writing /kaggle/working/insider_code/app/ml/sampling/deepsmote.py


In [14]:
%%writefile /kaggle/working/insider_code/app/ml/detection.py
"""
Detection pipeline.

Detector interface (`BaseDetector`, `get_detector`) + the five classical
detectors. The LSTM detector (app/ml/lstm_detector.py) implements the same
interface. Two things live here, mirroring the two layers from the paper this project
extends ("Malicious Insider Threat Detection Using Variation of Sampling
Methods for Anomaly Detection in Cloud Environment"):

1. `run_unsupervised_detection` — production path. No ground-truth labels
   needed. Runs one of the paper's Layer-2 anomaly detectors (OCSVM, IF,
   Robust Covariance, LOF, KNN-distance) over the feature windows and
   returns a normalized anomaly score per row. This is what powers the
   live dashboard.

2. `run_sampling_benchmark` — evaluation path. If you have labeled data
   (CERT ships an answer key identifying which users were scripted as
   malicious), this reproduces the paper's Layer-1 comparison: try each
   sampling technique (ROS/SMOTE/ADASYN/RUS/NM-1/NM-2/T-L/ENN), score with
   an SVM, and report precision/recall/f-score/accuracy so you can compare
   your results against the paper's reported numbers (NM-2 + OCSVM: recall
   100%, f-score 78.72%, accuracy 82.46%).
"""
from abc import ABC, abstractmethod
from typing import Tuple

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM, SVC
from sklearn.ensemble import IsolationForest
from sklearn.covariance import EllipticEnvelope
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

from app.ml.feature_core import FEATURE_COLUMNS

CLASSICAL_DETECTORS = {"OCSVM", "IF", "RC", "LOF", "KNN"}
DETECTORS = CLASSICAL_DETECTORS | {"LSTM"}   # LSTM lives in app/ml/lstm_detector.py (needs PyTorch)
SAMPLING_TECHNIQUES = {"ROS", "SMOTE", "ADASYN", "RUS", "NM-1", "NM-2", "T-L", "ENN", "none"}


def _scale(features_df: pd.DataFrame) -> Tuple[np.ndarray, StandardScaler]:
    X = features_df[FEATURE_COLUMNS].fillna(0).to_numpy(dtype=float)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, scaler


class BaseDetector(ABC):
    """
    Common detector interface. Every detector -- the five classical ones and
    the LSTM -- takes the output of build_feature_windows and returns a copy
    with three added columns:

        raw_score     detector's native output (higher = more anomalous)
        score         normalized 0-1, higher = more anomalous
        is_anomalous  0/1 flag

    so they are swappable at runtime (the pipeline router only ever calls
    `get_detector(name).score(features_df)`).
    """
    name: str = "base"

    @abstractmethod
    def score(self, features_df: pd.DataFrame) -> pd.DataFrame:
        ...


class ClassicalDetector(BaseDetector):
    """OCSVM / IF / RC / LOF / KNN. Unsupervised: no labels needed."""

    def __init__(self, name: str, contamination: float = 0.05):
        if name not in CLASSICAL_DETECTORS:
            raise ValueError(f"Unknown detector '{name}'. Choose from {sorted(DETECTORS)}")
        self.name = name
        self.contamination = contamination
        self._model = None

    # -- transductive path: fit and score the same data (what powers the live dashboard)
    def score(self, features_df: pd.DataFrame) -> pd.DataFrame:
        detector, contamination = self.name, self.contamination
        if features_df.empty:
            return features_df.assign(raw_score=[], score=[], is_anomalous=[])

        X, _ = _scale(features_df)
        out = features_df.copy()

        if detector == "OCSVM":
            model = OneClassSVM(kernel="rbf", gamma="auto", nu=max(contamination, 0.01))
            model.fit(X)
            raw = -model.decision_function(X)   # flip sign: higher = more anomalous
            pred = model.predict(X)             # -1 anomaly, 1 normal

        elif detector == "IF":
            model = IsolationForest(n_estimators=100, contamination=contamination, random_state=42)
            model.fit(X)
            raw = -model.decision_function(X)
            pred = model.predict(X)

        elif detector == "RC":
            model = EllipticEnvelope(contamination=contamination, random_state=42)
            model.fit(X)
            raw = -model.decision_function(X)
            pred = model.predict(X)

        elif detector == "LOF":
            n_neighbors = min(35, max(2, len(X) - 1))
            model = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=contamination)
            pred = model.fit_predict(X)
            raw = -model.negative_outlier_factor_

        else:  # KNN distance-based
            n_neighbors = min(10, max(2, len(X) - 1))
            nn = NearestNeighbors(n_neighbors=n_neighbors)
            nn.fit(X)
            distances, _ = nn.kneighbors(X)
            raw = distances.mean(axis=1)
            threshold = np.quantile(raw, 1 - contamination)
            pred = np.where(raw >= threshold, -1, 1)

        # normalize raw score to 0-1 for display
        raw = np.asarray(raw, dtype=float)
        rmin, rmax = raw.min(), raw.max()
        score = (raw - rmin) / (rmax - rmin) if rmax > rmin else np.zeros_like(raw)

        out["raw_score"] = raw
        out["score"] = score
        out["is_anomalous"] = (pred == -1).astype(int)
        return out

    # -- inductive path: fit on train rows, score unseen rows (used by the evaluation
    #    harness so classical detectors are compared on the same user-level split)
    def fit(self, X_train: np.ndarray) -> "ClassicalDetector":
        n, c = len(X_train), self.contamination
        if self.name == "OCSVM":
            self._model = OneClassSVM(kernel="rbf", gamma="auto", nu=max(c, 0.01)).fit(X_train)
        elif self.name == "IF":
            self._model = IsolationForest(n_estimators=100, contamination=c, random_state=42).fit(X_train)
        elif self.name == "RC":
            self._model = EllipticEnvelope(contamination=c, random_state=42).fit(X_train)
        elif self.name == "LOF":
            self._model = LocalOutlierFactor(n_neighbors=min(35, max(2, n - 1)), contamination=c, novelty=True).fit(X_train)
        else:
            self._model = NearestNeighbors(n_neighbors=min(10, max(2, n - 1))).fit(X_train)
        return self

    def raw_scores(self, X: np.ndarray) -> np.ndarray:
        """Anomaly score for unseen rows, higher = more anomalous."""
        if self._model is None:
            raise RuntimeError("call fit() first")
        if self.name in ("OCSVM", "IF", "RC"):
            return -self._model.decision_function(X)
        if self.name == "LOF":
            return -self._model.score_samples(X)
        dist, _ = self._model.kneighbors(X)
        return dist.mean(axis=1)


def get_detector(name: str, contamination: float = 0.05, **kwargs) -> BaseDetector:
    """Registry: name -> detector instance. 'LSTM' requires `artifact_dir=`."""
    if name == "LSTM":
        from app.ml.lstm_detector import LSTMDetector    # lazy: keeps PyTorch optional
        return LSTMDetector(**kwargs)
    return ClassicalDetector(name, contamination=contamination)


def run_unsupervised_detection(
    features_df: pd.DataFrame,
    detector: str = "OCSVM",
    contamination: float = 0.05,
) -> pd.DataFrame:
    """
    Scores each row of `features_df` (output of build_feature_windows) with
    the chosen classical anomaly detector. Adds `raw_score`, `score`
    (normalized 0-1, higher = more anomalous) and `is_anomalous` (0/1).
    Kept as a thin wrapper for backwards compatibility.
    """
    return ClassicalDetector(detector, contamination).score(features_df)


def compute_top_reasons(features_df: pd.DataFrame, row_index: int, top_n: int = 3) -> dict:
    """
    Lightweight explainability: for the given row, compute each feature's
    z-score relative to the whole population in `features_df` and return
    the top_n features with the highest positive deviation. This is what
    populates "flagged due to: ..." in the alert UI.
    """
    X = features_df[FEATURE_COLUMNS].fillna(0)
    means, stds = X.mean(), X.std().replace(0, 1)
    z = (X.loc[row_index] - means) / stds
    top = z.sort_values(ascending=False).head(top_n)
    return {feat: round(float(val), 2) for feat, val in top.items() if val > 0}


def _apply_sampler(technique: str, X: np.ndarray, y: np.ndarray):
    if technique == "none":
        return X, y

    if technique == "ROS":
        from imblearn.over_sampling import RandomOverSampler
        sampler = RandomOverSampler(random_state=42)
    elif technique == "SMOTE":
        from imblearn.over_sampling import SMOTE
        sampler = SMOTE(random_state=42, k_neighbors=min(5, y.sum() - 1) if y.sum() > 1 else 1)
    elif technique == "ADASYN":
        from imblearn.over_sampling import ADASYN
        sampler = ADASYN(random_state=42, n_neighbors=min(5, y.sum() - 1) if y.sum() > 1 else 1)
    elif technique == "RUS":
        from imblearn.under_sampling import RandomUnderSampler
        sampler = RandomUnderSampler(random_state=42)
    elif technique == "NM-1":
        from imblearn.under_sampling import NearMiss
        sampler = NearMiss(version=1)
    elif technique == "NM-2":
        from imblearn.under_sampling import NearMiss
        sampler = NearMiss(version=2)
    elif technique == "T-L":
        from imblearn.under_sampling import TomekLinks
        sampler = TomekLinks()
    elif technique == "ENN":
        from imblearn.under_sampling import EditedNearestNeighbours
        sampler = EditedNearestNeighbours()
    else:
        raise ValueError(f"Unknown sampling technique '{technique}'. Choose from {sorted(SAMPLING_TECHNIQUES)}")

    return sampler.fit_resample(X, y)


def run_sampling_benchmark(
    features_df: pd.DataFrame,
    labels: pd.Series,
    sampling_technique: str = "NM-2",
) -> dict:
    """
    Evaluation-only path (needs ground-truth `labels`, aligned by index with
    `features_df`, 1 = known malicious insider, 0 = genuine). Reproduces the
    paper's Layer 1: apply a sampling technique, train an SVM, report
    precision/recall/f-score/accuracy on a held-out split.
    """
    X, _ = _scale(features_df)
    y = labels.to_numpy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y if y.sum() > 1 else None
    )

    X_res, y_res = _apply_sampler(sampling_technique, X_train, y_train)

    # class_weight="balanced" compensates for any residual imbalance the
    # sampler didn't fully correct; gamma="scale" adapts to this project's
    # feature count/variance rather than reusing the paper's fixed gamma,
    # which was tuned for their specific feature set.
    clf = SVC(kernel="rbf", gamma="scale", C=1.0, class_weight="balanced")
    clf.fit(X_res, y_res)
    y_pred = clf.predict(X_test)

    return {
        "sampling_technique": sampling_technique,
        "precision": round(float(precision_score(y_test, y_pred, zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, y_pred, zero_division=0)), 4),
        "f_score": round(float(f1_score(y_test, y_pred, zero_division=0)), 4),
        "accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
        "train_size": len(y_res),
        "test_size": len(y_test),
    }

Writing /kaggle/working/insider_code/app/ml/detection.py


In [15]:
%%writefile /kaggle/working/insider_code/app/ml/lstm_detector.py
"""
LSTM classifier + detector (architecture.md sections 3.6 and 4.1).

Supervised sequence model: input is a T x F matrix (last T time windows of
F behavioral features), output is the probability that the user is
malicious in that period. It is trained OFFLINE (app/ml/train.py) and saved
as an artifact; the API only ever loads the artifact and scores.

Why supervised rather than an autoencoder: Layer 1 exists to fix the class
imbalance a classifier trains on. An autoencoder trained on normal behavior
never sees the minority class, which would make the sampling stage largely
irrelevant.

Implementation note: the architecture doc says "sigmoid output, binary
cross-entropy loss". We train with BCEWithLogitsLoss (sigmoid + BCE fused, the
numerically stable form) and apply the sigmoid at inference.
"""
import copy
import json
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from app.ml.detection import BaseDetector
from app.ml.metrics import pr_auc
from app.ml.sequences import FeatureScaler, build_sequences
from app.ml.torch_utils import resolve_device, set_seed

ARTIFACT_FORMAT_VERSION = 1


@dataclass
class LSTMConfig:
    hidden_size: int = 64
    num_layers: int = 2
    dropout: float = 0.3
    lr: float = 1e-3
    weight_decay: float = 1e-5
    epochs: int = 40
    batch_size: int = 128
    patience: Optional[int] = 8     # early stopping on validation PR-AUC; None = fixed epochs

    def scaled_for(self, n_train: int) -> "LSTMConfig":
        """Larger batches for very large train sets (>~100k samples) so epochs stay tractable; unchanged for small data."""
        import dataclasses
        return dataclasses.replace(self, batch_size=max(self.batch_size, min(1024, n_train // 500)))

    @classmethod
    def evaluator(cls) -> "LSTMConfig":
        """Small LSTM used to compare samplers in Layer 1 (1 layer, ~32 units, few epochs)."""
        return cls(hidden_size=32, num_layers=1, dropout=0.2, epochs=20, batch_size=64, patience=4)

    @classmethod
    def production(cls) -> "LSTMConfig":
        return cls()


class LSTMClassifier(nn.Module):
    def __init__(self, n_features: int, hidden_size: int = 64, num_layers: int = 2, dropout: float = 0.3):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:      # x: (N, T, F) -> logits (N,)
        out, _ = self.lstm(x)
        return self.head(self.drop(out[:, -1, :])).squeeze(-1)


class BiLSTMAttnClassifier(nn.Module):
    """Bidirectional LSTM + additive attention over the T windows (ensemble member; see app/ml/ensemble.py)."""
    def __init__(self, n_features: int, hidden_size: int = 64, num_layers: int = 1, dropout: float = 0.3):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_size, num_layers, batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.attn = nn.Linear(2 * hidden_size, 1)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(2 * hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:      # x: (N, T, F) -> logits (N,)
        out, _ = self.lstm(x)                                  # (N, T, 2H)
        w = torch.softmax(self.attn(out).squeeze(-1), dim=1)   # (N, T) attention over windows
        return self.head(self.drop((out * w.unsqueeze(-1)).sum(1))).squeeze(-1)


def build_model(arch: str, n_features: int, cfg: "LSTMConfig") -> nn.Module:
    if arch == "lstm":
        return LSTMClassifier(n_features, cfg.hidden_size, cfg.num_layers, cfg.dropout)
    if arch == "bilstm_attn":
        return BiLSTMAttnClassifier(n_features, cfg.hidden_size, cfg.num_layers, cfg.dropout)
    raise ValueError(f"unknown architecture '{arch}'")


def predict_proba(model: nn.Module, X: np.ndarray, device=None, batch_size: int = 4096) -> np.ndarray:
    device = resolve_device(device)
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.from_numpy(np.ascontiguousarray(X[i:i + batch_size], dtype=np.float32)).to(device)
            out.append(torch.sigmoid(model(xb)).cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)


def train_lstm(X: np.ndarray, y: np.ndarray, cfg: LSTMConfig, seed: int = 0,
               pos_weight: Optional[float] = None, X_val: Optional[np.ndarray] = None,
               y_val: Optional[np.ndarray] = None, device=None, arch: str = "lstm"):
    """
    Trains on (N, T, F) scaled sequences. If a validation set is given, the
    epoch with the best validation PR-AUC is restored (and `cfg.patience`
    enables early stopping). Returns (model, history).
    """
    set_seed(seed)
    device = resolve_device(device)
    model = build_model(arch, X.shape[2], cfg).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    pw = torch.tensor(float(pos_weight), device=device) if pos_weight else None
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)

    Xt = torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32))
    yt = torch.from_numpy(np.asarray(y, dtype=np.float32))
    if device.type == "cuda" and Xt.numel() * 4 < 2_000_000_000:   # whole train set fits on the GPU: no per-batch copies
        Xt, yt = Xt.to(device), yt.to(device)
    gen = torch.Generator().manual_seed(seed)
    use_val = X_val is not None and y_val is not None and int(np.sum(y_val)) > 0

    best_state, best, bad, history = None, -np.inf, 0, []
    for epoch in range(cfg.epochs):
        model.train()
        perm = torch.randperm(len(Xt), generator=gen).to(Xt.device)
        total = 0.0
        for i in range(0, len(perm), cfg.batch_size):
            idx = perm[i:i + cfg.batch_size]
            xb, yb = Xt[idx].to(device), yt[idx].to(device)
            loss = loss_fn(model(xb), yb)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += float(loss.detach().cpu()) * len(idx)
        rec = {"epoch": epoch + 1, "train_loss": round(total / len(Xt), 5)}
        if use_val:
            m = pr_auc(y_val, predict_proba(model, X_val, device))
            rec["val_pr_auc"] = round(m, 4)
            if m > best + 1e-6:
                best, bad = m, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                bad += 1
        history.append(rec)
        if use_val and cfg.patience and bad >= cfg.patience:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


# ------------------------------------------------------------------ artifact

@dataclass
class LSTMArtifact:
    """Everything the API needs to score: weights, scaler, threshold, sampler choice."""
    state_dict: dict
    config: dict
    scaler: FeatureScaler
    threshold: float
    sampler: str
    seq_len: int
    window_days: int
    feature_columns: list
    dataset: Optional[str] = None
    seed: int = 0
    metrics: Optional[dict] = None
    user_threshold: Optional[float] = None     # user-level alert threshold (see user_level.py)
    user_agg: Optional[dict] = None            # how window probabilities are aggregated per user

    def save(self, directory) -> Path:
        d = Path(directory)
        d.mkdir(parents=True, exist_ok=True)
        torch.save(self.state_dict, d / "model.pt")
        meta = {
            "format_version": ARTIFACT_FORMAT_VERSION, "config": self.config,
            "scaler": self.scaler.to_dict(), "threshold": self.threshold, "sampler": self.sampler,
            "seq_len": self.seq_len, "window_days": self.window_days,
            "feature_columns": self.feature_columns, "dataset": self.dataset,
            "seed": self.seed, "metrics": self.metrics, "torch_version": torch.__version__,
            "user_threshold": self.user_threshold, "user_agg": self.user_agg,
        }
        (d / "meta.json").write_text(json.dumps(meta, indent=2))
        return d

    @classmethod
    def load(cls, directory, map_location="cpu") -> "LSTMArtifact":
        d = Path(directory)
        if not (d / "meta.json").exists() or not (d / "model.pt").exists():
            raise FileNotFoundError(f"No LSTM artifact found in {d}")
        meta = json.loads((d / "meta.json").read_text())
        if meta.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(f"Unsupported artifact format {meta.get('format_version')}")
        state = torch.load(d / "model.pt", map_location=map_location, weights_only=True)
        return cls(
            state_dict=state, config=meta["config"], scaler=FeatureScaler.from_dict(meta["scaler"]),
            threshold=float(meta["threshold"]), sampler=meta["sampler"], seq_len=int(meta["seq_len"]),
            window_days=int(meta["window_days"]), feature_columns=list(meta["feature_columns"]),
            dataset=meta.get("dataset"), seed=int(meta.get("seed", 0)), metrics=meta.get("metrics"),
            user_threshold=meta.get("user_threshold"), user_agg=meta.get("user_agg"),
        )

    def build_model(self, device=None) -> LSTMClassifier:
        m = LSTMClassifier(len(self.feature_columns), self.config["hidden_size"],
                           self.config["num_layers"], self.config["dropout"])
        m.load_state_dict(self.state_dict)
        return m.to(resolve_device(device)).eval()


# ------------------------------------------------------------------ detector

class LSTMDetector(BaseDetector):
    """
    Drop-in detector: same interface as the five classical ones (see
    BaseDetector). Scores REAL, non-resampled data -- resampling only ever
    touches the training split.

      score        = malicious probability (0-1)
      raw_score    = logit
      is_anomalous = probability >= the threshold tuned on the validation set
    """
    name = "LSTM"

    def __init__(self, artifact_dir=None, artifact: Optional[LSTMArtifact] = None, device=None):
        if artifact is None:
            if artifact_dir is None:
                raise ValueError("LSTMDetector needs artifact_dir= (a trained model directory)")
            artifact = LSTMArtifact.load(artifact_dir)
        self.artifact = artifact
        self.device = resolve_device(device)
        self.model = artifact.build_model(self.device)

    @property
    def threshold(self) -> float:
        return self.artifact.threshold

    def score(self, features_df: pd.DataFrame) -> pd.DataFrame:
        a = self.artifact
        if features_df.empty:
            return features_df.assign(raw_score=[], score=[], is_anomalous=[])
        step_days = (pd.to_datetime(features_df["window_end"].iloc[0]) - pd.to_datetime(features_df["window_start"].iloc[0])).days
        if step_days != a.window_days:
            raise ValueError(
                f"This model was trained on {a.window_days}-day windows but the data uses "
                f"{step_days}-day windows. Rebuild features with window_days={a.window_days}."
            )
        seq = build_sequences(features_df, a.seq_len, a.feature_columns)
        prob = predict_proba(self.model, a.scaler.transform(seq.X), self.device)
        p = np.clip(prob.astype(float), 1e-7, 1 - 1e-7)
        out = features_df.copy()
        out["raw_score"] = np.log(p / (1 - p))
        out["score"] = prob.astype(float)
        out["is_anomalous"] = (prob >= a.threshold).astype(int)
        return out
    def score_users(self, features_df: pd.DataFrame) -> pd.DataFrame:
        """One row per user: aggregated score + flagged, using the artifact's stored aggregation and user threshold."""
        from app.ml.user_level import flag_users
        a = self.artifact
        if a.user_threshold is None or a.user_agg is None:
            raise ValueError("This artifact has no user-level threshold; retrain with the current train.py")
        return flag_users(self.score(features_df), a.user_agg, a.user_threshold, a.window_days, user_col="user_id")


Writing /kaggle/working/insider_code/app/ml/lstm_detector.py


In [16]:
%%writefile /kaggle/working/insider_code/app/ml/train.py
"""
Offline training job (architecture.md sections 3.5, 3.6, 4.2).

    python -m app.ml.train --mode train    --answers seed/synthetic_cert/answers.csv --window-days 2
    python -m app.ml.train --mode compare  ...     # Layer 1 only (sampler comparison)
    python -m app.ml.train --mode evaluate ...     # full evaluation table (section 5)

Training NEVER happens inside an API request: the API launches this module as
a separate process (see app/jobs.py) and the job reports back through its
DetectionRun row (status, metrics, config).

Pipeline
--------
  events -> 12 features/user/window -> T x 12 sequences -> USER-LEVEL split
  Layer 1  every candidate sampler (NM-2 / DeepSMOTE / none / class_weight)
           is applied to the TRAIN split only, a small LSTM is trained with
           identical seeds, and candidates are ranked on the VALIDATION split
           (test is only reported, never used to choose). Repeated over
           several seeds -> mean +/- std.
  Layer 2  the production LSTM is trained once on the winning sampler's
           output; the alert threshold is tuned on validation (recall
           first) and frozen; test metrics are reported; the artifact
           (weights, scaler, threshold, sampler) is saved.
"""
import argparse
import logging
import sys
import traceback
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Callable, Dict, List, Optional

import numpy as np
import pandas as pd

from app.ml.metrics import ThresholdPolicy, aggregate, choose_threshold, compute_metrics, metrics_at_budget, to_jsonable
from app.ml.sampling import SAMPLER_CANDIDATES, SWEEP_CANDIDATES, base_of, kwargs_for, resample
from app.ml.user_level import DEFAULT_AGG, evaluate_user_level
from app.ml.sequences import (
    FeatureScaler, SequenceData, UserSplit, build_sequences, flatten, insider_flags,
    label_samples, load_labels, unflatten, user_level_split, kfold_user_splits,
)

log = logging.getLogger("train")

DEFAULT_SEEDS = (0, 1, 2)
DEFAULT_WINDOW_POLICY = ThresholdPolicy(mode="budget", budget=0.01)   # flag the top 1% of windows
DEFAULT_USER_POLICY = ThresholdPolicy(mode="budget", budget=0.10)     # flag the top 10% of users (~prevalence x1.5)
USER_KEYS = ["recall", "precision", "f1", "pr_auc", "n_flagged", "true_flagged",
             "precision@5", "precision@10", "precision@15", "precision@20", "recall@10", "recall@15", "recall@20", "recall@30"]


class SplitError(ValueError):
    """A user-level split that can't support training/evaluation (e.g. no positives in val)."""


# ------------------------------------------------------------------ data prep

@dataclass
class PreparedData:
    features_df: pd.DataFrame
    seq: SequenceData
    y: np.ndarray                 # per-sample label (N,)
    unique_users: np.ndarray
    user_is_insider: np.ndarray   # per unique user
    dataset: Optional[str]
    window_days: int
    seq_len: int


@dataclass
class SplitData:
    split: UserSplit
    scaler: FeatureScaler
    X_train: np.ndarray           # scaled (N, T, F)
    y_train: np.ndarray
    X_val: np.ndarray
    y_val: np.ndarray
    X_test: np.ndarray
    y_test: np.ndarray
    idx_train: np.ndarray         # positions into PreparedData.features_df
    idx_val: np.ndarray
    idx_test: np.ndarray


def prepare_data(features_df: pd.DataFrame, labels: pd.DataFrame, user_code_by_id: Dict[int, str],
                 window_days: int, seq_len: int, dataset: Optional[str] = None,
                 feature_columns: Optional[List[str]] = None) -> PreparedData:
    if features_df.empty:
        raise ValueError("No feature windows: ingest events first (POST /pipeline/ingest).")
    features_df = features_df.reset_index(drop=True)
    seq = build_sequences(features_df, seq_len, feature_columns)
    y = label_samples(features_df, labels, user_code_by_id)
    users = np.unique(features_df["user_id"].to_numpy())
    flags = insider_flags(users, labels, user_code_by_id)
    if flags.sum() == 0:
        raise ValueError("None of the labeled insiders appear in the ingested data for this dataset.")
    return PreparedData(features_df, seq, y, users, flags, dataset, window_days, seq_len)


def prepare_from_db(db, answers_csv: str, dataset: str = "cert", window_days: int = 2, seq_len: int = 7,
                    labels_dataset: Optional[str] = None) -> PreparedData:
    from app.ml.features import build_feature_windows
    from app.models import User
    features_df = build_feature_windows(db, window_days=window_days, dataset=dataset)
    labels = load_labels(answers_csv, dataset=labels_dataset)
    # Non-CERT users are stored as "<dataset>:<id>" to avoid colliding with CERT ids;
    # labels files use the bare id, so strip the prefix when matching.
    prefix = f"{dataset}:" if dataset != "cert" else ""
    user_code_by_id = {u.id: (u.user_code[len(prefix):] if prefix and u.user_code.startswith(prefix) else u.user_code)
                       for u in db.query(User).all()}
    return prepare_data(features_df, labels, user_code_by_id, window_days, seq_len, dataset)


def prepare_from_features(features_df: pd.DataFrame, user_codes: Dict[int, str], answers_csv: str,
                          dataset: str = "cert", window_days: int = 2, seq_len: int = 7,
                          labels_dataset: Optional[str] = None,
                          feature_columns: Optional[List[str]] = None) -> PreparedData:
    """DB-free entry point: features from app/ml/offline_features.py (streamed from CSVs, or a cached file)."""
    labels = load_labels(answers_csv, dataset=labels_dataset)
    return prepare_data(features_df, labels, user_codes, window_days, seq_len, dataset, feature_columns)


def make_split(prep: PreparedData, seed: int, val_frac: float = 0.15, test_frac: float = 0.15,
               split: Optional[UserSplit] = None) -> SplitData:
    if split is None:
        split = user_level_split(prep.unique_users, prep.user_is_insider, val_frac, test_frac, seed)
    uid = prep.features_df["user_id"].to_numpy()
    m_tr, m_va, m_te = split.masks(uid)
    # scaler is fit on TRAIN-split users only (one row per user-window)
    scaler = FeatureScaler().fit(prep.features_df[m_tr], prep.seq.feature_columns)
    Xs = scaler.transform(prep.seq.X)
    sd = SplitData(
        split, scaler,
        Xs[m_tr], prep.y[m_tr], Xs[m_va], prep.y[m_va], Xs[m_te], prep.y[m_te],
        np.flatnonzero(m_tr), np.flatnonzero(m_va), np.flatnonzero(m_te),
    )
    for name, yy in (("train", sd.y_train), ("val", sd.y_val), ("test", sd.y_test)):
        if yy.sum() == 0:
            raise SplitError(f"seed {seed}: the {name} split has no positive samples "
                             "(insider malicious period falls outside its windows?)")
    return sd


# ------------------------------------------------------------------ Layer 1

EVALUATORS = ("hgb", "lstm")     # model that scores each sampler candidate in Layer 1


def _fit_eval_candidate(sd: SplitData, name: str, cfg, seed: int, window_policy, sampler_kwargs: dict,
                        keep_predictions: bool, seq_len: int, n_feat: int, prep: PreparedData = None,
                        user_policy=None, ae_cache: Optional[dict] = None, evaluator: str = "hgb") -> dict:
    kw = kwargs_for(name, sampler_kwargs)
    if base_of(name) == "DeepSMOTE" and ae_cache is not None:
        kw["cache"] = ae_cache                      # autoencoder is trained once per split, reused across ratios
    rr = resample(name, flatten(sd.X_train), sd.y_train, seed=seed, **kw)
    if evaluator == "hgb":
        # same model + same class-weighting rule as production Layer 2 (ensemble._fit_hgb), so the winner is the
        # best sampler for the model that actually ships
        from app.ml import ensemble
        pr = ensemble.fit_members(sd, rr, ["hgb"], None, seed, 1, seq_len, n_feat)["hgb"]
        p_val, p_test = pr["val"], pr["test"]
    elif evaluator == "lstm":
        from app.ml.lstm_detector import predict_proba, train_lstm
        Xr = unflatten(rr.X, seq_len, n_feat)
        model, _ = train_lstm(Xr, rr.y, cfg.scaled_for(len(rr.y)), seed=seed, pos_weight=rr.pos_weight,
                              X_val=sd.X_val, y_val=sd.y_val)
        p_val, p_test = predict_proba(model, sd.X_val), predict_proba(model, sd.X_test)
    else:
        raise ValueError(f"unknown evaluator '{evaluator}'; choose from {EVALUATORS}")
    thr = choose_threshold(sd.y_val, p_val, window_policy)
    rec = {
        "seed": seed, "threshold": round(thr, 6),
        "val": compute_metrics(sd.y_val, p_val, thr),
        "test": compute_metrics(sd.y_test, p_test, thr),
        "train_size_after_sampling": int(len(rr.y)),
        "train_positives_after_sampling": int(rr.y.sum()),
        "sampler_info": rr.info,
    }
    if window_policy.mode == "budget":
        rec["test_at_budget"] = metrics_at_budget(sd.y_test, p_test, window_policy.budget)
    if prep is not None:
        rec["user"] = evaluate_user_level(prep, sd.idx_val, sd.idx_test, p_val, p_test, user_policy)
    if keep_predictions:
        rec["_pred"] = {"val_prob": p_val, "test_prob": p_test, "threshold": thr}
    return rec


def run_sampler_comparison(
    prep: PreparedData, seeds=DEFAULT_SEEDS, candidates=SAMPLER_CANDIDATES, eval_cfg=None,
    selection_metric: str = "pr_auc", min_recall: float = 0.9, sampler_kwargs: Optional[dict] = None,
    keep_predictions: bool = False, val_frac: float = 0.15, test_frac: float = 0.15,
    progress: Callable[[str], None] = log.info, window_policy=None, user_policy=None,
    kfold: Optional[int] = None, evaluator: str = "hgb",
) -> dict:
    """
    Layer 1. For each seed: one user-level split shared by every candidate
    (identical seeds and splits), the sampler touches only the train split,
    the `evaluator` model is trained ("hgb" = production Layer-2 model, default;
    "lstm" = the original small LSTM proxy, needs PyTorch), and the candidate is
    scored on the untouched validation and test sets (original class ratio).

    Selection uses mean VALIDATION `selection_metric` (default PR-AUC, which
    is threshold-independent and robust under heavy imbalance). Test metrics
    are reported for every candidate but never influence the choice.
    A candidate is excluded from winning if it failed on any usable seed, or
    (DeepSMOTE) failed to converge on more than half of them -- and that is
    recorded as a finding instead of being forced.
    """
    if evaluator not in EVALUATORS:
        raise ValueError(f"unknown evaluator '{evaluator}'; choose from {EVALUATORS}")
    if evaluator == "lstm":
        from app.ml.lstm_detector import LSTMConfig
        eval_cfg = eval_cfg or LSTMConfig.evaluator()
    sampler_kwargs = sampler_kwargs or {}
    # `min_recall` is only used by the legacy "recall" threshold mode
    window_policy = ThresholdPolicy.coerce(window_policy) if window_policy is not None else DEFAULT_WINDOW_POLICY
    if window_policy.mode == "recall":
        window_policy = ThresholdPolicy(mode="recall", min_recall=min_recall)
    user_policy = ThresholdPolicy.coerce(user_policy) if user_policy is not None else DEFAULT_USER_POLICY
    T, F = prep.seq_len, len(prep.seq.feature_columns)

    splits, skipped = {}, []
    if kfold:      # every user is a test user exactly once; the "seed" label becomes the fold index
        plan = dict(enumerate(kfold_user_splits(prep.unique_users, prep.user_is_insider, kfold)))
    else:
        plan = {s: None for s in seeds}
    for s, sp in plan.items():
        try:
            splits[s] = make_split(prep, s, val_frac, test_frac, split=sp)
        except ValueError as e:
            skipped.append({"seed": s, "reason": str(e)})
            progress(f"skipping seed {s}: {e}")
    if not splits:
        raise SplitError("No usable seeds: " + "; ".join(x["reason"] for x in skipped))

    results = {}
    ae_caches = {s: {} for s in splits}
    for name in candidates:
        per_seed, errors = [], []
        for s, sd in splits.items():
            progress(f"[layer1] {name} seed={s}")
            try:
                per_seed.append(_fit_eval_candidate(sd, name, eval_cfg, s, window_policy, sampler_kwargs,
                                                    keep_predictions, T, F, prep, user_policy, ae_caches[s], evaluator))
            except Exception as e:      # a sampler failing is a finding, not a crash
                errors.append({"seed": s, "error": f"{type(e).__name__}: {e}"})
                progress(f"[layer1] {name} seed={s} FAILED: {e}")
        unstable = sum(1 for r in per_seed if r["sampler_info"].get("converged") is False)
        entry = {
            "n_ok_seeds": len(per_seed), "errors": errors, "unstable_seeds": unstable, "per_seed": per_seed,
            "summary": {
                "val": aggregate([r["val"] for r in per_seed]),
                "test": aggregate([r["test"] for r in per_seed]),
                "test_at_budget": aggregate([r["test_at_budget"] for r in per_seed if "test_at_budget" in r], ["recall", "precision"]),
                "user": {m: {"val": aggregate([r["user"][m]["val"] for r in per_seed], USER_KEYS),
                             "test": aggregate([r["user"][m]["test"] for r in per_seed], USER_KEYS)}
                         for m in per_seed[0].get("user", {})},
            } if per_seed else None,
        }
        if base_of(name) == "DeepSMOTE" and per_seed:
            entry["convergence_note"] = (
                f"autoencoder converged in {len(per_seed) - unstable}/{len(per_seed)} seeds "
                f"(mean reconstruction ratio {np.mean([r['sampler_info'].get('recon_ratio', np.nan) for r in per_seed]):.2f})"
            )
        results[name] = entry

    excluded, eligible = {}, {}
    for i, (name, e) in enumerate(results.items()):
        if e["n_ok_seeds"] == 0:
            excluded[name] = "failed on every seed: " + (e["errors"][0]["error"] if e["errors"] else "unknown")
        elif e["errors"]:
            excluded[name] = f"failed on {len(e['errors'])} of {len(splits)} seeds"
        elif e["unstable_seeds"] * 2 > e["n_ok_seeds"]:
            excluded[name] = f"did not train reliably (autoencoder unconverged on {e['unstable_seeds']}/{e['n_ok_seeds']} seeds)"
        else:
            sel = e["summary"]["val"][selection_metric]["mean"]
            f1 = e["summary"]["val"]["f1"]["mean"] or 0.0
            if sel is not None:
                eligible[name] = (sel, f1, -i)
    winner = max(eligible, key=eligible.get) if eligible else None
    return {
        "seeds_used": list(splits), "seeds_skipped": skipped, "selection_metric": selection_metric,
        "selection_split": "validation", "min_recall": min_recall,
        "window_policy": window_policy.__dict__, "user_policy": user_policy.__dict__,
        "evaluator": evaluator, "eval_config": asdict(eval_cfg) if eval_cfg is not None else None, "candidates": results,
        "excluded": excluded, "winner": winner,
        "winner_reason": (f"highest mean validation {selection_metric} = {eligible[winner][0]:.4f}" if winner else "no eligible candidate"),
    }   # NB: may hold private `_pred` arrays (keep_predictions=True); run to_jsonable() before persisting


# ------------------------------------------------------------------ Layer 2

def train_production(prep: PreparedData, sampler: str, seed: int, cfg=None, min_recall: float = 0.9,
                     sampler_kwargs: Optional[dict] = None, artifact_dir: Optional[Path] = None,
                     val_frac: float = 0.15, test_frac: float = 0.15,
                     window_policy=None, user_policy=None, user_agg: Optional[dict] = None):
    """Trains the production LSTM once on the chosen sampler's output; tunes + freezes the threshold on val."""
    from app.ml.lstm_detector import LSTMArtifact, LSTMConfig, predict_proba, train_lstm
    cfg = cfg or LSTMConfig.production()
    T, F = prep.seq_len, len(prep.seq.feature_columns)
    sd = make_split(prep, seed, val_frac, test_frac)
    rr = resample(sampler, flatten(sd.X_train), sd.y_train, seed=seed, **kwargs_for(sampler, sampler_kwargs))
    model, history = train_lstm(unflatten(rr.X, T, F), rr.y, cfg.scaled_for(len(rr.y)), seed=seed, pos_weight=rr.pos_weight,
                                X_val=sd.X_val, y_val=sd.y_val)
    p_val, p_test = predict_proba(model, sd.X_val), predict_proba(model, sd.X_test)
    window_policy = ThresholdPolicy.coerce(window_policy) if window_policy is not None else DEFAULT_WINDOW_POLICY
    if window_policy.mode == "recall":
        window_policy = ThresholdPolicy(mode="recall", min_recall=min_recall)
    user_policy = ThresholdPolicy.coerce(user_policy) if user_policy is not None else DEFAULT_USER_POLICY
    thr = choose_threshold(sd.y_val, p_val, window_policy)     # tuned on validation, then frozen
    user_agg = dict(user_agg or DEFAULT_AGG)
    methods = {"max": {"method": "max"}, "topk": {"method": "topk", "k": 3}, "consecutive": {"method": "consecutive", "n": 3}}
    methods[user_agg["method"]] = user_agg
    user_res = evaluate_user_level(prep, sd.idx_val, sd.idx_test, p_val, p_test, user_policy, methods)
    chosen = user_res[user_agg["method"]]
    metrics = {
        "window_policy": window_policy.__dict__, "user_policy": user_policy.__dict__,
        "user": user_res, "user_selected": user_agg["method"],
        "val": compute_metrics(sd.y_val, p_val, thr), "test": compute_metrics(sd.y_test, p_test, thr),
        "threshold": round(thr, 6), "epochs_run": len(history),
        "train_size_after_sampling": int(len(rr.y)), "sampler_info": rr.info,
        "split": {"train_users": int(len(sd.split.train)), "val_users": int(len(sd.split.val)),
                  "test_users": int(len(sd.split.test))},
    }
    artifact = LSTMArtifact(
        state_dict={k: v.detach().cpu() for k, v in model.state_dict().items()}, config=asdict(cfg),
        scaler=sd.scaler, threshold=thr, sampler=sampler, seq_len=T, window_days=prep.window_days,
        feature_columns=list(prep.seq.feature_columns), dataset=prep.dataset, seed=seed,
        metrics=to_jsonable(metrics), user_threshold=chosen["threshold"], user_agg=chosen["agg"],
    )
    if artifact_dir is not None:
        artifact.save(artifact_dir)
    return artifact, metrics, history


# ------------------------------------------------------------------ out-of-fold scores for EVERY user

def run_oof(prep: PreparedData, sampler: str = "none", k: int = 7, cfg=None, n_inits: int = 3,
            user_agg: Optional[dict] = None, user_policy=None, progress: Callable[[str], None] = print) -> pd.DataFrame:
    """
    k-fold over users. Each user is scored by models that never trained on them (fold i = test, fold i+1 = val).
    Several initialisations per fold are averaged. The user threshold is tuned on that fold's validation users and
    applied to its test users. Returns one row per user: user_id, fold, user_score, threshold, flagged, is_insider.
    """
    from app.ml.lstm_detector import LSTMConfig, predict_proba, train_lstm
    from app.ml.user_level import aggregate_user_scores, user_truth
    cfg = cfg or LSTMConfig.production()
    agg = user_agg or DEFAULT_AGG
    kw = {a: v for a, v in agg.items() if a in ("method", "k", "n")}
    pol = ThresholdPolicy.coerce(user_policy) if user_policy is not None else DEFAULT_USER_POLICY
    T, F = prep.seq_len, len(prep.seq.feature_columns)
    uid, ws = prep.features_df["user_id"].to_numpy(), prep.seq.window_starts
    out = []
    for i, sp in enumerate(kfold_user_splits(prep.unique_users, prep.user_is_insider, k)):
        progress(f"[oof] fold {i + 1}/{k}")
        sd = make_split(prep, i, split=sp)
        rr = resample(sampler, flatten(sd.X_train), sd.y_train, seed=i, **kwargs_for(sampler, None))
        Xr = unflatten(rr.X, T, F)
        pv = pt = 0.0
        for j in range(n_inits):
            model, _ = train_lstm(Xr, rr.y, cfg.scaled_for(len(rr.y)), seed=100 * i + j, pos_weight=rr.pos_weight,
                                  X_val=sd.X_val, y_val=sd.y_val)
            pv = pv + predict_proba(model, sd.X_val) / n_inits
            pt = pt + predict_proba(model, sd.X_test) / n_inits
        sv = aggregate_user_scores(uid[sd.idx_val], ws[sd.idx_val], pv, window_days=prep.window_days, **kw)
        st = aggregate_user_scores(uid[sd.idx_test], ws[sd.idx_test], pt, window_days=prep.window_days, **kw)
        tv = user_truth(sv.index.to_numpy(), prep.unique_users, prep.user_is_insider)
        thr = choose_threshold(tv, sv.to_numpy(), pol)
        tt = user_truth(st.index.to_numpy(), prep.unique_users, prep.user_is_insider)
        n_top = max(1, int(round((pol.budget if pol.mode == "budget" else 0.10) * len(st))))
        top = st.rank(ascending=False, method="first").to_numpy() <= n_top      # threshold-free: top X% of this fold
        out.append(pd.DataFrame({"user_id": st.index.to_numpy(), "fold": i, "user_score": st.to_numpy(),
                                 "threshold": thr, "flagged": st.to_numpy() >= thr, "flagged_at_budget": top,
                                 "is_insider": tt.astype(bool)}))
    return pd.concat(out, ignore_index=True)


# ------------------------------------------------------------------ run bookkeeping
# Two recorders with the same interface: the Postgres/API path stores results on a DetectionRun row;
# the DB-free (Kaggle) path writes them to JSON files in --out-dir.

class _DbRecorder:
    def __init__(self, db, run):
        self.db, self.run = db, run

    def update(self, **fields):
        for k, v in fields.items():
            setattr(self.run, k, v)
        self.db.commit()

    def rollback(self):
        self.db.rollback()


class _FileRecorder:
    def __init__(self, out_dir: Path, kind: str):
        self.path, self.state = Path(out_dir) / f"result_{kind}.json", {}
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def update(self, **fields):
        import json
        self.state.update({k: (v.isoformat() if isinstance(v, datetime) else v) for k, v in fields.items()})
        self.path.write_text(json.dumps(to_jsonable(self.state), indent=2))

    def rollback(self):
        pass


def _acquire_data(args, out_dir: Path, use_db: bool, db=None):
    """Returns (PreparedData, extra_config). Source: cached features file > streamed CSVs > database."""
    import pickle
    from app.ml.offline_features import (
        build_features_from_csvs, find_dataset_dir, list_users, load_features, save_features, select_users,
    )
    extra: dict = {}
    if use_db:
        return prepare_from_db(db, args.answers, args.dataset, args.window_days, args.seq_len, args.labels_dataset), extra

    cache = Path(args.features) if args.features else out_dir / f"features_w{args.window_days}.pkl"
    labels = load_labels(args.answers, dataset=args.labels_dataset)
    insiders = set(labels.loc[labels["malicious"] == 1, "user"])
    if cache.exists() and not args.rebuild_features:
        log.info("loading cached features from %s", cache)
        features_df, user_codes, meta = load_features(cache)
        if meta.get("window_days") not in (None, args.window_days):
            raise ValueError(f"cached features use window_days={meta['window_days']}, requested {args.window_days} "
                             "(delete the cache or pass --rebuild-features)")
    elif args.csv_dir:
        directory = find_dataset_dir(args.csv_dir)
        log.info("streaming CERT CSVs from %s (this is the slow, one-time pass)", directory)
        all_users = list_users(directory)
        keep = select_users(all_users, insiders, args.max_benign_users) if args.max_benign_users else None
        features_df, user_codes = build_features_from_csvs(
            directory, args.window_days, args.internal_domain, keep, chunksize=args.chunksize, progress=log.info)
        meta = {"window_days": args.window_days, "internal_domain": args.internal_domain,
                "n_users_source": len(all_users), "max_benign_users": args.max_benign_users}
        save_features(cache, features_df, user_codes, meta)
        log.info("cached features to %s", cache)
    else:
        raise FileNotFoundError(f"No cached features at {cache} and no --csv-dir given to build them.")

    # optional post-hoc benign-user sampling (also applies to a cache built on more users)
    codes_present = set(user_codes.values())
    if args.max_benign_users:
        keep = select_users(codes_present, insiders, args.max_benign_users)
        keep_ids = {i for i, c in user_codes.items() if c in keep}
        features_df = features_df[features_df["user_id"].isin(keep_ids)].reset_index(drop=True)
        user_codes = {i: c for i, c in user_codes.items() if i in keep_ids}
    n_source = meta.get("n_users_source") or len(codes_present)
    n_kept = len(set(user_codes.values()))
    extra = {"features_cache": str(cache), "users_in_source": n_source, "users_used": n_kept, "internal_domain": meta.get("internal_domain")}
    if n_kept < n_source:
        extra["sampling_note"] = (f"benign users subsampled ({n_kept} of {n_source} users kept, all insiders retained): "
                                  "insider prevalence in val/test is higher than in the full population, so precision and "
                                  "PR-AUC are optimistic relative to the real population")
    return prepare_from_features(features_df, user_codes, args.answers, args.dataset, args.window_days,
                                 args.seq_len, args.labels_dataset), extra


def main(argv: Optional[List[str]] = None) -> int:
    ap = argparse.ArgumentParser(description="Offline LSTM training / sampler comparison / evaluation job")
    ap.add_argument("--mode", choices=["compare", "train", "evaluate"], default="train")
    ap.add_argument("--answers", required=True, help="ground-truth CSV (user,malicious[,start,end] or CERT insiders.csv)")
    ap.add_argument("--dataset", default="cert", help="cert (CERT r4.2 only)")
    ap.add_argument("--labels-dataset", default=None, help="filter for CERT insiders.csv 'dataset' column, e.g. r4.2")
    ap.add_argument("--window-days", type=int, default=2)
    ap.add_argument("--seq-len", type=int, default=7)
    ap.add_argument("--seeds", type=int, nargs="+", default=list(DEFAULT_SEEDS))
    ap.add_argument("--candidates", nargs="+", default=None, help="sampler specs, e.g. none NM-2@0.05 DeepSMOTE@0.05+cw")
    ap.add_argument("--selection-metric", default="pr_auc", choices=["pr_auc", "f1", "recall", "precision"])
    ap.add_argument("--min-recall", type=float, default=0.9, help="only used with --threshold-mode recall")
    ap.add_argument("--threshold-mode", default="budget", choices=["budget", "precision", "fbeta", "recall"])
    ap.add_argument("--window-budget", type=float, default=0.01, help="budget mode: fraction of windows to flag")
    ap.add_argument("--user-budget", type=float, default=0.10, help="fraction of users to flag")
    ap.add_argument("--user-agg", default="topk", choices=["max", "topk", "consecutive"])
    ap.add_argument("--evaluator", default="hgb", choices=list(EVALUATORS), help="Layer 1: model that scores each sampler candidate")
    ap.add_argument("--sweep", action="store_true", help="use the partial-ratio / hybrid candidate sweep")
    # --- data source (default: the Postgres database, as launched by the API)
    ap.add_argument("--csv-dir", default=None, help="DB-FREE: stream the raw CERT CSVs from here (e.g. /kaggle/input/<dataset>)")
    ap.add_argument("--features", default=None, help="DB-FREE: path of the cached features file (built from --csv-dir if missing)")
    ap.add_argument("--rebuild-features", action="store_true", help="ignore an existing features cache")
    ap.add_argument("--max-benign-users", type=int, default=None, help="keep all insiders + this many random benign users (memory/time)")
    ap.add_argument("--internal-domain", default="@company.com", help='e-mail domain counted as internal (real CERT: "@dtaa.com")')
    ap.add_argument("--chunksize", type=int, default=2_000_000, help="rows per CSV chunk when streaming")
    ap.add_argument("--out-dir", default="insider_out", help="DB-free mode: where features, model and result JSON are written")
    # --- DB mode bookkeeping
    ap.add_argument("--run-id", type=int, default=None, help="existing DetectionRun to update (set by the API)")
    ap.add_argument("--artifact-dir", default=None)
    args = ap.parse_args(argv)
    args.candidates = args.candidates or list(SWEEP_CANDIDATES if args.sweep else SAMPLER_CANDIDATES)
    wpol = ThresholdPolicy(mode=args.threshold_mode, budget=args.window_budget, min_recall=args.min_recall)
    upol = ThresholdPolicy(mode="budget", budget=args.user_budget)
    uagg = {"method": args.user_agg, "k": 3, "n": 3}

    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", stream=sys.stdout)
    use_db = not (args.csv_dir or args.features)
    kind = {"compare": "sampler_comparison", "train": "train", "evaluate": "evaluation"}[args.mode]
    out_dir = Path(args.out_dir)
    db = None

    if use_db:
        from app.config import ARTIFACT_DIR
        from app.database import SessionLocal, engine
        from app.migrations import ensure_schema
        from app.models import DetectionRun
        ensure_schema(engine)
        db = SessionLocal()
        run = db.query(DetectionRun).get(args.run_id) if args.run_id else None
        if run is None:
            run = DetectionRun(sampling_technique="pending", detector="LSTM", window_days=args.window_days)
            db.add(run)
            db.commit()
        rec = _DbRecorder(db, run)
        rec.update(run_kind=kind, model_type="LSTM", detector="LSTM", dataset=args.dataset,
                   window_days=args.window_days, status="running", notes=None)
        art_dir = Path(args.artifact_dir) if args.artifact_dir else ARTIFACT_DIR / f"run_{run.id}"
    else:
        rec = _FileRecorder(out_dir, kind)
        rec.update(run_kind=kind, model_type="LSTM", detector="LSTM", dataset=args.dataset,
                   window_days=args.window_days, status="running")
        art_dir = Path(args.artifact_dir) if args.artifact_dir else out_dir / "model"

    try:
        prep, extra = _acquire_data(args, out_dir, use_db, db)
        try:
            import torch
            extra["device"] = "cuda" if torch.cuda.is_available() else "cpu"
        except ImportError:
            pass
        log.info("prepared %d samples, %d users (%d insiders), %d positive samples",
                 len(prep.y), len(prep.unique_users), int(prep.user_is_insider.sum()), int(prep.y.sum()))
        base_cfg = {"dataset": args.dataset, "window_days": args.window_days, "seq_len": args.seq_len,
                    "seeds": args.seeds, "candidates": args.candidates, "selection_metric": args.selection_metric,
                    "min_recall": args.min_recall, "answers": args.answers, **extra}

        if args.mode == "evaluate":
            from app.ml.evaluation import render_markdown, run_full_evaluation
            report = run_full_evaluation(prep, args.seeds, args.candidates, args.selection_metric, args.min_recall, progress=log.info,
                                          window_policy=wpol, user_policy=upol)
            art_dir.mkdir(parents=True, exist_ok=True)
            (art_dir / f"evaluation_{args.dataset}.md").write_text(render_markdown(report))
            rec.update(status="completed", finished_at=datetime.utcnow(),
                       sampling_technique=report["layer1"].get("winner") or "n/a",
                       metrics=to_jsonable(report), config=to_jsonable({**base_cfg, "report_dir": str(art_dir)}),
                       notes=f"full evaluation over seeds {report['layer1']['seeds_used']}")
            log.info("report written to %s", art_dir / f"evaluation_{args.dataset}.md")
            return 0

        comp = run_sampler_comparison(prep, args.seeds, args.candidates, selection_metric=args.selection_metric,
                                      min_recall=args.min_recall, progress=log.info,
                                      window_policy=wpol, user_policy=upol, evaluator=args.evaluator)
        winner = comp["winner"]
        if args.mode == "compare":
            rec.update(status="completed", finished_at=datetime.utcnow(), sampling_technique=winner or "n/a",
                       metrics=to_jsonable({"layer1": comp}), config=to_jsonable(base_cfg),
                       notes=f"winner: {winner} ({comp['winner_reason']})")
            return 0

        if winner is None:
            raise RuntimeError("No eligible sampler in Layer 1: " + str(comp["excluded"]))
        final_seed = comp["seeds_used"][0]
        log.info("[layer2] training production LSTM with sampler=%s seed=%d", winner, final_seed)
        art, m, _ = train_production(prep, winner, final_seed, min_recall=args.min_recall, artifact_dir=art_dir,
                                  window_policy=wpol, user_policy=upol, user_agg=uagg)
        t = m["test"]
        rec.update(status="completed", finished_at=datetime.utcnow(), sampling_technique=winner,
                   precision=t["precision"], recall=t["recall"], f_score=t["f1"], accuracy=t["accuracy"],
                   metrics=to_jsonable({"production": m, "layer1": comp}),
                   config=to_jsonable({**base_cfg, "sampler": winner, "final_seed": final_seed,
                                       "lstm_config": art.config, "threshold": art.threshold,
                                       "artifact_dir": str(art_dir)}),
                   notes=f"LSTM + {winner}: test recall={t['recall']} precision={t['precision']} pr_auc={t['pr_auc']}")
        log.info("artifact saved to %s", art_dir)
        return 0
    except Exception as e:
        rec.rollback()
        rec.update(status="failed", finished_at=datetime.utcnow(), notes=f"{type(e).__name__}: {e}")
        traceback.print_exc()
        return 1
    finally:
        if db is not None:
            db.close()


if __name__ == "__main__":
    sys.exit(main())

Writing /kaggle/working/insider_code/app/ml/train.py


In [17]:
%%writefile /kaggle/working/insider_code/app/ml/ensemble.py
"""
Layer-2 ensemble (architecture.md section 6).

Members       lstm  |  bilstm_attn  |  hgb   (gradient boosting on last / max / mean over the T-window history)
Combination   stack      logistic regression on the members' logits, fitted on VALIDATION users' window scores
                         (users no member trained on). Validation scores are produced cross-fitted by user, so
                         the user threshold tuned on them is not in-sample for the stacker.
              rank_avg   mean percentile rank (fallback comparison)
Evaluation    run_oof_ensemble: the same k-fold user split as run_oof (fold i test, i+1 val, rest train); each user is
              scored only by models that never trained on them. Alert-budget metrics, ablation, paired bootstrap.

Layer 1 (the sampler) is unchanged: the winning spec is applied to the TRAIN split only, for every member.
"""
import pickle
from pathlib import Path
from typing import Callable, Dict, List, Optional, Sequence

import numpy as np
import pandas as pd

from app.ml.metrics import ThresholdPolicy, choose_threshold, compute_metrics, pr_auc, to_jsonable
from app.ml.sampling import kwargs_for, resample
from app.ml.sequences import flatten, kfold_user_splits, unflatten
from app.ml.user_level import DEFAULT_AGG, aggregate_user_scores, evaluate_user_level, user_truth

MEMBERS_DEFAULT = ("lstm", "bilstm_attn", "hgb")
NN_MEMBERS = ("lstm", "bilstm_attn")
BLENDS = ("rank_avg", "stack")


# ------------------------------------------------------------------ members

def summarize_history(X: np.ndarray) -> np.ndarray:
    """(N, T, F) -> (N, 3F): last window + max + mean over the history (same summary as the 7c sanity check)."""
    X = np.nan_to_num(X)
    return np.concatenate([X[:, -1], X.max(1), X.mean(1)], axis=1)


def _fit_hgb(Xr: np.ndarray, yr: np.ndarray, pos_weight: Optional[float], seed: int):
    from sklearn.ensemble import HistGradientBoostingClassifier
    n_pos = max(int((yr == 1).sum()), 1)
    pw = min(50.0, float(pos_weight)) if pos_weight else min(50.0, float(np.sqrt((yr == 0).sum() / n_pos)))
    w = np.where(yr == 1, pw, 1.0)
    return HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, random_state=seed) \
        .fit(summarize_history(Xr), yr, sample_weight=w)


def fit_members(sd, rr, members: Sequence[str], cfg, seed: int, n_inits: int, T: int, F: int, keep_models: bool = False) -> dict:
    """
    Trains every member on the resampled TRAIN split `rr` and predicts the untouched val/test windows.
    Returns {member: {"val": p, "test": p, "models": [...]}}  (models only when keep_models).
    """
    Xr = unflatten(rr.X, T, F)
    out = {}
    for name in members:
        if name == "hgb":
            clf = _fit_hgb(Xr, rr.y, rr.pos_weight, seed)
            pv = clf.predict_proba(summarize_history(sd.X_val))[:, 1]
            pt = clf.predict_proba(summarize_history(sd.X_test))[:, 1]
            out[name] = {"val": pv, "test": pt, "models": [clf] if keep_models else []}
        elif name in NN_MEMBERS:
            from app.ml.lstm_detector import predict_proba, train_lstm      # torch is only needed for the neural members
            pv, pt, models = 0.0, 0.0, []
            for j in range(n_inits):
                m, _ = train_lstm(Xr, rr.y, cfg.scaled_for(len(rr.y)), seed=seed * 100 + j, pos_weight=rr.pos_weight,
                                  X_val=sd.X_val, y_val=sd.y_val, arch=name)
                pv = pv + predict_proba(m, sd.X_val) / n_inits
                pt = pt + predict_proba(m, sd.X_test) / n_inits
                if keep_models:
                    models.append({k: v.detach().cpu() for k, v in m.state_dict().items()})
            out[name] = {"val": pv, "test": pt, "models": models}
        else:
            raise ValueError(f"unknown ensemble member '{name}'; choose from {MEMBERS_DEFAULT}")
    return out


# ------------------------------------------------------------------ blending

def _logit(p: np.ndarray) -> np.ndarray:
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def _rank01(p: np.ndarray) -> np.ndarray:
    from scipy.stats import rankdata
    return rankdata(p) / len(p)


def _stack(member_val: Dict[str, np.ndarray], member_test: Dict[str, np.ndarray], y_val: np.ndarray,
           val_users: np.ndarray, cv_groups: int = 3):
    """
    Logistic-regression stacker on member logits. Returns (val_scores, test_scores, coefs).
    val_scores are cross-fitted by user (GroupKFold) so a threshold tuned on them is not in-sample;
    test_scores come from the stacker fitted on all validation windows.
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import GroupKFold
    names = list(member_val)
    Zv = np.column_stack([_logit(member_val[n]) for n in names])
    Zt = np.column_stack([_logit(member_test[n]) for n in names])
    y = np.asarray(y_val).astype(int)
    mk = lambda: LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000)
    if len(np.unique(y)) < 2:                                   # cannot fit: plain mean logit
        return Zv.mean(1), Zt.mean(1), {n: 1.0 / len(names) for n in names}
    final = mk().fit(Zv, y)
    sv = np.full(len(y), np.nan)
    try:
        for tr, te in GroupKFold(n_splits=min(cv_groups, len(np.unique(val_users)))).split(Zv, y, groups=val_users):
            if len(np.unique(y[tr])) < 2:
                raise ValueError("a cross-fit fold has no positives")
            sv[te] = mk().fit(Zv[tr], y[tr]).decision_function(Zv[te])
    except ValueError:
        sv = final.decision_function(Zv)                        # fall back to in-sample validation scores
    return sv, final.decision_function(Zt), {n: float(round(c, 4)) for n, c in zip(names, final.coef_[0])}


def blend_all(preds: dict, y_val: np.ndarray, val_users: np.ndarray, blends: Sequence[str] = BLENDS):
    """{method: (val_scores, test_scores)} for each member, rank_avg and stack, plus the stacker coefficients."""
    out = {n: (p["val"], p["test"]) for n, p in preds.items()}
    coefs = {}
    if len(preds) > 1:
        if "rank_avg" in blends:
            out["rank_avg"] = (np.mean([_rank01(p["val"]) for p in preds.values()], axis=0),
                               np.mean([_rank01(p["test"]) for p in preds.values()], axis=0))
        if "stack" in blends:
            sv, st, coefs = _stack({n: p["val"] for n, p in preds.items()}, {n: p["test"] for n, p in preds.items()},
                                   y_val, val_users)
            out["stack"] = (sv, st)
    return out, coefs


# ------------------------------------------------------------------ out-of-fold evaluation

def run_oof_ensemble(prep, sampler: str = "none", k: int = 7, cfg=None, n_inits: int = 2,
                     members: Sequence[str] = MEMBERS_DEFAULT, user_agg: Optional[dict] = None,
                     progress: Callable[[str], None] = print, make_split=None):
    """
    Returns (users, windows, coefs):
      users    one row per (user, method): user_id, fold, method, user_score, rank (1 = highest in its fold), n_fold, is_insider
      windows  one row per (fold, method): test window PR-AUC
      coefs    one row per fold: the stacker's member weights
    """
    if make_split is None:
        from app.ml.train import make_split
    if cfg is None and any(m in NN_MEMBERS for m in members):          # HGB-only runs need no torch
        from app.ml.lstm_detector import LSTMConfig
        cfg = LSTMConfig(hidden_size=64, num_layers=1, dropout=0.2, epochs=25, patience=5)
    agg = user_agg or DEFAULT_AGG
    kw = {a: v for a, v in agg.items() if a in ("method", "k", "n")}
    T, F = prep.seq_len, len(prep.seq.feature_columns)
    uid, ws = prep.features_df["user_id"].to_numpy(), prep.seq.window_starts
    urows, wrows, crows = [], [], []
    for i, sp in enumerate(kfold_user_splits(prep.unique_users, prep.user_is_insider, k)):
        progress(f"[oof-ens:{sampler}] fold {i + 1}/{k}")
        sd = make_split(prep, i, split=sp)
        rr = resample(sampler, flatten(sd.X_train), sd.y_train, seed=i, **kwargs_for(sampler, None))
        preds = fit_members(sd, rr, members, cfg, i, n_inits, T, F)
        scores, coefs = blend_all(preds, sd.y_val, uid[sd.idx_val])
        if coefs:
            crows.append({"fold": i, **coefs})
        for method, (_, pt) in scores.items():
            st = aggregate_user_scores(uid[sd.idx_test], ws[sd.idx_test], pt, window_days=prep.window_days, **kw)
            tt = user_truth(st.index.to_numpy(), prep.unique_users, prep.user_is_insider)
            urows.append(pd.DataFrame({
                "user_id": st.index.to_numpy(), "fold": i, "method": method, "user_score": st.to_numpy(),
                "rank": st.rank(ascending=False, method="first").to_numpy(), "n_fold": len(st), "is_insider": tt.astype(bool)}))
            wrows.append({"fold": i, "method": method, "window_pr_auc": pr_auc(sd.y_test, pt)})
    return (pd.concat(urows, ignore_index=True), pd.DataFrame(wrows), pd.DataFrame(crows))


def summarize_oof(users: pd.DataFrame, windows: Optional[pd.DataFrame] = None,
                  budgets: Sequence[float] = (0.05, 0.10, 0.15, 0.30)) -> pd.DataFrame:
    """
    One row per method. precision@X / recall@X flag the top X% of users WITHIN EACH FOLD and pool the counts, so no
    validation threshold has to carry over. user PR-AUC is computed per fold, then mean +/- std over folds.
    """
    from sklearn.metrics import average_precision_score
    rows = {}
    for method, g in users.groupby("method", sort=False):
        n_ins = int(g["is_insider"].sum())
        aps = [average_precision_score(f["is_insider"], f["user_score"]) for _, f in g.groupby("fold") if f["is_insider"].any()]
        row = {"user PR-AUC": f"{np.mean(aps):.3f} ± {np.std(aps):.3f}"}
        if windows is not None and len(windows):
            w = windows[windows["method"] == method]["window_pr_auc"].dropna()
            row["window PR-AUC"] = f"{w.mean():.3f} ± {w.std(ddof=0):.3f}"
        for b in budgets:
            top = g["rank"] <= np.maximum(1, np.round(b * g["n_fold"]))
            tp, nf = int((top & g["is_insider"]).sum()), int(top.sum())
            row[f"precision@{b:.0%}"] = round(tp / max(nf, 1), 3)
            row[f"recall@{b:.0%}"] = round(tp / max(n_ins, 1), 3)
        rows[method] = row
    return pd.DataFrame(rows).T


def best_method(users: pd.DataFrame, candidates: Sequence[str]) -> str:
    """The candidate with the highest mean per-fold user PR-AUC (used to pick 'best single model' for the bootstrap)."""
    from sklearn.metrics import average_precision_score
    def score(m):
        g = users[users["method"] == m]
        return np.mean([average_precision_score(f["is_insider"], f["user_score"]) for _, f in g.groupby("fold") if f["is_insider"].any()])
    return max([c for c in candidates if c in set(users["method"])], key=score)


def paired_bootstrap(users: pd.DataFrame, a: str, b: str, metric: str = "recall", budget: float = 0.10,
                     n_boot: int = 1000, seed: int = 0) -> dict:
    """
    Paired bootstrap over users (resampled within each fold, same users for both methods): difference a - b.
    metric = "recall" (at `budget`, flagging the top X% of each fold) or "pr_auc" (mean of per-fold user PR-AUC).
    """
    from sklearn.metrics import average_precision_score
    ua = users[users["method"] == a].set_index("user_id")
    ub = users[users["method"] == b].set_index("user_id").loc[ua.index]
    rng = np.random.RandomState(seed)
    folds = [np.flatnonzero(ua["fold"].to_numpy() == f) for f in np.unique(ua["fold"])]
    y = ua["is_insider"].to_numpy()
    sa, sb = ua["user_score"].to_numpy(), ub["user_score"].to_numpy()
    fa = (ua["rank"] <= np.maximum(1, np.round(budget * ua["n_fold"]))).to_numpy()
    fb = (ub["rank"] <= np.maximum(1, np.round(budget * ub["n_fold"]))).to_numpy()
    diffs = []
    for _ in range(n_boot):
        idx = [f[rng.randint(0, len(f), len(f))] for f in folds]
        if metric == "recall":
            ii = np.concatenate(idx)
            n_ins = max(int(y[ii].sum()), 1)
            diffs.append((fa[ii] & y[ii]).sum() / n_ins - (fb[ii] & y[ii]).sum() / n_ins)
        else:
            d = [average_precision_score(y[ix], sa[ix]) - average_precision_score(y[ix], sb[ix]) for ix in idx if y[ix].any()]
            diffs.append(np.mean(d))
    diffs = np.asarray(diffs)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {"a": a, "b": b, "metric": metric if metric != "recall" else f"recall@{budget:.0%}",
            "mean_diff": round(float(diffs.mean()), 4), "ci95": (round(float(lo), 4), round(float(hi), 4)),
            "p_diff_le_0": round(float((diffs <= 0).mean()), 3)}


# ------------------------------------------------------------------ production ensemble (single held-out split)

def train_production_ensemble(prep, sampler: str, seed: int, members: Sequence[str] = MEMBERS_DEFAULT, cfg=None,
                              n_inits: int = 2, window_policy=None, user_policy=None, user_agg: Optional[dict] = None,
                              artifact_dir: Optional[str] = None, make_split=None):
    """
    Fits every member on the winning sampler's output for one user-level split, blends them, tunes the window and user
    thresholds on VALIDATION and reports TEST. Returns (artifact_dict, table, per_method_results).
    The saved ensemble.pkl holds weights + stacker + scaler + thresholds; an API-side loader still has to be written.
    """
    from app.ml.lstm_detector import LSTMConfig
    from app.ml.train import DEFAULT_USER_POLICY, DEFAULT_WINDOW_POLICY
    from app.ml.train import make_split as _default_split
    make_split = make_split or _default_split
    cfg = cfg or LSTMConfig.production()
    T, F = prep.seq_len, len(prep.seq.feature_columns)
    wpol = ThresholdPolicy.coerce(window_policy) if window_policy is not None else DEFAULT_WINDOW_POLICY
    upol = ThresholdPolicy.coerce(user_policy) if user_policy is not None else DEFAULT_USER_POLICY
    agg = dict(user_agg or DEFAULT_AGG)
    sd = make_split(prep, seed)
    rr = resample(sampler, flatten(sd.X_train), sd.y_train, seed=seed, **kwargs_for(sampler, None))
    preds = fit_members(sd, rr, members, cfg, seed, n_inits, T, F, keep_models=True)
    uid = prep.features_df["user_id"].to_numpy()
    scores, coefs = blend_all(preds, sd.y_val, uid[sd.idx_val])
    results, rows = {}, {}
    for method, (pv, pt) in scores.items():
        thr = choose_threshold(sd.y_val, pv, wpol)
        ur = evaluate_user_level(prep, sd.idx_val, sd.idx_test, pv, pt, upol, {"sel": agg})["sel"]
        t = ur["test"]
        results[method] = {"window_threshold": thr, "window_test": compute_metrics(sd.y_test, pt, thr), "user": ur}
        rows[method] = {"window PR-AUC (test)": round(pr_auc(sd.y_test, pt), 3), "user PR-AUC (test)": t["pr_auc"],
                        "user precision": t["precision"], "user recall": t["recall"], "flagged": t["n_flagged"],
                        "precision@10": t.get("precision@10"), "recall@15": t.get("recall@15"), "recall@30": t.get("recall@30")}
    artifact = {"format": "ensemble-v1", "sampler": sampler, "members": list(preds), "stack_coefs": coefs,
                "member_models": {n: p["models"] for n, p in preds.items()}, "scaler": sd.scaler,
                "seq_len": T, "window_days": prep.window_days, "feature_columns": list(prep.seq.feature_columns),
                "user_agg": agg, "thresholds": {m: {"window": r["window_threshold"], "user": r["user"]["threshold"]} for m, r in results.items()},
                "seed": seed, "metrics": to_jsonable(results)}
    if artifact_dir:
        Path(artifact_dir).mkdir(parents=True, exist_ok=True)
        with open(Path(artifact_dir) / "ensemble.pkl", "wb") as fh:
            pickle.dump(artifact, fh)
    return artifact, pd.DataFrame(rows).T, results


Writing /kaggle/working/insider_code/app/ml/ensemble.py


In [18]:
%%writefile /kaggle/working/insider_code/app/ml/hgb_detector.py
"""
Layer-2 detector: gradient boosting (HistGradientBoostingClassifier). This is THE production Layer-2 model.

Why: on CERT r4.2 (7-fold user-level out-of-fold, 1000 users / 70 insiders) HGB on last / max / mean summaries of the
T-window history beat the LSTM, the BiLSTM-attention and every blend of them (user PR-AUC 0.976 vs 0.937 / 0.939 /
0.939 stack; window PR-AUC 0.855 vs 0.631). The stacked ensemble was significantly WORSE than HGB alone, so the
sequence models are kept only as an ablation (app/ml/ensemble.py, notebook section 7c).

Layer 1 (the sampler) is unchanged: the winning spec is applied to the TRAIN split only, then HGB is fitted on it.

  train_production_hgb   fit on one user-level split, tune window + user thresholds on VALIDATION, report TEST, save artifact
  HGBDetector            load an artifact and score windows / users
  run_oof_hgb            k-fold user-level out-of-fold scores for EVERY user (main evidence)
  budget_table           pooled precision / recall / F1 when the top X% of users of each fold are flagged
  matched_recall         precision / F1 at the smallest budget that reaches a target recall (paper comparison)
  importance_table       permutation importance per behavioural feature (explainability)
"""
import pickle
from pathlib import Path
from typing import Callable, Dict, Optional, Sequence

import numpy as np
import pandas as pd

from app.ml.metrics import ThresholdPolicy, choose_threshold, compute_metrics, pr_auc, to_jsonable
from app.ml.sampling import kwargs_for, resample
from app.ml.sequences import flatten
from app.ml.user_level import DEFAULT_AGG, aggregate_user_scores, evaluate_user_level

# Reference numbers reported by the selected paper (Asha, Shanmugapriya & Padmavathi 2023): NM-2 + One-Class SVM.
PAPER = {"recall": 1.0, "f1": 0.7872, "accuracy": 0.8246}
PAPER["precision_implied"] = round(PAPER["f1"] * PAPER["recall"] / (2 * PAPER["recall"] - PAPER["f1"]), 3)   # ~0.649


def _summary(X: np.ndarray) -> np.ndarray:
    from app.ml.ensemble import summarize_history
    return summarize_history(X)


def summary_columns(feature_columns: Sequence[str]) -> list:
    """Column names of summarize_history(): last-window, max and mean of every behavioural feature."""
    return [f"{c}__{s}" for s in ("last", "max", "mean") for c in feature_columns]


# ------------------------------------------------------------------ production model (one split)

def train_production_hgb(prep, sampler: str, seed: int, window_policy=None, user_policy=None,
                         user_agg: Optional[dict] = None, artifact_dir: Optional[str] = None,
                         val_frac: float = 0.15, test_frac: float = 0.15):
    """
    Fits HGB on the winning sampler's output for one user-level split. Both thresholds (window + user) are tuned on
    VALIDATION and frozen before TEST is touched. Returns (artifact, metrics, split_data).
    """
    from app.ml import ensemble
    from app.ml.train import DEFAULT_USER_POLICY, DEFAULT_WINDOW_POLICY, make_split
    T, F = prep.seq_len, len(prep.seq.feature_columns)
    wpol = ThresholdPolicy.coerce(window_policy) if window_policy is not None else DEFAULT_WINDOW_POLICY
    upol = ThresholdPolicy.coerce(user_policy) if user_policy is not None else DEFAULT_USER_POLICY
    agg = dict(user_agg or DEFAULT_AGG)
    sd = make_split(prep, seed, val_frac, test_frac)
    rr = resample(sampler, flatten(sd.X_train), sd.y_train, seed=seed, **kwargs_for(sampler, None))
    preds = ensemble.fit_members(sd, rr, ["hgb"], None, seed, 1, T, F, keep_models=True)
    clf, pv, pt = preds["hgb"]["models"][0], preds["hgb"]["val"], preds["hgb"]["test"]
    thr = choose_threshold(sd.y_val, pv, wpol)                           # tuned on validation, then frozen
    methods = {"max": {"method": "max"}, "topk": {"method": "topk", "k": 3}, "consecutive": {"method": "consecutive", "n": 3}}
    methods[agg["method"]] = agg
    user_res = evaluate_user_level(prep, sd.idx_val, sd.idx_test, pv, pt, upol, methods)
    chosen = user_res[agg["method"]]
    metrics = {
        "window_policy": wpol.__dict__, "user_policy": upol.__dict__, "user": user_res, "user_selected": agg["method"],
        "val": compute_metrics(sd.y_val, pv, thr), "test": compute_metrics(sd.y_test, pt, thr),
        "threshold": round(float(thr), 6), "train_size_after_sampling": int(len(rr.y)), "sampler_info": rr.info,
        "split": {"train_users": int(len(sd.split.train)), "val_users": int(len(sd.split.val)), "test_users": int(len(sd.split.test))},
    }
    artifact = {
        "format": "hgb-v1", "model": clf, "scaler": sd.scaler, "sampler": sampler, "seed": seed,
        "seq_len": T, "window_days": prep.window_days, "feature_columns": list(prep.seq.feature_columns),
        "window_threshold": float(thr), "user_threshold": chosen["threshold"], "user_agg": chosen["agg"],
        "metrics": to_jsonable(metrics),
    }
    if artifact_dir:
        Path(artifact_dir).mkdir(parents=True, exist_ok=True)
        with open(Path(artifact_dir) / "hgb.pkl", "wb") as fh:
            pickle.dump(artifact, fh)
    return artifact, metrics, sd


class HGBDetector:
    """Scores windows / users with a saved (or in-memory) hgb-v1 artifact. Inputs are SCALED (T, F) sequences."""

    def __init__(self, artifact: dict):
        if artifact.get("format") != "hgb-v1":
            raise ValueError("not an hgb-v1 artifact")
        self.a = artifact

    @classmethod
    def load(cls, path: str) -> "HGBDetector":
        p = Path(path)
        with open(p / "hgb.pkl" if p.is_dir() else p, "rb") as fh:
            return cls(pickle.load(fh))

    def score_windows(self, X_scaled: np.ndarray) -> np.ndarray:
        return self.a["model"].predict_proba(_summary(X_scaled))[:, 1]

    def score_users(self, user_ids, window_starts, X_scaled: np.ndarray) -> pd.DataFrame:
        """One row per user, sorted by score: user_score + flagged (stored aggregation and user threshold)."""
        agg = {k: v for k, v in self.a["user_agg"].items() if k in ("method", "k", "n")}
        s = aggregate_user_scores(user_ids, window_starts, self.score_windows(X_scaled),
                                  window_days=self.a["window_days"], **agg)
        out = s.sort_values(ascending=False).to_frame("user_score")
        out["flagged"] = out["user_score"] >= self.a["user_threshold"]
        return out


# ------------------------------------------------------------------ out-of-fold evaluation (main evidence)

def run_oof_hgb(prep, sampler: str = "none", k: int = 7, user_agg: Optional[dict] = None,
                progress: Callable[[str], None] = print):
    """
    Same user k-fold as the ensemble ablation (fold i = test, i+1 = validation, rest = train): every user is scored only
    by a model that never trained on them. Returns (users, windows); users has method == "hgb".
    """
    from app.ml import ensemble
    users, windows, _ = ensemble.run_oof_ensemble(prep, sampler, k=k, n_inits=1, members=["hgb"],
                                                  user_agg=user_agg, progress=progress)
    return users, windows


def _pooled(g: pd.DataFrame, top: pd.Series) -> dict:
    n_ins = int(g["is_insider"].sum())
    tp, nf = int((top & g["is_insider"]).sum()), int(top.sum())
    prec, rec = tp / max(nf, 1), tp / max(n_ins, 1)
    return {"flagged": nf, "caught": tp, "insiders": n_ins, "precision": round(prec, 3), "recall": round(rec, 3),
            "f1": round(2 * prec * rec / (prec + rec), 3) if prec + rec else 0.0}


def budget_table(users: pd.DataFrame, method: str = "hgb", budgets: Sequence[float] = (0.05, 0.10, 0.15, 0.20, 0.30)) -> pd.DataFrame:
    """Flag the top X% of users WITHIN EACH FOLD, pool the counts. No validation threshold has to carry over."""
    g = users[users["method"] == method]
    rows = {}
    for b in budgets:
        top = g["rank"] <= np.maximum(1, np.round(b * g["n_fold"]))
        rows[f"top {b:.0%}"] = _pooled(g, top)
    return pd.DataFrame(rows).T


def matched_recall(users: pd.DataFrame, method: str = "hgb", target: float = 1.0) -> dict:
    """Precision / F1 at the smallest per-fold budget (1% steps) whose pooled recall reaches `target`."""
    g = users[users["method"] == method]
    for b in np.arange(0.01, 1.0001, 0.01):
        top = g["rank"] <= np.maximum(1, np.round(b * g["n_fold"]))
        r = _pooled(g, top)
        if r["recall"] >= target - 1e-9:
            return {"budget": round(float(b), 2), **r}
    return {"budget": 1.0, **_pooled(g, g["rank"] > 0)}


def paper_comparison(users: pd.DataFrame, method: str = "hgb", budget: float = 0.10) -> pd.DataFrame:
    """Paper row next to ours at (a) a fixed analyst budget and (b) matched (100%) recall. Accuracy is NOT compared
    (user-level accuracy is dominated by the majority class and is not measured the same way as the paper's)."""
    g = users[users["method"] == method]
    fixed = _pooled(g, g["rank"] <= np.maximum(1, np.round(budget * g["n_fold"])))
    matched = matched_recall(users, method, 1.0)
    rows = {
        "paper (NM-2 + OCSVM)": {"flagged": np.nan, "recall": PAPER["recall"], "precision": PAPER["precision_implied"], "f1": PAPER["f1"]},
        f"{method} @ top {budget:.0%} of users": {k: fixed[k] for k in ("flagged", "recall", "precision", "f1")},
        f"{method} @ matched recall ({matched['budget']:.0%} budget)": {k: matched[k] for k in ("flagged", "recall", "precision", "f1")},
    }
    return pd.DataFrame(rows).T


# ------------------------------------------------------------------ explainability

def importance_table(artifact: dict, X_scaled: np.ndarray, y: np.ndarray, n_repeats: int = 3, seed: int = 0) -> pd.DataFrame:
    """
    Permutation importance (drop in window PR-AUC when one column is shuffled) on held-out windows, summed over the
    last / max / mean summaries of each behavioural feature. Global explanation for the alert-detail page.
    """
    from sklearn.inspection import permutation_importance
    Xs = _summary(X_scaled)
    r = permutation_importance(artifact["model"], Xs, y, scoring="average_precision", n_repeats=n_repeats,
                               random_state=seed, n_jobs=1)
    cols = summary_columns(artifact["feature_columns"])
    df = pd.DataFrame({"column": cols, "importance": r.importances_mean})
    df["feature"] = df["column"].str.rsplit("__", n=1).str[0]
    out = df.groupby("feature")["importance"].sum().sort_values(ascending=False).round(4)
    return out.to_frame("PR-AUC drop when shuffled")


Writing /kaggle/working/insider_code/app/ml/hgb_detector.py


In [19]:
%%writefile /kaggle/working/insider_code/app/ml/temporal_eval.py
"""
Temporal component of the final system (architecture.md section on temporal modelling).

The selected paper's stated gap: every window is scored independently, so slow escalation is invisible. Two ways of
closing it are built here and compared on the SAME 7-fold user-level out-of-fold setup as section 7b:

  1. Trend features inside HGB   summarize_trend(): last / max / mean (as before) PLUS, for every behavioural feature,
                                 the linear slope over the T-window history, last window minus the mean of the earlier
                                 ones, and the length of the trailing run of rising windows.
  2. CUSUM as a ranking score    cusum_peak_by_user(): one-sided CUSUM over each user's out-of-fold window scores
                                 (logit scale), against an EXPANDING per-user baseline (prior windows only, no
                                 lookahead). The user's score is the PEAK of the CUSUM statistic, so it ranks users
                                 and works with the alert-budget metrics -- no fixed alarm threshold h has to be
                                 carried over from synthetic data (that was the old failure: h=8 flagged every user
                                 on real CERT scores). The slack k is chosen per fold on VALIDATION users only.
                                 An equivalent alarm threshold h is frozen from the validation users' peaks.

Compared methods (all out-of-fold; every user is scored only by models that never trained on them):

  hgb                 baseline (identical to section 7b)
  hgb_trend           HGB with the trend features
  cusum               CUSUM peak over the baseline HGB window scores
  hgb+cusum           mean within-fold percentile rank of hgb (top-k user score) and cusum peak
  hgb_trend+cusum     same, using hgb_trend

Output frames have the same columns as ensemble.run_oof_ensemble, so ensemble.summarize_oof / paired_bootstrap work.
"""
from typing import Callable, Optional, Sequence

import numpy as np
import pandas as pd

from app.ml.metrics import pr_auc
from app.ml.sampling import kwargs_for, resample
from app.ml.sequences import flatten, kfold_user_splits, unflatten
from app.ml.user_level import DEFAULT_AGG, aggregate_user_scores, user_truth

K_GRID_DEFAULT = (0.25, 0.5, 1.0, 2.0)
MIN_BASELINE = 3          # windows of own history needed before a user's deviations accumulate
LOGIT_CLIP = 1e-6
STD_FLOOR = 0.5           # logit units; stops a near-constant user's tiny std from exploding z


# ------------------------------------------------------------------ 1. trend features

def summarize_trend(X: np.ndarray) -> np.ndarray:
    """(N, T, F) -> (N, 6F): last / max / mean (as summarize_history) + slope + last-minus-earlier-mean + rising run."""
    X = np.nan_to_num(X).astype(np.float64)
    N, T, F = X.shape
    base = np.concatenate([X[:, -1], X.max(1), X.mean(1)], axis=1)
    tc = np.arange(T, dtype=np.float64) - (T - 1) / 2.0
    slope = (X * tc[None, :, None]).sum(1) / (tc ** 2).sum()
    delta = X[:, -1] - X[:, :-1].mean(1) if T > 1 else np.zeros((N, F))
    rises = X[:, 1:] > X[:, :-1] + 1e-9                       # (N, T-1, F)
    run = np.zeros((N, F))
    alive = np.ones((N, F), dtype=bool)
    for j in range(T - 2, -1, -1):                            # trailing run of consecutive rises, newest first
        alive &= rises[:, j]
        run += alive
    return np.concatenate([base, slope, delta, run], axis=1).astype(np.float32)


def trend_columns(feature_columns: Sequence[str]) -> list:
    return [f"{c}__{s}" for s in ("last", "max", "mean", "slope", "delta", "rise_run") for c in feature_columns]


def _fit_hgb_trend(Xr: np.ndarray, yr: np.ndarray, pos_weight: Optional[float], seed: int):
    """Same hyper-parameters and class weighting as ensemble._fit_hgb; only the summary differs."""
    from sklearn.ensemble import HistGradientBoostingClassifier
    n_pos = max(int((yr == 1).sum()), 1)
    pw = min(50.0, float(pos_weight)) if pos_weight else min(50.0, float(np.sqrt((yr == 0).sum() / n_pos)))
    w = np.where(yr == 1, pw, 1.0)
    return HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, random_state=seed) \
        .fit(summarize_trend(Xr), yr, sample_weight=w)


# ------------------------------------------------------------------ 2. CUSUM peak score

def _logit(p: np.ndarray) -> np.ndarray:
    p = np.clip(np.asarray(p, dtype=np.float64), LOGIT_CLIP, 1 - LOGIT_CLIP)
    return np.log(p / (1 - p))


def cusum_series(x: np.ndarray, k: float) -> np.ndarray:
    """
    One-sided CUSUM of one user's chronologically ordered scores (logit scale).
    z[t] = (x[t] - mean(x[:t])) / max(std(x[:t]), STD_FLOOR)       (prior windows only; z = 0 until MIN_BASELINE)
    c[t] = max(0, c[t-1] + z[t] - k)
    """
    n = len(x)
    out = np.zeros(n)
    if n <= MIN_BASELINE:
        return out
    cs, cs2 = np.cumsum(x), np.cumsum(x * x)
    c = 0.0
    for t in range(MIN_BASELINE, n):
        m = cs[t - 1] / t
        var = max(cs2[t - 1] / t - m * m, 0.0)
        z = (x[t] - m) / max(np.sqrt(var), STD_FLOOR)
        c = max(0.0, c + z - k)
        out[t] = c
    return out


def cusum_peak_by_user(user_ids, window_starts, probs, k: float) -> pd.Series:
    """One score per user: the peak of that user's CUSUM statistic over the given window scores."""
    df = pd.DataFrame({"user": np.asarray(user_ids), "ws": pd.to_datetime(np.asarray(window_starts)),
                       "x": _logit(probs)}).sort_values(["user", "ws"], kind="stable")
    u, x = df["user"].to_numpy(), df["x"].to_numpy()
    bounds = np.flatnonzero(np.r_[True, u[1:] != u[:-1], True])
    idx, peaks = [], []
    for a, b in zip(bounds[:-1], bounds[1:]):
        idx.append(u[a])
        peaks.append(cusum_series(x[a:b], k).max())
    return pd.Series(peaks, index=pd.Index(idx, name="user"), dtype=float).sort_index()


# ------------------------------------------------------------------ out-of-fold comparison

def _pct(s: pd.Series) -> pd.Series:
    return s.rank(pct=True, method="average")


def _rows(st: pd.Series, truth: np.ndarray, fold: int, method: str) -> pd.DataFrame:
    return pd.DataFrame({
        "user_id": st.index.to_numpy(), "fold": fold, "method": method, "user_score": st.to_numpy(),
        "rank": st.rank(ascending=False, method="first").to_numpy(), "n_fold": len(st), "is_insider": truth.astype(bool)})


def run_oof_temporal(prep, sampler: str = "none", k: int = 7, user_agg: Optional[dict] = None,
                     k_grid: Sequence[float] = K_GRID_DEFAULT, make_split=None,
                     progress: Callable[[str], None] = print):
    """
    Returns (users, windows, window_scores, cusum_info):
      users          one row per (user, method) -- see module docstring for the methods
      windows        one row per (fold, hgb / hgb_trend): test window PR-AUC
      window_scores  every window of every user scored out-of-fold: user_id, window_start, fold, y, p_hgb, p_hgb_trend
      cusum_info     one row per (fold, base model): chosen slack k, frozen alarm threshold h (top-10% of validation
                     users' peaks), and the alarm's pooled result on the fold's test users
    """
    if make_split is None:
        from app.ml.train import make_split
    from app.ml import ensemble
    agg = user_agg or DEFAULT_AGG
    kw = {a: v for a, v in agg.items() if a in ("method", "k", "n")}
    T, F = prep.seq_len, len(prep.seq.feature_columns)
    uid, ws = prep.features_df["user_id"].to_numpy(), prep.seq.window_starts
    urows, wrows, srows, irows = [], [], [], []

    for i, sp in enumerate(kfold_user_splits(prep.unique_users, prep.user_is_insider, k)):
        progress(f"[oof-temporal:{sampler}] fold {i + 1}/{k}")
        sd = make_split(prep, i, split=sp)
        rr = resample(sampler, flatten(sd.X_train), sd.y_train, seed=i, **kwargs_for(sampler, None))
        base = ensemble.fit_members(sd, rr, ["hgb"], None, i, 1, T, F)["hgb"]            # identical to section 7b
        clf = _fit_hgb_trend(unflatten(rr.X, T, F), rr.y, rr.pos_weight, i)
        P = {"hgb": (base["val"], base["test"]),
             "hgb_trend": (clf.predict_proba(summarize_trend(sd.X_val))[:, 1],
                           clf.predict_proba(summarize_trend(sd.X_test))[:, 1])}

        srows.append(pd.DataFrame({"user_id": uid[sd.idx_test], "window_start": ws[sd.idx_test], "fold": i,
                                   "y": sd.y_test, "p_hgb": P["hgb"][1], "p_hgb_trend": P["hgb_trend"][1]}))
        score_val, score_test = {}, {}
        for name, (pv, pt) in P.items():
            sv = aggregate_user_scores(uid[sd.idx_val], ws[sd.idx_val], pv, window_days=prep.window_days, **kw)
            st = aggregate_user_scores(uid[sd.idx_test], ws[sd.idx_test], pt, window_days=prep.window_days, **kw)
            tv = user_truth(sv.index.to_numpy(), prep.unique_users, prep.user_is_insider)
            tt = user_truth(st.index.to_numpy(), prep.unique_users, prep.user_is_insider)
            urows.append(_rows(st, tt, i, name))
            wrows.append({"fold": i, "method": name, "window_pr_auc": pr_auc(sd.y_test, pt)})

            # CUSUM on this model's window scores: slack k chosen on VALIDATION users only
            best_k, best_ap = None, -1.0
            for kk in k_grid:
                ap = pr_auc(tv, cusum_peak_by_user(uid[sd.idx_val], ws[sd.idx_val], pv, kk).reindex(sv.index).to_numpy())
                if ap > best_ap:
                    best_k, best_ap = kk, ap
            cv = cusum_peak_by_user(uid[sd.idx_val], ws[sd.idx_val], pv, best_k).reindex(sv.index)
            ct = cusum_peak_by_user(uid[sd.idx_test], ws[sd.idx_test], pt, best_k).reindex(st.index)
            h = float(np.quantile(cv.to_numpy(), 0.90))                                  # frozen alarm threshold
            alarm = ct.to_numpy() > h
            irows.append({"fold": i, "base": name, "k": best_k, "h_frozen": round(h, 3), "val_user_pr_auc": round(best_ap, 3),
                          "flagged": int(alarm.sum()), "caught": int((alarm & (tt == 1)).sum()), "insiders": int(tt.sum())})
            score_val[name], score_test[name] = sv, st
            if name == "hgb":
                urows.append(_rows(ct, tt, i, "cusum"))
            urows.append(_rows((0.5 * (_pct(st) + _pct(ct))).rename(None), tt, i, f"{name}+cusum"))

    users = pd.concat(urows, ignore_index=True)
    return (users, pd.DataFrame(wrows), pd.concat(srows, ignore_index=True), pd.DataFrame(irows))


Writing /kaggle/working/insider_code/app/ml/temporal_eval.py


In [20]:
import sys, os, time, json, glob
for m in [m for m in sys.modules if m == "app" or m.startswith("app.")]:
    del sys.modules[m]                       # in case a package called `app` was already imported
sys.path.insert(0, "/kaggle/working/insider_code")

import numpy as np, pandas as pd, torch
from app.ml import train, offline_features as off, ensemble, hgb_detector as hgb, temporal_eval as tev
from app.ml.sequences import load_labels
try:
    from IPython.display import display, Markdown
except ImportError:                          # plain-python fallback
    display, Markdown = print, str

print("pandas", pd.__version__, "| numpy", np.__version__, "| torch", torch.__version__,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

pandas 2.3.3 | numpy 2.1.3 | torch 2.11.0+cpu | GPU: none (CPU)


## 2. Config — edit this cell

* `MAX_BENIGN_USERS` keeps **all insiders + a random sample of normal users**. This is what makes a 16 GB dataset
  tractable (time *and* memory). `None` = every user (slow; needs lots of RAM). **Trade-off:** it raises the share of
  insiders in val/test, so precision / PR-AUC come out *optimistic* versus the full population. The notebook prints this.
* `INTERNAL_DOMAIN`: the company e-mail domain (real CERT data uses `dtaa.com`). Mail to other domains counts as external.
* `LABELS_RELEASE`: only needed if your `insiders.csv` covers several CERT releases (e.g. `"r4.2"`).
* To **skip the slow pass** in a later session: run once, then attach this notebook's *output* as an input dataset and set
  `FEATURES_CACHE` to the `features_w2.pkl` inside it.

In [21]:
# ---------------- edit me ----------------
WINDOW_DAYS      = 2          # length of one time window
SEQ_LEN          = 7          # T = how many windows the LSTM looks back (7-14 suggested)
SEEDS            = [0]        # ignored while KFOLD is set
KFOLD            = 7          # user-level k-fold: every user (and all 70 insiders) is a test user exactly once. None = old random seeds
N_INITS          = 2          # ablation only (7d): LSTM initialisations averaged per fold (runtime ~ KFOLD x N_INITS trainings)
MAX_BENIGN_USERS = 1000       # None = keep every user
INTERNAL_DOMAIN  = "@dtaa.com"
RELEASE          = "r4.2"     # the ONLY CERT release used (labels and every evaluation); r6.2 is not used anywhere
LABELS_RELEASE   = None       # resolved below from RELEASE, using the spelling in your insiders.csv ("r4.2" or "4.2")
CHUNKSIZE        = 2_000_000  # CSV rows per chunk (lower it if you hit out-of-memory)

# --- Layer 1 sweep. Spec = base[@ratio][+cw][^power]; ratio = minority:majority AFTER resampling (0.05 = 1:20).
#     +cw = also weight the loss for the residual imbalance (default power 0.5); ^p softens a weight: (n_neg/n_pos)**p
#     Trim this list for a faster run (the 3 full-data candidates -- none / class_weight* -- dominate run time).
CANDIDATES = [
    "none", "class_weight^0.5",
    "RUS@0.05",                                    # diagnostic: random undersampling
    "NM-2@0.5",                                    # NM-2 kept ~half the majority; @0.02-0.1 trained on hard negatives only and collapsed
    "DeepSMOTE@0.05+cw",
]

LAYER1_EVALUATOR = "hgb"      # model that scores each sampler in Layer 1: "hgb" (same as production Layer 2, fast, no torch) | "lstm" (old proxy, ~35 min)

# --- Layer 2 = HGB (gradient boosting, architecture.md v3). Layer 1 above is unchanged.
REPORT_BUDGETS   = (0.05, 0.10, 0.15, 0.30)   # alert budgets for precision@ / recall@ (top X% of users in each fold)
PAPER_BUDGET     = 0.10       # analyst budget used in the paper-comparison table
RUN_IMPORTANCE   = True       # permutation feature importance of the production HGB (about a minute)

# --- Optional ablation (section 7d): do the sequence models or a blend beat HGB?  ~30 min extra; needs PyTorch.
RUN_ABLATION     = True       # section 7d: keeps LSTM / BiLSTM / blends as reported ablations (about 30 min)
ABLATION_MEMBERS = ["lstm", "bilstm_attn", "hgb"]   # blends (rank_avg, stack) are built from these; hgb is always included
ABLATION_VS_NONE = True       # also run the ablation with sampler="none" when the Layer 1 winner is something else
BOOTSTRAP_N      = 1000       # paired-bootstrap resamples (HGB minus each alternative)

# --- Temporal component (section 7e)
RUN_TEMPORAL     = True       # trend features + CUSUM peak score, compared with plain HGB (a few minutes, no torch needed)
TEMPORAL_K_GRID  = (0.25, 0.5, 1.0, 2.0)   # CUSUM slack candidates (std units); chosen per fold on VALIDATION users only

# --- Alert thresholds (chosen on VALIDATION only, then frozen).
#     budget = flag the top X of items. Depends on the many normal items, not on a few dozen positives, so it transfers
#     far better than a recall target. Other modes: "precision" (min_precision=0.3), "fbeta" (beta=0.5), "recall" (min_recall=0.9).
WINDOW_POLICY = dict(mode="budget", budget=0.01)    # window level: flag the top 1% of windows
USER_POLICY   = dict(mode="budget", budget=0.10)    # user level: analysts review the top 10% of users
USER_AGG      = dict(method="topk", k=3, n=3)       # "max" (old) | "topk" (mean of k highest windows) | "consecutive" (n windows in a row)
# ------------------------------------------

INPUT_ROOT = os.environ.get("INSIDER_INPUT", "/kaggle/input")
WORK       = os.environ.get("INSIDER_WORK", "/kaggle/working")
OUT_DIR    = f"{WORK}/insider_out"
os.makedirs(OUT_DIR, exist_ok=True)
FEATURES_CACHE = f"{OUT_DIR}/features_v3_w{WINDOW_DAYS}.pkl"     # or a path inside an attached dataset

DATA_DIR = off.find_dataset_dir(INPUT_ROOT)                   # folder that contains logon.csv / http.csv / ...
def _find(names):
    for n in names:
        hits = sorted(glob.glob(f"{INPUT_ROOT}/**/{n}", recursive=True))
        if hits: return hits[0]
ANSWERS = _find(["insiders.csv", "answers.csv"])              # ground truth; set manually if not found
print("CERT CSV folder :", DATA_DIR)
print("Ground truth    :", ANSWERS)
for f in off.SOURCE_FILES:
    p = os.path.join(DATA_DIR, f)
    if os.path.exists(p): print(f"  {f:11s} {os.path.getsize(p)/1e9:6.2f} GB")
assert ANSWERS, "Could not find insiders.csv / answers.csv -- set ANSWERS = '/kaggle/input/<dataset>/.../insiders.csv'"
raw = pd.read_csv(ANSWERS)
if "dataset" in raw.columns:                    # keep only RELEASE, matching the file's own spelling
    _norm = lambda v: str(v).strip().lower().lstrip("r")
    _hit = [v for v in raw["dataset"].astype(str).unique() if _norm(v) == _norm(RELEASE)]
    assert _hit, f"insiders.csv has no rows for {RELEASE}; it lists {sorted(raw['dataset'].astype(str).unique())}"
    LABELS_RELEASE = _hit[0]
    print(f"labels: {RELEASE} only -> dataset == {LABELS_RELEASE!r} ({int((raw['dataset'].astype(str) == LABELS_RELEASE).sum())} insider rows)")
else:
    LABELS_RELEASE = None
    print(f"insiders file has no 'dataset' column -> assuming it contains {RELEASE} only")
display(raw.head())

CERT CSV folder : /kaggle/input/datasets/andrihjonior/cert-insider-threat-dataset-r4-2/r4.2
Ground truth    : /kaggle/input/datasets/andrihjonior/cert-insider-threat-dataset-r4-2/answers/insiders.csv
  logon.csv     0.06 GB
  device.csv    0.03 GB
  email.csv     1.36 GB
  file.csv      0.19 GB
  http.csv     14.54 GB
labels: r4.2 only -> dataset == '4.2' (70 insider rows)


,dataset,scenario,details,user,start,end
0,2.0,1,r2.csv,ONS0995,3/6/2010 1:41:56,3/20/2010 8:10:12
1,3.1,1,r3.1-1.csv,CSF0929,07/01/2010 01:24:58,07/16/2010 06:52:00
2,3.1,2,r3.1-2.csv,CCH0959,08/02/2010 10:34:31,09/30/2010 15:04:03
3,3.2,1,r3.2-1.csv,RCW0822,09/29/2010 21:10:27,10/15/2010 06:34:52
4,3.2,2,r3.2-2.csv,JCE0258,07/12/2010 08:16:02,09/03/2010 16:16:29


## 3. Features — streamed from the raw CSVs (slow once, then cached)
Reads only the columns needed (never the huge `content` field), so memory stays around one chunk plus small aggregates.
Progress prints per chunk. Safe to re-run: if the cache file exists it is loaded instead.

In [22]:
labels   = load_labels(ANSWERS, dataset=LABELS_RELEASE)
insiders = set(labels.loc[labels["malicious"] == 1, "user"])

if os.path.exists(FEATURES_CACHE):
    features_df, user_codes, meta = off.load_features(FEATURES_CACHE)
    print("loaded cached features:", FEATURES_CACHE, meta)
    assert meta.get("window_days") in (None, WINDOW_DAYS), "cache was built with a different WINDOW_DAYS -- delete it"
    if meta.get("max_benign_users") != MAX_BENIGN_USERS:
        print(f"NOTE: cache was built with MAX_BENIGN_USERS={meta.get('max_benign_users')} (now {MAX_BENIGN_USERS}); delete the cache to rebuild")
else:
    all_users = off.list_users(DATA_DIR)
    keep = off.select_users(all_users, insiders, MAX_BENIGN_USERS) if MAX_BENIGN_USERS else None
    print(f"{len(all_users):,} users in the data; {len(insiders & set(all_users))} labeled insiders; keeping {len(keep) if keep else len(all_users):,}")
    features_df, user_codes = off.build_features_from_csvs(
        DATA_DIR, WINDOW_DAYS, INTERNAL_DOMAIN, keep, chunksize=CHUNKSIZE, progress=print)
    meta = {"window_days": WINDOW_DAYS, "internal_domain": INTERNAL_DOMAIN,
            "n_users_source": len(all_users), "max_benign_users": MAX_BENIGN_USERS}
    off.save_features(FEATURES_CACHE, features_df, user_codes, meta)
    print("cached ->", FEATURES_CACHE)
print("file_usb_count total:", int(features_df["file_usb_count"].sum()), "(0 => dead feature: CERT pc names never contain 'usb')")
print("email_attachment_count total:", int(features_df["email_attachment_count"].sum()), "vs email_count total:", int(features_df["email_count"].sum()))
from app.ml import sequences as seqmod
features_df, FEATURE_COLS = seqmod.engineer_features(features_df)   # per-user baseline z-scores + log1p counts
print(len(FEATURE_COLS), "model features")
display(features_df.head())

1,000 users in the data; 70 labeled insiders; keeping 1,000
[origin] logon.csv: earliest so far 2010-01-02 06:49:00
[origin] device.csv: earliest so far 2010-01-02 06:49:00
[origin] email.csv: earliest so far 2010-01-02 06:49:00
[origin] file.csv: earliest so far 2010-01-02 06:49:00
[origin] http.csv: earliest so far 2010-01-02 06:49:00
[features] window grid origin = 2010-01-02 06:49:00, window_days = 2
[features] logon.csv: 854,859 rows read, 854,859 kept (6s)
[features] finished logon.csv
[features] device.csv: 405,380 rows read, 405,380 kept (9s)
[features] finished device.csv
[features] email.csv: 2,000,000 rows read, 2,000,000 kept (45s)
[features] email.csv: 2,629,979 rows read, 2,629,979 kept (55s)
[features] finished email.csv
[features] file.csv: 445,581 rows read, 445,581 kept (61s)
[features] finished file.csv
[features] http.csv: 2,000,000 rows read, 2,000,000 kept (146s)
[features] http.csv: 4,000,000 rows read, 4,000,000 kept (228s)
[features] http.csv: 6,000,000 rows re

,user_id,window_start,window_end,logon_count,after_hours_logon_count,weekend_logon_count,distinct_pc_count,device_connect_count,email_count,email_external_ratio,...,file_usb_count_z,http_count_z,http_distinct_domains_z,after_hours_device_count_z,after_hours_file_count_z,after_hours_http_count_z,after_hours_email_count_z,http_leak_count_z,http_job_count_z,http_cloud_count_z
0,1,2010-01-02 06:49:00,2010-01-04 06:49:00,1.386294,0.693147,1.386294,0.693147,0.000000,2.302585,0.666667,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,2010-01-02 06:49:00,2010-01-04 06:49:00,1.791759,1.098612,1.791759,1.098612,1.386294,2.708050,0.285714,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,2010-01-02 06:49:00,2010-01-04 06:49:00,2.197225,0.000000,2.197225,1.098612,0.693147,2.944439,0.722222,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,2010-01-02 06:49:00,2010-01-04 06:49:00,1.098612,0.000000,1.098612,0.693147,2.079442,2.302585,0.444444,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,2010-01-02 06:49:00,2010-01-04 06:49:00,1.791759,0.000000,1.791759,0.693147,0.000000,1.945910,0.666667,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 4. Sequences, labels and the user-level split

In [23]:
prep = train.prepare_from_features(features_df, user_codes, ANSWERS, "cert", WINDOW_DAYS, SEQ_LEN, LABELS_RELEASE,
                                   feature_columns=FEATURE_COLS)
n_src, n_used = meta.get("n_users_source") or len(user_codes), len(set(user_codes.values()))
print(f"samples: {len(prep.y):,} | users: {len(prep.unique_users):,} (insiders: {int(prep.user_is_insider.sum())})"
      f" | malicious samples: {int(prep.y.sum()):,} ({prep.y.mean():.3%})")
if n_used < n_src:
    print(f"\nWARNING: {n_used:,} of {n_src:,} users used (all insiders kept). Insider prevalence is higher than in the "
          "full population, so precision / PR-AUC below are optimistic. Recall is unaffected.")
sd = train.make_split(prep, SEEDS[0])
print(f"seed {SEEDS[0]} split -> users train/val/test: {len(sd.split.train)}/{len(sd.split.val)}/{len(sd.split.test)}"
      f" | positive samples: {int(sd.y_train.sum())}/{int(sd.y_val.sum())}/{int(sd.y_test.sum())}")
del sd

samples: 198,608 | users: 1,000 (insiders: 70) | malicious samples: 839 (0.422%)
seed 0 split -> users train/val/test: 700/150/150 | positive samples: 550/153/136


## 5. Layer 1 — which sampler / ratio works best for HGB?
Each candidate touches **only the training users**; one HGB (the same model and class-weighting rule as production Layer 2) is trained per seed, or per fold under k-fold; candidates are ranked on the
**validation** users (test is reported, never used to choose). The sweep covers partial balance ratios (full 1:1 balancing
throws away ~99% of normal behaviour) and hybrids (resample a little, then weight the loss a little).
Rows marked `≈` are within one std of the best validation PR-AUC, i.e. **not separable from it with this few seeds**.
Thresholds use the alert-budget policy from the Config cell. A user-level table (max vs top-k vs sustained) follows.

In [24]:
t0 = time.time()
comp = train.run_sampler_comparison(prep, SEEDS, CANDIDATES, progress=print,
                                    window_policy=WINDOW_POLICY, user_policy=USER_POLICY, kfold=KFOLD,
                                    evaluator=LAYER1_EVALUATOR)
print(f"Layer 1 evaluator: {comp['evaluator']}")
print(f"\nLayer 1 took {(time.time()-t0)/60:.1f} min")

def _cell(c): return "n/a" if not c or c.get("mean") is None else f"{c['mean']:.3f} ± {c['std']:.3f}"
rows, best = [], None
for name, e in comp["candidates"].items():
    if e["summary"]:
        v, t = e["summary"]["val"], e["summary"]["test"]
        rows.append({"sampler": name, "vmean": v["pr_auc"]["mean"] or 0, "vstd": v["pr_auc"]["std"] or 0,
                     "val PR-AUC": _cell(v["pr_auc"]), "test PR-AUC": _cell(t["pr_auc"]),
                     "test recall": _cell(t["recall"]), "test precision": _cell(t["precision"]), "test F1": _cell(t["f1"]),
                     "test recall@budget": _cell(e["summary"]["test_at_budget"]["recall"]),
                     "test precision@budget": _cell(e["summary"]["test_at_budget"]["precision"])})
tab = pd.DataFrame(rows).sort_values("vmean", ascending=False)
top_v, top_s = tab.iloc[0]["vmean"], tab.iloc[0]["vstd"]
tab["sampler"] = [("≈ " if (r.vmean >= top_v - top_s and i > 0) else "") + r.sampler for i, r in enumerate(tab.itertuples())]
display(tab.drop(columns=["vmean", "vstd"]).set_index("sampler"))
print("WINNER:", comp["winner"], "-", comp["winner_reason"])
for n, why in comp["excluded"].items(): print("excluded:", n, "-", why)
for n, e in comp["candidates"].items():
    if e.get("convergence_note"): print(f"{n}:", e["convergence_note"])

# ---- user level (what an analyst actually sees): same candidates, three ways to turn window scores into a user score
urows = []
for name, e in comp["candidates"].items():
    for meth, d in (e["summary"] or {}).get("user", {}).items():
        t = d["test"]
        urows.append({"sampler": name, "aggregation": meth, "test PR-AUC": _cell(t["pr_auc"]), "precision": _cell(t["precision"]),
                      "recall": _cell(t["recall"]), "precision@10": _cell(t.get("precision@10")), "recall@15": _cell(t.get("recall@15")), "recall@30": _cell(t.get("recall@30")), "flagged": _cell(t["n_flagged"])})
print(f"\nUSER LEVEL (test users, flag top {USER_POLICY.get('budget', 0):.0%} of users; mean ± std over seeds)")
display(pd.DataFrame(urows).set_index(["sampler", "aggregation"]))


[layer1] none seed=0
[layer1] none seed=1
[layer1] none seed=2
[layer1] none seed=3
[layer1] none seed=4
[layer1] none seed=5
[layer1] none seed=6
[layer1] class_weight^0.5 seed=0
[layer1] class_weight^0.5 seed=1
[layer1] class_weight^0.5 seed=2
[layer1] class_weight^0.5 seed=3
[layer1] class_weight^0.5 seed=4
[layer1] class_weight^0.5 seed=5
[layer1] class_weight^0.5 seed=6
[layer1] RUS@0.05 seed=0
[layer1] RUS@0.05 seed=1
[layer1] RUS@0.05 seed=2
[layer1] RUS@0.05 seed=3
[layer1] RUS@0.05 seed=4
[layer1] RUS@0.05 seed=5
[layer1] RUS@0.05 seed=6
[layer1] NM-2@0.5 seed=0
[layer1] NM-2@0.5 seed=1
[layer1] NM-2@0.5 seed=2
[layer1] NM-2@0.5 seed=3
[layer1] NM-2@0.5 seed=4
[layer1] NM-2@0.5 seed=5
[layer1] NM-2@0.5 seed=6
[layer1] DeepSMOTE@0.05+cw seed=0
[layer1] DeepSMOTE@0.05+cw seed=1
[layer1] DeepSMOTE@0.05+cw seed=2
[layer1] DeepSMOTE@0.05+cw seed=3
[layer1] DeepSMOTE@0.05+cw seed=4
[layer1] DeepSMOTE@0.05+cw seed=5
[layer1] DeepSMOTE@0.05+cw seed=6
Layer 1 evaluator: hgb

Layer 1 to

,val PR-AUC,test PR-AUC,test recall,test precision,test F1,test recall@budget,test precision@budget
sampler,,,,,,,
DeepSMOTE@0.05+cw,0.856 ± 0.091,0.857 ± 0.078,0.918 ± 0.068,0.427 ± 0.164,0.569 ± 0.148,0.916 ± 0.072,0.383 ± 0.095
≈ none,0.853 ± 0.092,0.855 ± 0.062,0.926 ± 0.060,0.391 ± 0.159,0.535 ± 0.147,0.904 ± 0.070,0.377 ± 0.083
≈ class_weight^0.5,0.853 ± 0.092,0.855 ± 0.062,0.926 ± 0.060,0.391 ± 0.159,0.535 ± 0.147,0.904 ± 0.070,0.377 ± 0.083
≈ RUS@0.05,0.798 ± 0.133,0.818 ± 0.100,0.911 ± 0.062,0.405 ± 0.117,0.552 ± 0.118,0.905 ± 0.081,0.377 ± 0.085
NM-2@0.5,0.376 ± 0.254,0.345 ± 0.101,0.578 ± 0.177,0.236 ± 0.043,0.326 ± 0.050,0.585 ± 0.161,0.233 ± 0.038


WINNER: DeepSMOTE@0.05+cw - highest mean validation pr_auc = 0.8559
DeepSMOTE@0.05+cw: autoencoder converged in 7/7 seeds (mean reconstruction ratio 0.33)

USER LEVEL (test users, flag top 10% of users; mean ± std over seeds)


test PR-AUC      precision         recall  \
sampler           aggregation                                                
none              max          0.977 ± 0.024  0.689 ± 0.074  0.971 ± 0.049   
                  topk         0.976 ± 0.021  0.691 ± 0.089  0.971 ± 0.049   
                  consecutive  0.843 ± 0.088  0.636 ± 0.075  0.800 ± 0.129   
class_weight^0.5  max          0.977 ± 0.024  0.689 ± 0.074  0.971 ± 0.049   
                  topk         0.976 ± 0.021  0.691 ± 0.089  0.971 ± 0.049   
                  consecutive  0.843 ± 0.088  0.636 ± 0.075  0.800 ± 0.129   
RUS@0.05          max          0.982 ± 0.022  0.734 ± 0.088  0.971 ± 0.049   
                  topk         0.957 ± 0.024  0.716 ± 0.125  0.957 ± 0.053   
                  consecutive  0.846 ± 0.101  0.661 ± 0.102  0.843 ± 0.113   
NM-2@0.5          max          0.555 ± 0.106  0.441 ± 0.112  0.614 ± 0.135   
                  topk         0.547 ± 0.100  0.421 ± 0.084  0.586 ± 0.069   
                  consecutive  0.530 ± 0.120  0.401 ± 0.068  0.543 ± 0.127   
DeepSMOTE@0.05+cw max          0.982 ± 0.019  0.756 ± 0.096  0.957 ± 0.053   
                  topk         0.976 ± 0.020  0.753 ± 0.078  0.957 ± 0.053   
                  consecutive  0.837 ± 0.106  0.676 ± 0.106  0.771 ± 0.125   

                                precision@10      recall@15      recall@30  \
sampler           aggregation                                                
none              max          0.929 ± 0.076  0.971 ± 0.049  1.000 ± 0.000   
                  topk         0.914 ± 0.069  0.971 ± 0.049  1.000 ± 0.000   
                  consecutive  0.771 ± 0.111  0.843 ± 0.079  0.886 ± 0.090   
class_weight^0.5  max          0.929 ± 0.076  0.971 ± 0.049  1.000 ± 0.000   
                  topk         0.914 ± 0.069  0.971 ± 0.049  1.000 ± 0.000   
                  consecutive  0.771 ± 0.111  0.843 ± 0.079  0.886 ± 0.090   
RUS@0.05          max          0.929 ± 0.049  0.986 ± 0.038  1.000 ± 0.000   
                  topk         0.871 ± 0.076  0.986 ± 0.038  1.000 ± 0.000   
                  consecutive  0.814 ± 0.107  0.857 ± 0.098  0.886 ± 0.090   
NM-2@0.5          max          0.543 ± 0.113  0.643 ± 0.098  0.843 ± 0.053   
                  topk         0.500 ± 0.058  0.600 ± 0.082  0.771 ± 0.138   
                  consecutive  0.486 ± 0.135  0.571 ± 0.111  0.700 ± 0.116   
DeepSMOTE@0.05+cw max          0.943 ± 0.053  0.971 ± 0.049  1.000 ± 0.000   
                  topk         0.929 ± 0.049  0.957 ± 0.053  1.000 ± 0.000   
                  consecutive  0.757 ± 0.098  0.814 ± 0.121  0.886 ± 0.121   

                                      flagged  
sampler           aggregation                  
none              max          14.286 ± 2.059  
                  topk         14.286 ± 2.215  
                  consecutive  12.714 ± 2.430  
class_weight^0.5  max          14.286 ± 2.059  
                  topk         14.286 ± 2.215  
                  consecutive  12.714 ± 2.430  
RUS@0.05          max          13.429 ± 1.988  
                  topk         13.857 ± 3.288  
                  consecutive  13.000 ± 2.582  
NM-2@0.5          max          14.286 ± 3.251  
                  topk         14.429 ± 3.457  
                  consecutive  13.571 ± 2.225  
DeepSMOTE@0.05+cw max          12.857 ± 1.952  
                  topk         12.857 ± 1.773  
                  consecutive  11.714 ± 2.928

> ⚠ **Stale numbers:** the figures below came from the earlier run (Layer 1 scored with the LSTM). Re-run the notebook and rewrite this cell.

**Reading the output (Layer 1).** `none` and `class_weight^0.5` are tied within noise (test PR-AUC 0.567 vs 0.569); `none` wins on validation PR-AUC (0.624) so it is the
selected sampler. RUS and DeepSMOTE are slightly lower (0.531, 0.536). **NM-2@0.5, the paper's sampler, collapses** (test PR-AUC 0.061, recall 0.183), so resampling
did not help on r4.2.

## 6. Layer 2 — production HGB (gradient boosting)
One `HistGradientBoostingClassifier` on **last / max / mean over the T-window history** of the scaled features, trained once on the winning sampler's output
(softened class weight unless the sampler already rebalances). Both thresholds are tuned on **validation** with the alert-budget policy and frozen before the
test set is touched: a window threshold, and a user threshold on the aggregated user score (`USER_AGG`). The artifact (model, scaler, both thresholds,
aggregation, sampler) is saved to `/kaggle/working/insider_out/hgb_model/hgb.pkl`.

> Layer 1 now scores every sampler candidate with HGB itself (`LAYER1_EVALUATOR = "hgb"`), so the winner is the best sampler for the model that ships.

In [25]:
assert comp["winner"], "no eligible sampler: " + str(comp["excluded"])
hgb_art, m, sd_h = hgb.train_production_hgb(prep, comp["winner"], comp["seeds_used"][0], window_policy=WINDOW_POLICY,
                                            user_policy=USER_POLICY, user_agg=USER_AGG, artifact_dir=f"{OUT_DIR}/hgb_model")
print(f"Layer 2 = HGB | sampler = {hgb_art['sampler']} | window threshold = {hgb_art['window_threshold']:.4f} "
      f"| train windows after sampling = {m['train_size_after_sampling']:,}")
print("WINDOW LEVEL")
display(pd.DataFrame({"validation": m["val"], "TEST": m["test"]}).loc[["recall", "precision", "f1", "pr_auc", "accuracy", "n", "n_positive", "n_flagged"]])
print(f"USER LEVEL (selected: {m['user_selected']}; threshold {hgb_art['user_threshold']:.4f})")
ucols = ["recall", "precision", "f1", "pr_auc", "n_users", "n_insiders", "n_flagged", "true_flagged", "precision@5", "precision@10", "recall@10"]
display(pd.DataFrame({f"{k} ({s})": {c: m["user"][k][s].get(c) for c in ucols} for k in m["user"] for s in ("val", "test")}))

from app.ml.metrics import to_jsonable
json.dump(to_jsonable({"config": {"layer2": "hgb", "window_days": WINDOW_DAYS, "seq_len": SEQ_LEN, "seeds": SEEDS, "candidates": CANDIDATES,
                                   "window_policy": WINDOW_POLICY, "user_policy": USER_POLICY, "user_agg": USER_AGG,
                                   "max_benign_users": MAX_BENIGN_USERS, "users_used": n_used, "users_in_source": n_src},
                       "layer1": comp, "production": m}), open(f"{OUT_DIR}/results.json", "w"), indent=2)
print("saved ->", f"{OUT_DIR}/results.json", "and", f"{OUT_DIR}/hgb_model/")


Layer 2 = HGB | sampler = DeepSMOTE@0.05+cw | window threshold = 0.0764 | train windows after sampling = 145,427
WINDOW LEVEL


,validation,TEST
recall,0.8954,0.8015
precision,0.4613,0.5767
f1,0.6089,0.6708
pr_auc,0.8825,0.7808
accuracy,0.9941,0.9964
n,29656.0000,29900.0000
n_positive,153.0000,136.0000
n_flagged,297.0000,189.0000


USER LEVEL (selected: topk; threshold 0.0868)


,max (val),max (test),topk (val),topk (test),consecutive (val),consecutive (test)
recall,1.0000,1.0000,1.0000,1.0000,0.9000,1.0000
precision,0.6667,0.7692,0.6667,0.7692,0.6000,0.7143
f1,0.8000,0.8696,0.8000,0.8696,0.7200,0.8333
pr_auc,0.9909,0.9769,0.9769,0.9769,0.8977,0.9532
n_users,150.0000,150.0000,150.0000,150.0000,150.0000,150.0000
n_insiders,10.0000,10.0000,10.0000,10.0000,10.0000,10.0000
n_flagged,15.0000,13.0000,15.0000,13.0000,15.0000,14.0000
true_flagged,10.0000,10.0000,10.0000,10.0000,9.0000,10.0000
precision@5,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
precision@10,0.9000,0.9000,0.9000,0.9000,0.8000,0.8000


saved -> /kaggle/working/insider_out/results.json and /kaggle/working/insider_out/hgb_model/


> ⚠ **Stale numbers:** the figures below came from the earlier run (Layer 1 scored with the LSTM). Re-run the notebook and rewrite this cell.

**Reading the output (one held-out split).** At user level the 150 test users contain 10 insiders; all 10 are flagged with 14 users flagged (precision 0.714, recall 1.0).
At window level test recall is 0.779, precision 0.558, PR-AUC 0.800. This is one split with ten insiders, so treat it as an illustration; **7b is the main evidence.**

## 7. Who does the model flag? (held-out TEST users only)
Scores only the test users of the production split (never seen in training) and applies the stored user-level aggregation + threshold.

In [26]:
det = hgb.HGBDetector(hgb_art)
ranked = det.score_users(prep.features_df["user_id"].to_numpy()[sd_h.idx_test], prep.seq.window_starts[sd_h.idx_test], sd_h.X_test)
ranked["user"] = ranked.index.map(user_codes)
ranked["is_labeled_insider"] = ranked["user"].isin(insiders)
ranked = ranked.set_index("user")
display(ranked.head(25))
nf, ni, nt = int(ranked["flagged"].sum()), int(ranked["is_labeled_insider"].sum()), len(ranked)
tp = int((ranked["flagged"] & ranked["is_labeled_insider"]).sum())
print(f"test users: {nt} ({ni} insiders) | flagged: {nf} | true insiders among flagged: {tp} "
      f"| precision {tp/max(nf,1):.2f} | recall {tp/max(ni,1):.2f} | aggregation {hgb_art['user_agg']} | user threshold {hgb_art['user_threshold']:.4f}")


,user_score,flagged,is_labeled_insider
user,,,
EGD0132,0.998565,True,True
VSS0154,0.980063,True,True
BTL0226,0.977014,True,True
AAM0658,0.953715,True,True
AKR0057,0.922080,True,True
RMW0542,0.851921,True,True
FTM0406,0.786670,True,True
GHL0460,0.733186,True,True
KLH0596,0.338006,True,True


test users: 150 (10 insiders) | flagged: 13 | true insiders among flagged: 10 | precision 0.77 | recall 1.00 | aggregation {'method': 'topk', 'k': 3, 'n': 3} | user threshold 0.0868


> ⚠ **Stale numbers:** the figures below came from the earlier run (Layer 1 scored with the LSTM). Re-run the notebook and rewrite this cell.

**Reading the output (ranking).** Eight insiders score above 0.84. One insider (MOS0047) only just clears the frozen threshold (0.138 vs 0.123), and four benign users are
flagged at scores between 0.14 and 0.40.

## 7b. Out-of-fold scores for EVERY user (HGB) — the main evidence
One HGB per fold (fold *i* = test, fold *i+1* = validation, rest = train); every user (all 70 insiders) is scored only by a model that never trained on them.
Metrics flag the **top X% of users in each fold** and pool the counts, so no validation threshold has to carry over. The **paper comparison** sets our
numbers next to the paper's (recall 1.00, F1 0.787, implied precision ≈ 0.65) at a fixed analyst budget and at matched (100%) recall. Accuracy is not compared:
user-level accuracy is dominated by the majority class and is not measured the same way as in the paper. With 70 insiders in 1000 users every figure carries wide
uncertainty, and the paper's evaluation protocol differs from this user-level k-fold, so read the comparison as indicative.

In [27]:
t0 = time.time()
users_h, wins_h = hgb.run_oof_hgb(prep, comp["winner"], k=KFOLD or 7, user_agg=USER_AGG, progress=print)
n_ins = int(prep.user_is_insider.sum())
print(f"HGB OOF took {(time.time()-t0)/60:.1f} min | users scored: {users_h['user_id'].nunique()} | insiders: {n_ins} | sampler: {comp['winner']}")
display(Markdown("**Per-fold user / window PR-AUC and fixed-budget precision / recall**"))
display(ensemble.summarize_oof(users_h, wins_h, REPORT_BUDGETS))
display(Markdown("**Pooled counts when the top X% of users of each fold are flagged**"))
display(hgb.budget_table(users_h, "hgb", (0.05, 0.10, 0.15, 0.20, 0.30)))
display(Markdown(f"**Paper comparison** (paper: NM-2 + One-Class SVM; precision is implied by its recall and F1)"))
display(hgb.paper_comparison(users_h, "hgb", PAPER_BUDGET))
oof_out = users_h.assign(user=users_h["user_id"].map(user_codes)).sort_values("user_score", ascending=False)
oof_out.to_csv(f"{OUT_DIR}/oof_user_scores_hgb.csv", index=False)
display(oof_out.head(25))


[oof-ens:DeepSMOTE@0.05+cw] fold 1/7
[oof-ens:DeepSMOTE@0.05+cw] fold 2/7
[oof-ens:DeepSMOTE@0.05+cw] fold 3/7
[oof-ens:DeepSMOTE@0.05+cw] fold 4/7
[oof-ens:DeepSMOTE@0.05+cw] fold 5/7
[oof-ens:DeepSMOTE@0.05+cw] fold 6/7
[oof-ens:DeepSMOTE@0.05+cw] fold 7/7
HGB OOF took 12.0 min | users scored: 1000 | insiders: 70 | sampler: DeepSMOTE@0.05+cw


**Per-fold user / window PR-AUC and fixed-budget precision / recall**

,user PR-AUC,window PR-AUC,precision@5%,recall@5%,precision@10%,recall@10%,precision@15%,recall@15%,precision@30%,recall@30%
hgb,0.976 ± 0.018,0.857 ± 0.072,1.0,0.7,0.684,0.957,0.476,1.0,0.233,1.0


**Pooled counts when the top X% of users of each fold are flagged**

,flagged,caught,insiders,precision,recall,f1
top 5%,49.0,49.0,70.0,1.000,0.700,0.824
top 10%,98.0,67.0,70.0,0.684,0.957,0.798
top 15%,147.0,70.0,70.0,0.476,1.000,0.645
top 20%,202.0,70.0,70.0,0.347,1.000,0.515
top 30%,301.0,70.0,70.0,0.233,1.000,0.377


**Paper comparison** (paper: NM-2 + One-Class SVM; precision is implied by its recall and F1)

,flagged,recall,precision,f1
paper (NM-2 + OCSVM),NaN,1.000,0.649,0.7872
hgb @ top 10% of users,98.0,0.957,0.684,0.7980
hgb @ matched recall (13% budget),132.0,1.000,0.530,0.6930


,user_id,fold,method,user_score,rank,n_fold,is_insider,user
447,137,3,hgb,0.999446,1.0,143,True,HXL0968
45,355,0,hgb,0.998747,1.0,143,True,BDV0168
267,868,1,hgb,0.998722,1.0,143,True,EGD0132
29,278,0,hgb,0.998666,2.0,143,True,RAR0725
47,371,0,hgb,0.998268,3.0,143,True,IKR0401
806,669,5,hgb,0.998015,1.0,143,True,LQC0479
547,802,3,hgb,0.997546,2.0,143,True,MDH0580
982,854,6,hgb,0.997520,1.0,142,True,PNL0301
492,461,3,hgb,0.997222,3.0,143,True,CCL0068
915,333,6,hgb,0.996454,2.0,142,True,EDB0714


> ⚠ **Stale numbers:** the figures below came from the earlier run (Layer 1 scored with the LSTM). Re-run the notebook and rewrite this cell.

**Reading the output (7b, all 1000 users scored out-of-fold).** User PR-AUC is 0.976 ± 0.019 and window PR-AUC 0.855 ± 0.057.
Flagging the top 5% of users (49) catches 49 insiders (precision 1.0, recall 0.70). Flagging the top 10% (98 users) catches 68 of 70 (precision 0.694, recall 0.971, F1 0.810),
and the top 15% catches all 70.

**Against the paper** (recall 1.0, implied precision 0.649, F1 0.787): at a 10% budget HGB has higher precision and F1 but slightly lower recall. At matched recall of 1.0 it needs
a 14% budget and precision falls to 0.500, below the paper's 0.649. The honest claim is a better precision/F1 trade-off at a fixed analyst budget, not a strict win.
The paper's split and sampling differ from ours, so the comparison is indicative only.

## 7c. What drives the score? (permutation importance, held-out test windows)
How much window PR-AUC drops when one behavioural feature is shuffled, summed over its last / max / mean summaries. A global explanation for the alert-detail page;
correlated features share credit, so read it as a ranking, not exact attribution.

In [28]:
if RUN_IMPORTANCE:
    t0 = time.time()
    imp_df = hgb.importance_table(hgb_art, sd_h.X_test, sd_h.y_test)
    imp_df.to_csv(f"{OUT_DIR}/hgb_feature_importance.csv")
    print(f"took {time.time()-t0:.0f}s")
    display(imp_df.head(15))
else:
    print("skipped (RUN_IMPORTANCE = False)")


took 459s


,PR-AUC drop when shuffled
feature,
device_connect_count,0.2256
http_count,0.1612
http_job_count_z,0.1363
device_connect_count_z,0.1009
http_count_z,0.0478
http_job_count,0.0465
logon_count,0.0198
email_count,0.0159
email_count_z,0.0149


> ⚠ **Stale numbers:** the figures below came from the earlier run (Layer 1 scored with the LSTM). Re-run the notebook and rewrite this cell.

**Reading the output (7c).** Four features carry most of the signal: `device_connect_count` (0.272), `http_count` (0.189), `http_job_count_z` (0.140) and `device_connect_count_z` (0.098).
The rest are below 0.04. `file_usb_count` is all zeros in r4.2 (PC names never contain "usb"), so it is a dead feature.

## 7d. Ablation — do the sequence models or a blend beat HGB?
Same user k-fold as 7b, every user scored only by models that never trained on them. Members are the single LSTM, a BiLSTM with attention and HGB; blends are
`rank_avg` (mean percentile rank) and `stack` (logistic regression on member logits, fitted on each fold's validation users). **Paired bootstrap = HGB minus the
alternative** on recall at 10% budget and per-fold user PR-AUC: a positive difference with a CI above 0 means HGB is better. This cell is the evidence for keeping Layer 2 to HGB only.

In [29]:
if not RUN_ABLATION:
    print("skipped (RUN_ABLATION = False)")
else:
    from app.ml.lstm_detector import LSTMConfig
    oof_cfg = LSTMConfig(hidden_size=64, num_layers=1, dropout=0.2, epochs=25, patience=5)    # lighter than production: KFOLD x N_INITS trainings
    members = list(dict.fromkeys(list(ABLATION_MEMBERS) + ["hgb"]))
    samplers = [comp["winner"]] + (["none"] if ABLATION_VS_NONE and comp["winner"] != "none" else [])
    t0, runs = time.time(), {}
    for s in samplers:
        runs[s] = ensemble.run_oof_ensemble(prep, s, k=KFOLD or 7, cfg=oof_cfg, n_inits=N_INITS, members=members, user_agg=USER_AGG, progress=print)
    print(f"ablation OOF took {(time.time()-t0)/60:.1f} min")
    n_ins = int(prep.user_is_insider.sum())
    for s, (users, wins, coefs) in runs.items():
        display(Markdown(f"**Sampler `{s}`" + (" (Layer 1 winner)**" if s == comp["winner"] else " (no resampling)**") + f" -- {users['user_id'].nunique()} users, {n_ins} insiders"))
        display(ensemble.summarize_oof(users, wins, REPORT_BUDGETS))
        if len(coefs):
            print("stacker weights per fold (mean):", coefs.drop(columns="fold").mean().round(3).to_dict())
        users.assign(user=users["user_id"].map(user_codes)).to_csv(f"{OUT_DIR}/oof_ablation_{s.replace('/', '_')}.csv", index=False)

    users_w = runs[comp["winner"]][0]
    print(f"\nPaired bootstrap, sampler {comp['winner']} ({BOOTSTRAP_N} resamples): HGB minus ... (positive = HGB better)")
    for ref in ("lstm", "bilstm_attn", "rank_avg", "stack"):
        if ref not in set(users_w["method"]): continue
        for metric in ("recall", "pr_auc"):
            r = ensemble.paired_bootstrap(users_w, "hgb", ref, metric=metric, budget=0.10, n_boot=BOOTSTRAP_N)
            print(f"  vs {ref:12s} {r['metric']:11s} diff {r['mean_diff']:+.3f}  95% CI {r['ci95']}  P(diff<=0) = {r['p_diff_le_0']}")


[oof-ens:DeepSMOTE@0.05+cw] fold 1/7
[oof-ens:DeepSMOTE@0.05+cw] fold 2/7
[oof-ens:DeepSMOTE@0.05+cw] fold 3/7
[oof-ens:DeepSMOTE@0.05+cw] fold 4/7
[oof-ens:DeepSMOTE@0.05+cw] fold 5/7
[oof-ens:DeepSMOTE@0.05+cw] fold 6/7
[oof-ens:DeepSMOTE@0.05+cw] fold 7/7
[oof-ens:none] fold 1/7
[oof-ens:none] fold 2/7
[oof-ens:none] fold 3/7
[oof-ens:none] fold 4/7
[oof-ens:none] fold 5/7
[oof-ens:none] fold 6/7
[oof-ens:none] fold 7/7
ablation OOF took 139.5 min


**Sampler `DeepSMOTE@0.05+cw` (Layer 1 winner)** -- 1000 users, 70 insiders

,user PR-AUC,window PR-AUC,precision@5%,recall@5%,precision@10%,recall@10%,precision@15%,recall@15%,precision@30%,recall@30%
lstm,0.943 ± 0.037,0.609 ± 0.145,0.98,0.686,0.694,0.971,0.469,0.986,0.233,1.0
bilstm_attn,0.930 ± 0.036,0.581 ± 0.135,0.959,0.671,0.704,0.986,0.469,0.986,0.229,0.986
hgb,0.976 ± 0.018,0.857 ± 0.072,1.0,0.7,0.684,0.957,0.476,1.0,0.233,1.0
rank_avg,0.898 ± 0.070,0.715 ± 0.123,0.959,0.671,0.622,0.871,0.422,0.886,0.223,0.957
stack,0.955 ± 0.031,0.827 ± 0.085,1.0,0.7,0.653,0.914,0.463,0.971,0.233,1.0


stacker weights per fold (mean): {'lstm': 0.255, 'bilstm_attn': 0.036, 'hgb': 1.473}


**Sampler `none` (no resampling)** -- 1000 users, 70 insiders

,user PR-AUC,window PR-AUC,precision@5%,recall@5%,precision@10%,recall@10%,precision@15%,recall@15%,precision@30%,recall@30%
lstm,0.937 ± 0.034,0.631 ± 0.145,0.939,0.657,0.673,0.943,0.469,0.986,0.233,1.0
bilstm_attn,0.938 ± 0.040,0.622 ± 0.141,0.959,0.671,0.684,0.957,0.463,0.971,0.229,0.986
hgb,0.976 ± 0.019,0.855 ± 0.057,1.0,0.7,0.694,0.971,0.476,1.0,0.233,1.0
rank_avg,0.896 ± 0.075,0.734 ± 0.133,0.959,0.671,0.622,0.871,0.429,0.9,0.229,0.986
stack,0.941 ± 0.031,0.785 ± 0.117,0.959,0.671,0.653,0.914,0.476,1.0,0.233,1.0


stacker weights per fold (mean): {'lstm': 0.089, 'bilstm_attn': 0.255, 'hgb': 1.241}

Paired bootstrap, sampler DeepSMOTE@0.05+cw (1000 resamples): HGB minus ... (positive = HGB better)
  vs lstm         recall@10%  diff -0.015  95% CI (-0.0645, 0.0323)  P(diff<=0) = 0.822
  vs lstm         pr_auc      diff +0.032  95% CI (-0.0063, 0.0795)  P(diff<=0) = 0.058
  vs bilstm_attn  recall@10%  diff -0.029  95% CI (-0.0704, 0.0)  P(diff<=0) = 1.0
  vs bilstm_attn  pr_auc      diff +0.044  95% CI (0.0062, 0.0948)  P(diff<=0) = 0.01
  vs rank_avg     recall@10%  diff +0.082  95% CI (0.0133, 0.1572)  P(diff<=0) = 0.017
  vs rank_avg     pr_auc      diff +0.075  95% CI (0.0249, 0.1402)  P(diff<=0) = 0.001
  vs stack        recall@10%  diff +0.040  95% CI (-0.0161, 0.1067)  P(diff<=0) = 0.142
  vs stack        pr_auc      diff +0.020  95% CI (-0.0089, 0.0572)  P(diff<=0) = 0.089


> ⚠ **Stale numbers:** the figures below came from the earlier run (Layer 1 scored with the LSTM). Re-run the notebook and rewrite this cell.

**Reading the output (7d, 7-fold out-of-fold).** User PR-AUC: HGB 0.976, stack 0.941, BiLSTM-attention 0.938, LSTM 0.937, rank-average 0.896. HGB is significantly better on PR-AUC
than every alternative (all bootstrap CIs above 0). On recall at 10% HGB is significantly better than the stack (+0.055) and rank-average (+0.097) but **statistically tied with the LSTM
(+0.029, CI 0 to 0.078) and BiLSTM (+0.014, CI 0 to 0.05)**. The stacker puts almost all of its weight on HGB (1.241 vs 0.255 and 0.089). Conclusion: HGB-only, with the sequence
models kept as an ablation.

## 7e. Temporal component -- does it beat plain HGB?
The paper scores every window independently (its stated gap). Two ways of closing it, compared with the same 7-fold user-level out-of-fold setup, sampler and alert-budget metrics as 7b:

* **`hgb_trend`** -- HGB whose summary of the T-window history also contains the slope, last-minus-earlier-mean and rising-run length of every feature.
* **`cusum`** -- one-sided CUSUM over each user's out-of-fold window scores (logit scale, expanding per-user baseline, no lookahead); the user's score is the **peak** of the statistic, so it ranks users and fits the top-X% alert budget. Slack `k` is chosen per fold on validation users only; an equivalent alarm threshold `h` is frozen from validation peaks.
* **`hgb+cusum`, `hgb_trend+cusum`** -- mean within-fold percentile rank of the HGB user score and the CUSUM peak.

The `hgb` row must reproduce 7b exactly (same models, same folds). The verdict at the bottom says whether the temporal component is *supported* (95% bootstrap CI above 0 against `hgb`) or should be reported as a negative result.

In [30]:
if not RUN_TEMPORAL:
    print("skipped (RUN_TEMPORAL = False)")
else:
    t0 = time.time()
    users_t, wins_t, wscores_t, cinfo_t = tev.run_oof_temporal(prep, comp["winner"], k=KFOLD or 7, user_agg=USER_AGG,
                                                               k_grid=TEMPORAL_K_GRID, progress=print)
    print(f"temporal OOF took {(time.time()-t0)/60:.1f} min | users scored: {users_t['user_id'].nunique()} | sampler: {comp['winner']}")

    # consistency: the plain-HGB row must equal section 7b
    try:
        a = users_t[users_t["method"] == "hgb"].set_index("user_id")["user_score"].sort_index()
        b = users_h[users_h["method"] == "hgb"].set_index("user_id")["user_score"].sort_index()
        print("hgb row identical to 7b:", bool(np.allclose(a.to_numpy(), b.reindex(a.index).to_numpy())))
    except NameError:
        print("(run 7b first to check the hgb row against it)")

    order = ["hgb", "hgb_trend", "cusum", "hgb+cusum", "hgb_trend+cusum"]
    tbl = ensemble.summarize_oof(users_t, wins_t, REPORT_BUDGETS).reindex(order).astype(object).where(lambda d: d != "nan ± nan", "-")
    display(Markdown("**Out-of-fold, every user scored once (user PR-AUC: mean +/- std over folds; precision/recall pooled over folds)**"))
    display(tbl)

    display(Markdown("**CUSUM alarm (frozen on validation): chosen slack k, threshold h and pooled result on test users**"))
    display(cinfo_t)
    al = cinfo_t.groupby("base")[["flagged", "caught", "insiders"]].sum()
    al["precision"] = (al["caught"] / al["flagged"].clip(lower=1)).round(3)
    al["recall"] = (al["caught"] / al["insiders"]).round(3)
    display(al)

    display(Markdown(f"**Paired bootstrap ({BOOTSTRAP_N} resamples): candidate minus `hgb` (positive = candidate better)**"))
    verdict, boot = {}, {}
    for cand in order[1:]:
        for metric in ("recall", "pr_auc"):
            r = ensemble.paired_bootstrap(users_t, cand, "hgb", metric=metric, budget=0.10, n_boot=BOOTSTRAP_N)
            boot[f"{cand}|{r['metric']}"] = r
            print(f"  {cand:16s} {r['metric']:11s} diff {r['mean_diff']:+.3f}  95% CI {r['ci95']}  P(diff<=0) = {r['p_diff_le_0']}")
            verdict[cand] = verdict.get(cand, False) or r["ci95"][0] > 0
    print()
    wins = [c for c, ok in verdict.items() if ok]
    if wins:
        print("VERDICT: temporal component SUPPORTED by:", ", ".join(wins), "(95% CI above 0 on recall@10% or user PR-AUC vs plain HGB)")
    else:
        print("VERDICT: temporal component NOT supported on r4.2 -- no variant separates from plain HGB (95% CI includes 0). "
              "Report it as a documented negative result; the production Layer 2 stays plain HGB.")

    # outputs
    wscores_t.assign(user=wscores_t["user_id"].map(user_codes)).to_csv(f"{OUT_DIR}/oof_window_scores_hgb.csv", index=False)
    users_t.assign(user=users_t["user_id"].map(user_codes)).to_csv(f"{OUT_DIR}/oof_temporal_users.csv", index=False)
    cinfo_t.to_csv(f"{OUT_DIR}/cusum_info.csv", index=False)
    from app.ml.metrics import to_jsonable
    json.dump(to_jsonable({"sampler": comp["winner"], "table": tbl.to_dict(), "cusum_alarm": al.to_dict(), "bootstrap": boot,
                           "temporal_supported_by": wins}), open(f"{OUT_DIR}/results_temporal.json", "w"), indent=2)
    print("saved: oof_window_scores_hgb.csv, oof_temporal_users.csv, cusum_info.csv, results_temporal.json ->", OUT_DIR)


[oof-temporal:DeepSMOTE@0.05+cw] fold 1/7
[oof-temporal:DeepSMOTE@0.05+cw] fold 2/7
[oof-temporal:DeepSMOTE@0.05+cw] fold 3/7
[oof-temporal:DeepSMOTE@0.05+cw] fold 4/7
[oof-temporal:DeepSMOTE@0.05+cw] fold 5/7
[oof-temporal:DeepSMOTE@0.05+cw] fold 6/7
[oof-temporal:DeepSMOTE@0.05+cw] fold 7/7
temporal OOF took 15.7 min | users scored: 1000 | sampler: DeepSMOTE@0.05+cw
hgb row identical to 7b: True


**Out-of-fold, every user scored once (user PR-AUC: mean +/- std over folds; precision/recall pooled over folds)**

,user PR-AUC,window PR-AUC,precision@5%,recall@5%,precision@10%,recall@10%,precision@15%,recall@15%,precision@30%,recall@30%
hgb,0.976 ± 0.018,0.857 ± 0.072,1.0,0.7,0.684,0.957,0.476,1.0,0.233,1.0
hgb_trend,0.986 ± 0.020,0.877 ± 0.067,1.0,0.7,0.694,0.971,0.476,1.0,0.233,1.0
cusum,0.820 ± 0.115,-,0.878,0.614,0.571,0.8,0.422,0.886,0.213,0.914
hgb+cusum,0.930 ± 0.055,-,0.98,0.686,0.663,0.929,0.456,0.957,0.226,0.971
hgb_trend+cusum,0.952 ± 0.032,-,1.0,0.7,0.673,0.943,0.463,0.971,0.229,0.986


**CUSUM alarm (frozen on validation): chosen slack k, threshold h and pooled result on test users**

,fold,base,k,h_frozen,val_user_pr_auc,flagged,caught,insiders
0,0,hgb,2.0,9.598,0.794,21,10,10
1,0,hgb_trend,2.0,9.232,0.816,24,9,10
2,1,hgb,2.0,12.534,0.738,12,7,10
3,1,hgb_trend,2.0,13.794,0.774,11,7,10
4,2,hgb,2.0,9.429,0.939,21,7,10
5,2,hgb_trend,2.0,7.932,0.991,27,8,10
6,3,hgb,2.0,10.792,0.714,13,10,10
7,3,hgb_trend,2.0,9.363,0.753,13,10,10
8,4,hgb,2.0,9.372,0.763,17,7,10
9,4,hgb_trend,1.0,26.411,0.794,17,8,10


,flagged,caught,insiders,precision,recall
base,,,,,
hgb,116,59,70,0.509,0.843
hgb_trend,123,60,70,0.488,0.857


**Paired bootstrap (1000 resamples): candidate minus `hgb` (positive = candidate better)**

  hgb_trend        recall@10%  diff +0.015  95% CI (0.0, 0.0492)  P(diff<=0) = 0.315
  hgb_trend        pr_auc      diff +0.010  95% CI (-0.0009, 0.0304)  P(diff<=0) = 0.066
  cusum            recall@10%  diff -0.156  95% CI (-0.25, -0.0602)  P(diff<=0) = 1.0
  cusum            pr_auc      diff -0.154  95% CI (-0.2369, -0.0786)  P(diff<=0) = 1.0
  hgb+cusum        recall@10%  diff -0.028  95% CI (-0.0976, 0.0429)  P(diff<=0) = 0.849
  hgb+cusum        pr_auc      diff -0.046  95% CI (-0.1003, 0.0011)  P(diff<=0) = 0.972
  hgb_trend+cusum  recall@10%  diff -0.014  95% CI (-0.08, 0.0513)  P(diff<=0) = 0.758
  hgb_trend+cusum  pr_auc      diff -0.024  95% CI (-0.0641, 0.009)  P(diff<=0) = 0.926

VERDICT: temporal component NOT supported on r4.2 -- no variant separates from plain HGB (95% CI includes 0). Report it as a documented negative result; the production Layer 2 stays plain HGB.
saved: oof_window_scores_hgb.csv, oof_temporal_users.csv, cusum_info.csv, results_temporal.json -> /kaggl

> ⚠ **Stale numbers:** the figures below came from the earlier run (Layer 1 scored with the LSTM). Re-run the notebook and rewrite this cell.

**Reading the output (7e, temporal component).** The `hgb` row is identical to 7b, so the comparison is like for like.

* **Trend features help slightly.** `hgb_trend` is at least as good as `hgb` on every metric: user PR-AUC 0.986 vs 0.976, window PR-AUC 0.875 vs 0.855, recall at 10% 0.986 vs 0.971 (one more insider).
  The evidence is borderline: the PR-AUC gain has P(diff ≤ 0) = 0.031 but its CI touches 0, and the recall gain is within noise. The automatic verdict above uses a strict rule (CI lower bound above 0),
  which is why it prints "not supported"; read this as a small, consistent, **not conclusive** improvement.
* **CUSUM does not help.** Alone it is clearly worse than HGB (user PR-AUC 0.808, recall at 10% 0.771; both CIs entirely below 0), and blending it with HGB also hurts (0.930 / 0.900 for `hgb+cusum`).
  Its frozen alarm flags 109 users with precision 0.52 and recall 0.81. The old fixed threshold `h = 8` flagged every user (precision 0.067, the base rate), so re-tuning fixed that failure but did not make CUSUM competitive.
* **Caveat.** The slack `k` was chosen at the top of the grid (2.0) in every fold, so CUSUM was not tuned to an interior optimum.

A plausible reason is that HGB already sees a 7-window history, so most of the temporal signal is absorbed and CUSUM mostly adds noise from benign users' spikes; the results do not prove that.

> ⚠ **Stale numbers:** the figures below came from the earlier run (Layer 1 scored with the LSTM). Re-run the notebook and rewrite this cell.

## 8. Summary of results (r4.2, 1000 users, 70 insiders, 7-fold user-level out-of-fold)

* **Layer 1:** no sampler beats training on the data as is; the paper's NM-2 collapses (test PR-AUC 0.061).
* **Layer 2 = HGB:** user PR-AUC 0.976. Flagging the top 10% of users catches 68 of 70 insiders (precision 0.694, recall 0.971, F1 0.810); the top 15% catches all 70.
* **Versus the paper:** better precision and F1 at a 10% budget, slightly lower recall; at matched recall 1.0 precision is lower (0.500 vs 0.649). Not a like-for-like comparison.
* **Versus sequence models:** HGB is significantly better on PR-AUC than LSTM, BiLSTM-attention, rank-average and stack; on recall at 10% it is tied with the two neural nets.
* **Drivers:** device connections, web activity and job-site browsing dominate the score.
* **Temporal gap:** trend features give a small, borderline improvement (user PR-AUC 0.986 vs 0.976); CUSUM, even re-tuned, is worse than HGB alone and is reported as a negative result.

## 9. Outputs
Everything worth keeping is under `/kaggle/working/insider_out/`: `features_v3_w2.pkl` (feature cache), `hgb_model/hgb.pkl` and `results.json` (sections 6-7),
`oof_user_scores_hgb.csv` (7b), `hgb_feature_importance.csv` (7c), `oof_ablation_none.csv` (7d), and `oof_window_scores_hgb.csv`, `oof_temporal_users.csv`,
`cusum_info.csv`, `results_temporal.json` (7e). Use **Save Version → Save & Run All (Commit)** for long runs so they keep going after you close the tab.


In [31]:
for root, _, files in os.walk(OUT_DIR):
    for f in files:
        p = os.path.join(root, f); print(f"{os.path.getsize(p)/1e6:9.1f} MB  {os.path.relpath(p, OUT_DIR)}")

      0.1 MB  oof_user_scores_hgb.csv
      0.0 MB  cusum_info.csv
      0.0 MB  results_temporal.json
      0.0 MB  hgb_feature_importance.csv
      0.3 MB  oof_ablation_DeepSMOTE@0.05+cw.csv
     16.0 MB  oof_window_scores_hgb.csv
      0.3 MB  oof_temporal_users.csv
     35.0 MB  features_v3_w2.pkl
      0.3 MB  results.json
      0.3 MB  oof_ablation_none.csv
      0.8 MB  hgb_model/hgb.pkl
